# BayesMMRL Mechanism Diagnosis — rebuild from trained checkpoints (v2 parser fix)

这个 notebook 只做**训练后分析**：

1. 从 `case_root` 中读取日志/配置，推断 method、dataset、shot、seed、backbone、protocol；
2. 使用仓库自己的 `setup_cfg()` / `build_trainer()` 重建 trainer；
3. 加载已经训练好的 checkpoint；
4. 实际检查本地代码实现，而不是凭路径或名字猜测；
5. 重新跑 eval dataloader，生成 `logits / logits_rep / logits_fusion / posterior_mean / MC samples / features`；
6. 输出 BayesMMRL 的 posterior、branch、fusion、sampling、aggregation、uncertainty、paired outcome、class/confusion 等机制诊断表。

重点：**方法类型默认从日志/配置中读取，不从 `name` 或路径中的 `MMRL/BayesMMRL` 字段猜测**。`case_root` 只用于定位日志和权重目录。


## 0. 配置

把 `REPO_ROOT`、`DATA_ROOT`、`CASES` 改成你的本地路径。

`CASES` 中只需要给 `case_root`。如果权重目录和日志目录分开，可以额外给：

```python
{
    "name": "可选显示名",
    "case_root": Path("..."),      # 默认日志和权重都从这里找
    "log_root": Path("..."),       # 可选：只用于日志/配置解析
    "model_dir": Path("..."),      # 可选：只用于 trainer.load_model
    "load_epoch": None,            # 可选：传给 trainer.load_model
    "overrides": {                 # 可选：只有日志缺字段时才用
        "method": "BayesMMRL",
        "dataset": "caltech101",
        "shots": 16,
        "seed": 1,
        "protocol": "FS",
        "backbone": "ViT-B/16",
    },
}
```

默认严格模式 `STRICT_LOG_METADATA=True`：如果日志/配置无法识别关键字段，会报错，避免误把路径当成配置。


In [31]:
from pathlib import Path

# ======== 修改这里 ========
REPO_ROOT = Path('/root/autodl-tmp/MMRL').expanduser().resolve()
DATA_ROOT = REPO_ROOT / 'DATASETS'
# 'output_sweeps/bayes_mmrl_rep_tokens/coarse3/confirm_stage/BayesMMRL/FS/fewshot_train/caltech101/shots_16/ViT-B-16/rep_zero_sig-diagonal_pstd-1.0_kl-5e-2/seed1',
# 'output_refactor/MMRL/FS/fewshot_train/caltech101/shots_16/ViT-B-16/default/seed1',


CASES = [
    {
        'name': 'case_A',
        
        'case_root': REPO_ROOT / 'output_sweeps/bayes_mmrl_rep_tokens/coarse3/confirm_stage/BayesMMRL/FS/fewshot_train/ucf101/shots_16/ViT-B-16/rep_zero_sig-diagonal_pstd-0.5_kl-5e-2/seed1',
    },
    {
        'name': 'case_B',
        'case_root': REPO_ROOT / 'output_refactor/MMRL/FS/fewshot_train/ucf101/shots_16/ViT-B-16/default/seed1',
    },
]

SPLIT = 'test'               # 'test' / 'val' / 'train_x'
BAYES_N_MC = 20              # 分析时重新采样的 MC 次数；不影响训练
MAX_BATCHES = None           # debug 时设成 2/5；正式分析设 None
COLLECT_MC_FEATURES = True  # True 会保存 MC image/text features，内存更大
STRICT_LOG_METADATA = True   # True: 不用路径猜 method/dataset/shot/seed/backbone
DEVICE = None                # None = 使用 trainer.device

SAVE_DIR = REPO_ROOT / 'analysis_outputs' / 'bayes_mmrl_mechanism_rebuild'
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print('REPO_ROOT =', REPO_ROOT)
print('DATA_ROOT =', DATA_ROOT)
print('SAVE_DIR  =', SAVE_DIR)
print('CASES     =', len(CASES))


REPO_ROOT = /root/autodl-tmp/MMRL
DATA_ROOT = /root/autodl-tmp/MMRL/DATASETS
SAVE_DIR  = /root/autodl-tmp/MMRL/analysis_outputs/bayes_mmrl_mechanism_rebuild
CASES     = 2


## 1. Imports 与仓库初始化


In [32]:
import os
import sys
import re
import json
import math
import hashlib
import importlib
import warnings
from types import SimpleNamespace
from collections import defaultdict, Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

assert REPO_ROOT.exists(), f'REPO_ROOT 不存在: {REPO_ROOT}'
assert DATA_ROOT.exists(), f'DATA_ROOT 不存在: {DATA_ROOT}'

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from core.config import setup_cfg
from dassl.engine import build_trainer
try:
    from core.utils import import_optional_modules
except Exception:
    import_optional_modules = None

# 尽量复用仓库原来的动态注册逻辑
if import_optional_modules is not None:
    import_optional_modules([
        'datasets.oxford_pets', 'datasets.oxford_flowers', 'datasets.fgvc_aircraft',
        'datasets.dtd', 'datasets.eurosat', 'datasets.stanford_cars', 'datasets.food101',
        'datasets.sun397', 'datasets.caltech101', 'datasets.ucf101', 'datasets.imagenet',
        'datasets.imagenetv2', 'datasets.imagenet_sketch', 'datasets.imagenet_a', 'datasets.imagenet_r',
    ])

for module_name in [
    'trainers.refactor_runner',
    'methods.mmrl.model',
    'methods.bayes_mmrl.model',
    'methods.mmrl_mix.model',
    'methods.mmrlpp.model',
]:
    try:
        importlib.import_module(module_name)
    except Exception as e:
        warnings.warn(f'Optional import failed: {module_name}: {e}')

print('torch =', torch.__version__)
print('cuda available =', torch.cuda.is_available())


torch = 2.11.0+cu128
cuda available = True


## 2. 实际检查代码实现

这一步会读取本地仓库源码，确认当前 checkout 里是否真的有 Bayes posterior、MC sampling、eval aggregation、fusion 公式等实现。后续诊断逻辑以这个检查结果为准。


In [33]:
IMPLEMENTATION_FILES = {
    'bayes_modules': REPO_ROOT / 'methods/bayes_mmrl/modules.py',
    'bayes_model': REPO_ROOT / 'methods/bayes_mmrl/model.py',
    'mmrl_family_modules': REPO_ROOT / 'methods/mmrl_family/modules.py',
    'mmrl_family_base': REPO_ROOT / 'methods/mmrl_family/base.py',
    'core_config': REPO_ROOT / 'core/config.py',
}

REQUIRED_CODE_TOKENS = {
    'bayes_modules': [
        'class BayesianTensorParameter',
        'class BayesianMatrixNormalParameter',
        'posterior_mean',
        'prior_mean',
        'posterior_sigma',
        'sample_tensor_many',
        'kl_divergence',
        '_canonical_eval_mode',
        '_canonical_eval_aggregation',
        'posterior_mean',
        'mc_predictive',
        'mean_plus_mc',
        'prob_mean',
        'logit_mean',
    ],
    'bayes_model': [
        'class BayesMMRLMethod',
        'method_name = "BayesMMRL"',
        'self.n_mc_test',
        'self.eval_use_posterior_mean',
        'forward_eval',
        'num_samples=self.n_mc_test',
        'use_posterior_mean=self.eval_use_posterior_mean',
        'aux_logits={',
        '"rep": logits_rep',
        '"fusion": logits_fusion',
    ],
    'mmrl_family_modules': [
        'class MMRLFamilyModel',
        'logits = 100.0 * image_features @ text_features.t()',
        'logits_rep = 100.0 * image_features_rep @ text_features.t()',
        'logits_fusion = self.alpha * logits + (1.0 - self.alpha) * logits_rep',
    ],
    'mmrl_family_base': [
        'class BaseMMRLFamilyMethod',
        'forward_eval',
        'aux_logits={"rep": logits_rep, "fusion": logits_fusion}',
        'select_eval_logits',
    ],
    'core_config': [
        'cfg.BAYES_MMRL.N_MC_TEST',
        'cfg.BAYES_MMRL.EVAL_MODE',
        'cfg.BAYES_MMRL.EVAL_USE_POSTERIOR_MEAN',
        'cfg.BAYES_MMRL.EVAL_AGGREGATION',
        'cfg.BAYES_MMRL.BAYES_TARGET',
    ],
}

def file_sha1(path: Path):
    data = path.read_bytes()
    return hashlib.sha1(data).hexdigest()


def inspect_implementation(strict=True):
    rows = []
    for key, path in IMPLEMENTATION_FILES.items():
        row = {'component': key, 'path': str(path.relative_to(REPO_ROOT)), 'exists': path.exists()}
        if not path.exists():
            row.update({'sha1': None, 'ok': False, 'missing_tokens': ['FILE_MISSING']})
            rows.append(row)
            continue
        text = path.read_text(errors='replace')
        missing = [tok for tok in REQUIRED_CODE_TOKENS.get(key, []) if tok not in text]
        row.update({
            'sha1': file_sha1(path),
            'n_lines': text.count('\n') + 1,
            'ok': len(missing) == 0,
            'missing_tokens': missing,
        })
        rows.append(row)
    df = pd.DataFrame(rows)
    if strict and (not df['ok'].all()):
        display(df)
        raise RuntimeError('Implementation audit failed. 请先确认本地代码与分析逻辑匹配，或设置 strict=False 后人工检查。')
    return df

implementation_audit = inspect_implementation(strict=True)
display(implementation_audit)
implementation_audit.to_csv(SAVE_DIR / 'implementation_audit.csv', index=False)


,component,path,exists,sha1,n_lines,ok,missing_tokens
0,bayes_modules,methods/bayes_mmrl/modules.py,True,bc00e300e3ec1ea1a34e34ba75995acad41e385e,1562,True,[]
1,bayes_model,methods/bayes_mmrl/model.py,True,178b86c7cffa1aafde9a45136008b19eeb113283,454,True,[]
2,mmrl_family_modules,methods/mmrl_family/modules.py,True,cf66c6877b8c9f34c96ce3f6558ec7361dbaa4a6,228,True,[]
3,mmrl_family_base,methods/mmrl_family/base.py,True,229a4d51aa605f172f4c490e8c6f5a5fe3aaf06f,148,True,[]
4,core_config,core/config.py,True,9d5a18ec642e6353534538ddf175a3dd4ca1c7fb,310,True,[]


## 3. 从日志/配置读取 case 元数据

不会从路径里的 `MMRL/BayesMMRL/caltech101/shots_16/...` 推断方法和数据集。解析优先级：

1. `case['overrides']`：只有你显式提供时才使用；
2. 日志或配置文件中的 cfg dump / 命令行参数；
3. 如果 `STRICT_LOG_METADATA=False`，才允许非常保守地 fallback 到路径解析。


In [34]:
METHOD_CFG_MAP = {
    'MMRL': 'configs/methods/mmrl.yaml',
    'MMRLMix': 'configs/methods/mmrl_mix.yaml',
    'MMRLpp': 'configs/methods/mmrlpp.yaml',
    'MMRLPP': 'configs/methods/mmrlpp.yaml',
    'BayesMMRL': 'configs/methods/bayesmmrl.yaml',
    'ClipAdapters': 'configs/methods/clip_adapters.yaml',
    'ClipADAPTER': 'configs/methods/clip_adapters.yaml',
}

PROTOCOL_CFG_MAP = {
    'B2N': 'configs/protocols/b2n.yaml',
    'FS': 'configs/protocols/fs.yaml',
    'CD': 'configs/protocols/cd.yaml',
}

PROTOCOL_TO_SUBSAMPLE = {
    'B2N': 'base',
    'FS': 'all',
    'CD': 'all',
}

LOG_GLOBS = [
    'log.txt', '*.log', 'log*.txt', '*stdout*.txt', '*stderr*.txt',
    'config.yaml', 'config.yml', 'cfg.yaml', 'cfg.yml', '*.yaml', '*.yml',
]


def read_text_safely(path: Path, max_bytes=2_000_000):
    try:
        data = path.read_bytes()
        if len(data) > max_bytes:
            # 头尾都保留，避免 cfg 在开头、最终指标在结尾
            half = max_bytes // 2
            data = data[:half] + b'\n\n...[TRUNCATED_BY_NOTEBOOK]...\n\n' + data[-half:]
        return data.decode('utf-8', errors='replace')
    except Exception:
        return ''


def collect_log_text(log_root: Path):
    log_root = Path(log_root)
    files = []
    if log_root.is_file():
        files = [log_root]
    elif log_root.exists():
        seen = set()
        for glob in LOG_GLOBS:
            for p in log_root.rglob(glob):
                if p.is_file() and p not in seen:
                    seen.add(p)
                    files.append(p)
    texts = []
    for p in sorted(files):
        txt = read_text_safely(p)
        if txt.strip():
            texts.append(f'\n\n===== FILE: {p} =====\n\n{txt}')
    return '\n'.join(texts), files


def strip_value(v):
    if v is None:
        return None
    v = str(v).strip()
    v = v.strip('"\'')
    # remove trailing comments
    v = re.sub(r'\s+#.*$', '', v).strip()
    return v


def _is_missing_yaml_scalar(raw_value: str) -> bool:
    """Treat `KEY:` with no value as a parent / empty field, never as the next line."""
    if raw_value is None:
        return True
    v = strip_value(raw_value)
    return v in (None, '', '|', '>')


def _looks_like_a_key_not_a_value(value: str) -> bool:
    """Reject parser mistakes like NAME -> INIT_WEIGHTS:."""
    if value is None:
        return True
    v = strip_value(value)
    if v in (None, ''):
        return True
    # Most bad captures are exactly another config key.
    if re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*:', v):
        return True
    if v in {'INIT_WEIGHTS:', 'BACKBONE:', 'HEAD:', 'NAME:', 'MODEL:'}:
        return True
    return False


def find_nested_scalar(text, path):
    """Robust YACS/YAML-style nested scalar parser.

    It scans the log/config line by line and follows indentation, so a block like

        MODEL:
          BACKBONE:
            NAME:
            INIT_WEIGHTS:

    will NOT parse NAME as "INIT_WEIGHTS:". Empty scalar values are ignored.

    It also supports flat cfg dumps such as:
        MODEL.BACKBONE.NAME: ViT-B/16
    """
    dotted = '.'.join(path)
    flat_patterns = [
        rf'(?m)^\s*{re.escape(dotted)}\s*[:=]\s*(.+?)\s*$',
        rf'(?m)^\s*{re.escape(dotted.replace(".", "_"))}\s*[:=]\s*(.+?)\s*$',
    ]
    flat_vals = []
    for pat in flat_patterns:
        for m in re.finditer(pat, text):
            v = strip_value(m.group(1))
            if not _looks_like_a_key_not_a_value(v):
                flat_vals.append(v)

    target = tuple(path)
    vals = []
    stack = []  # list[(indent, key)]

    key_line = re.compile(r'^(?P<indent>[ \t]*)(?P<key>[A-Za-z_][A-Za-z0-9_]*):(?P<value>.*)$')
    for line in text.splitlines():
        # Ignore our file separators and traceback lines.
        if line.startswith('===== FILE:') or line.lstrip().startswith('File '):
            continue

        m = key_line.match(line)
        if not m:
            continue

        indent = len(m.group('indent').replace('\t', '  '))
        key = m.group('key')
        raw_value = m.group('value')

        while stack and indent <= stack[-1][0]:
            stack.pop()

        current_path = tuple([k for _, k in stack] + [key])

        if current_path == target:
            if not _is_missing_yaml_scalar(raw_value):
                v = strip_value(raw_value)
                if not _looks_like_a_key_not_a_value(v):
                    vals.append(v)

        # Push only if this line is a parent / block. A scalar line does not become a parent.
        if _is_missing_yaml_scalar(raw_value):
            stack.append((indent, key))

    vals = flat_vals + vals
    vals = [v for v in vals if not _looks_like_a_key_not_a_value(v)]
    return vals[-1] if vals else None


def find_first_regex(text, patterns):
    for pat in patterns:
        m = re.search(pat, text, flags=re.IGNORECASE | re.MULTILINE)
        if m:
            v = strip_value(m.group(1))
            if not _looks_like_a_key_not_a_value(v):
                return v
    return None


def normalize_method(x):
    if x is None:
        return None
    s = str(x).strip()
    aliases = {
        'bayesmmrl': 'BayesMMRL',
        'mmrl': 'MMRL',
        'mmrlmix': 'MMRLMix',
        'mmrl_mix': 'MMRLMix',
        'mmrlpp': 'MMRLpp',
        'mmrl++': 'MMRLpp',
        'clipadapter': 'ClipAdapters',
        'clipadapters': 'ClipAdapters',
    }
    return aliases.get(s.lower(), s)


def normalize_protocol(x):
    if x is None:
        return None
    s = str(x).strip().upper()
    if s in {'B2N', 'FS', 'CD'}:
        return s
    return str(x).strip()


def normalize_backbone_name(x):
    if x is None:
        return None
    s = strip_value(x)
    if _looks_like_a_key_not_a_value(s):
        return None

    # Path tokens sometimes encode ViT-B/16 as ViT-B-16. Logs usually contain ViT-B/16.
    # Keep this normalization harmless and explicit.
    m = re.fullmatch(r'(ViT-[A-Za-z]+)-(\d+)', s)
    if m:
        return f'{m.group(1)}/{m.group(2)}'
    return s


def to_int_or_none(x):
    if x is None:
        return None
    try:
        return int(str(x).strip())
    except Exception:
        m = re.search(r'-?\d+', str(x))
        return int(m.group(0)) if m else None


def find_dataset_config(dataset):
    if not dataset:
        return None
    candidates = [
        REPO_ROOT / 'configs/datasets' / f'{dataset}.yaml',
        REPO_ROOT / 'configs/datasets' / f'{dataset.lower()}.yaml',
        REPO_ROOT / 'configs/datasets' / f'{dataset.upper()}.yaml',
    ]
    for p in candidates:
        if p.exists():
            return str(p.relative_to(REPO_ROOT))
    all_cfgs = list((REPO_ROOT / 'configs/datasets').glob('*.yaml'))
    low = dataset.lower().replace('_', '')
    for p in all_cfgs:
        if p.stem.lower().replace('_', '') == low:
            return str(p.relative_to(REPO_ROOT))
    raise FileNotFoundError(f'找不到 dataset config for {dataset}')


def path_fallback_metadata(case_root: Path):
    # 只在 STRICT_LOG_METADATA=False 时使用
    parts = Path(case_root).resolve().parts
    meta = {}
    methods = set(METHOD_CFG_MAP)
    protocols = set(PROTOCOL_CFG_MAP)
    for i, part in enumerate(parts):
        if part in methods:
            meta['method'] = part
        if part in protocols:
            meta['protocol'] = part
        if part.startswith('shots_'):
            meta['shots'] = to_int_or_none(part)
        if part.startswith('seed'):
            meta['seed'] = to_int_or_none(part)
        if part.startswith('ViT-'):
            meta['backbone'] = normalize_backbone_name(part)
    # protocol 后第二个常常是 dataset，但这是猜测，仅 fallback
    if 'protocol' in meta:
        idx = parts.index(meta['protocol'])
        if idx + 2 < len(parts):
            meta['dataset'] = parts[idx + 2]
    return meta


def validate_case_meta(meta):
    # Fail before build_trainer if a parser mistake slipped through.
    if _looks_like_a_key_not_a_value(meta.get('backbone')):
        raise ValueError(
            f"backbone 解析异常: {meta.get('backbone')!r}. "
            "这通常是日志中 MODEL.BACKBONE.NAME 为空，旧解析器误读了 INIT_WEIGHTS:。"
        )
    if meta.get('method') not in METHOD_CFG_MAP:
        raise ValueError(f"未知 method={meta.get('method')!r}; expected one of {sorted(METHOD_CFG_MAP)}")
    if meta.get('protocol') not in PROTOCOL_CFG_MAP:
        raise ValueError(f"未知 protocol={meta.get('protocol')!r}; expected one of {sorted(PROTOCOL_CFG_MAP)}")


def parse_case_metadata_from_logs(case):
    case_root = Path(case['case_root']).expanduser().resolve()
    log_root = Path(case.get('log_root', case_root)).expanduser().resolve()
    text, log_files = collect_log_text(log_root)
    overrides = dict(case.get('overrides', {}))

    meta = {
        'name': case.get('name', case_root.name),
        'case_root': case_root,
        'log_root': log_root,
        'model_dir': Path(case.get('model_dir', case_root)).expanduser().resolve(),
        'load_epoch': case.get('load_epoch', None),
        'n_log_files': len(log_files),
        'log_files': [str(p) for p in log_files[:20]],
    }

    # cfg dump style. This parser follows indentation and will not treat the next key as a value.
    meta['method'] = normalize_method(find_nested_scalar(text, ['METHOD', 'NAME']))
    meta['protocol'] = normalize_protocol(find_nested_scalar(text, ['PROTOCOL', 'NAME']))
    meta['dataset'] = find_nested_scalar(text, ['DATASET', 'NAME'])
    meta['shots'] = to_int_or_none(find_nested_scalar(text, ['DATASET', 'NUM_SHOTS']))
    meta['subsample_classes'] = find_nested_scalar(text, ['DATASET', 'SUBSAMPLE_CLASSES'])
    meta['backbone'] = normalize_backbone_name(find_nested_scalar(text, ['MODEL', 'BACKBONE', 'NAME']))
    meta['seed'] = to_int_or_none(find_nested_scalar(text, ['SEED']))
    meta['exec_mode'] = find_nested_scalar(text, ['METHOD', 'EXEC_MODE']) or 'online'

    # Bayes-specific optional cfg
    for k, path in {
        'bayes_target': ['BAYES_MMRL', 'BAYES_TARGET'],
        'eval_mode': ['BAYES_MMRL', 'EVAL_MODE'],
        'eval_aggregation': ['BAYES_MMRL', 'EVAL_AGGREGATION'],
        'eval_use_posterior_mean': ['BAYES_MMRL', 'EVAL_USE_POSTERIOR_MEAN'],
        'n_mc_test_cfg': ['BAYES_MMRL', 'N_MC_TEST'],
        'rep_sigma_mode': ['BAYES_MMRL', 'REP_SIGMA_MODE'],
        'rep_prior_mode': ['BAYES_MMRL', 'REP_PRIOR_MODE'],
        'rep_prior_std': ['BAYES_MMRL', 'REP_PRIOR_STD'],
        'rep_kl_weight': ['BAYES_MMRL', 'REP_KL_WEIGHT'],
    }.items():
        meta[k] = find_nested_scalar(text, path)

    # command-line / printed args fallback, still from log text, not from path/name.
    regex_fillers = {
        'method': [
            r'--method\s+([A-Za-z0-9_+\-]+)',
            r'(?m)^\s*method\s*[:=]\s*([A-Za-z0-9_+\-]+)\s*$',
            r'\[(BayesMMRL|MMRL|MMRLMix|MMRLpp)Method\]',
        ],
        'protocol': [
            r'--protocol\s+([A-Za-z0-9_+\-]+)',
            r'(?m)^\s*protocol\s*[:=]\s*([A-Za-z0-9_+\-]+)\s*$',
        ],
        'dataset': [
            r'--dataset(?:_config_file)?\s+.*?/([A-Za-z0-9_+\-]+)\.ya?ml',
            r'(?m)^\s*DATASET\.NAME\s*[:=]\s*([A-Za-z0-9_+\-]+)\s*$',
        ],
        'shots': [
            r'(?m)^\s*DATASET\.NUM_SHOTS\s*[:=]\s*(\d+)\s*$',
            r'--shots?\s+(\d+)',
            r'(?m)^\s*NUM_SHOTS\s*[:=]\s*(\d+)\s*$',
        ],
        'seed': [
            r'--seed\s+(\d+)',
            r'(?m)^\s*SEED\s*[:=]\s*(\d+)\s*$',
        ],
        'backbone': [
            r'(?m)^\s*MODEL\.BACKBONE\.NAME\s*[:=]\s*([^\n\r]+?)\s*$',
            r'(?m)^\s*BACKBONE\.NAME\s*[:=]\s*([^\n\r]+?)\s*$',
            r'--backbone\s+([^\s]+)',
        ],
    }
    for key, patterns in regex_fillers.items():
        if meta.get(key) in (None, ''):
            val = find_first_regex(text, patterns)
            if key == 'method':
                val = normalize_method(val)
            if key == 'protocol':
                val = normalize_protocol(val)
            if key == 'backbone':
                val = normalize_backbone_name(val)
            if key in {'shots', 'seed'}:
                val = to_int_or_none(val)
            meta[key] = val

    # explicit overrides: only if supplied by user
    for k, v in overrides.items():
        if k in {'method'}:
            v = normalize_method(v)
        if k in {'protocol'}:
            v = normalize_protocol(v)
        if k in {'backbone'}:
            v = normalize_backbone_name(v)
        if k in {'shots', 'seed'}:
            v = to_int_or_none(v)
        meta[k] = v

    # Optional fallback to path, disabled by default.
    if not STRICT_LOG_METADATA:
        fb = path_fallback_metadata(case_root)
        for k, v in fb.items():
            if meta.get(k) in (None, ''):
                meta[k] = v

    if meta.get('subsample_classes') in (None, '') and meta.get('protocol'):
        meta['subsample_classes'] = PROTOCOL_TO_SUBSAMPLE.get(meta['protocol'], 'all')

    required = ['method', 'protocol', 'dataset', 'shots', 'seed', 'backbone']
    missing = [k for k in required if meta.get(k) in (None, '')]
    if missing:
        msg = [
            f'无法从日志/配置解析必要字段: {missing}',
            f'case_root={case_root}',
            f'log_root={log_root}',
            f'找到日志/配置文件数={len(log_files)}',
            '请检查 log/config 是否存在，或在 CASES 的 overrides 中显式提供这些字段。',
            '注意：STRICT_LOG_METADATA=True 时不会从路径猜 method/dataset/shot/seed/backbone。',
        ]
        raise ValueError('\n'.join(msg))

    validate_case_meta(meta)

    meta['method_config_file'] = METHOD_CFG_MAP[meta['method']]
    meta['protocol_config_file'] = PROTOCOL_CFG_MAP[meta['protocol']]
    meta['dataset_config_file'] = find_dataset_config(meta['dataset'])
    return meta

case_metas = [parse_case_metadata_from_logs(c) for c in CASES]
meta_df = pd.DataFrame([{k: (str(v) if isinstance(v, Path) else v) for k, v in m.items() if k != 'log_files'} for m in case_metas])
display(meta_df)
meta_df.to_csv(SAVE_DIR / 'case_metadata_from_logs.csv', index=False)


,name,case_root,log_root,model_dir,load_epoch,n_log_files,method,protocol,dataset,shots,subsample_classes,backbone,seed,exec_mode,bayes_target,eval_mode,eval_aggregation,eval_use_posterior_mean,n_mc_test_cfg,rep_sigma_mode,rep_prior_mode,rep_prior_std,rep_kl_weight,method_config_file,protocol_config_file,dataset_config_file
0,case_A,/root/autodl-tmp/MMRL/output_sweeps/bayes_mmrl...,/root/autodl-tmp/MMRL/output_sweeps/bayes_mmrl...,/root/autodl-tmp/MMRL/output_sweeps/bayes_mmrl...,None,2,BayesMMRL,FS,UCF101,16,all,ViT-B/16,1,online,rep_tokens,mc_predictive,logit_mean,False,10,diagonal,zero,0.5,0.05,configs/methods/bayesmmrl.yaml,configs/protocols/fs.yaml,configs/datasets/ucf101.yaml
1,case_B,/root/autodl-tmp/MMRL/output_refactor/MMRL/FS/...,/root/autodl-tmp/MMRL/output_refactor/MMRL/FS/...,/root/autodl-tmp/MMRL/output_refactor/MMRL/FS/...,None,2,MMRL,FS,UCF101,16,all,ViT-B/16,1,online,rep_tokens,mc_predictive,prob_mean,False,5,global,zero,0.05,0.0005,configs/methods/mmrl.yaml,configs/protocols/fs.yaml,configs/datasets/ucf101.yaml


## 4. 重建 trainer 并加载 checkpoint

不会训练，只调用仓库自己的 `build_trainer(cfg)` 和 `trainer.load_model(model_dir)`。


In [35]:
def make_args_from_meta(meta):

    if _looks_like_a_key_not_a_value(meta.get('backbone')):
        raise ValueError(f"Invalid backbone parsed from logs: {meta.get('backbone')!r}. Check case_metadata_from_logs or provide CASES[*]['overrides']['backbone'].")
    opts = [
        'DATASET.NUM_SHOTS', str(meta['shots']),
        'DATASET.SUBSAMPLE_CLASSES', str(meta.get('subsample_classes') or PROTOCOL_TO_SUBSAMPLE.get(meta['protocol'], 'all')),
        'MODEL.BACKBONE.NAME', str(meta['backbone']),
    ]
    # 保留日志中解析到的 Bayes eval 配置，避免重新构建时被默认值覆盖。
    # 这些只影响标准 forward_eval；本 notebook 的 MC 诊断会直接重复采样生成。
    if meta['method'] == 'BayesMMRL':
        bayes_opt_map = {
            'bayes_target': 'BAYES_MMRL.BAYES_TARGET',
            'eval_mode': 'BAYES_MMRL.EVAL_MODE',
            'eval_aggregation': 'BAYES_MMRL.EVAL_AGGREGATION',
            'eval_use_posterior_mean': 'BAYES_MMRL.EVAL_USE_POSTERIOR_MEAN',
            'n_mc_test_cfg': 'BAYES_MMRL.N_MC_TEST',
            'rep_sigma_mode': 'BAYES_MMRL.REP_SIGMA_MODE',
            'rep_prior_mode': 'BAYES_MMRL.REP_PRIOR_MODE',
            'rep_prior_std': 'BAYES_MMRL.REP_PRIOR_STD',
            'rep_kl_weight': 'BAYES_MMRL.REP_KL_WEIGHT',
        }
        for mk, ck in bayes_opt_map.items():
            v = meta.get(mk)
            if v not in (None, ''):
                opts.extend([ck, str(v)])

    args = SimpleNamespace(
        root=str(DATA_ROOT),
        output_dir=str(meta['model_dir']),
        dataset_config_file=meta['dataset_config_file'],
        method_config_file=meta['method_config_file'],
        protocol_config_file=meta['protocol_config_file'],
        runtime_config_file='configs/runtime/default.yaml',
        exp_config='',
        method=meta['method'],
        protocol=meta['protocol'],
        exec_mode=meta.get('exec_mode') or 'online',
        seed=int(meta['seed']),
        trainer='RefactorRunner',
        eval_only=True,
        model_dir=str(meta['model_dir']),
        load_epoch=meta.get('load_epoch', None),
        no_train=True,
        opts=opts,
    )
    return args


def build_and_load_case(meta):
    args = make_args_from_meta(meta)
    cfg = setup_cfg(args)
    trainer = build_trainer(cfg)
    load_epoch = meta.get('load_epoch', None)
    try:
        trainer.load_model(str(meta['model_dir']), epoch=load_epoch)
    except TypeError:
        if load_epoch is not None:
            warnings.warn('trainer.load_model 不接受 epoch 参数，将只传 model_dir。')
        trainer.load_model(str(meta['model_dir']))
    trainer.set_model_mode('eval')
    return trainer, cfg

trainers = {}
cfgs = {}
for meta in case_metas:
    print(f"\n=== Rebuild & load: {meta['name']} ===")
    trainer, cfg = build_and_load_case(meta)
    trainers[meta['name']] = trainer
    cfgs[meta['name']] = cfg
    print('cfg.METHOD.NAME =', cfg.METHOD.NAME)
    print('cfg.DATASET.NAME =', cfg.DATASET.NAME)
    print('cfg.DATASET.NUM_SHOTS =', cfg.DATASET.NUM_SHOTS)
    print('cfg.SEED =', cfg.SEED)
    if hasattr(cfg, 'BAYES_MMRL'):
        print('cfg.BAYES_MMRL.BAYES_TARGET =', getattr(cfg.BAYES_MMRL, 'BAYES_TARGET', None))
        print('cfg.BAYES_MMRL.N_MC_TEST =', getattr(cfg.BAYES_MMRL, 'N_MC_TEST', None))



=== Rebuild & load: case_A ===
Loading trainer: RefactorRunner
Loading dataset: UCF101
Reading split from /root/autodl-tmp/MMRL/DATASETS/ucf101/split_zhou_UCF101.json
Loading preprocessed few-shot data from /root/autodl-tmp/MMRL/DATASETS/ucf101/split_fewshot/shot_16-seed_1.pkl
Building transform_train
+ random resized crop (size=(224, 224), scale=(0.5, 1))
+ random flip
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711])
Building transform_test
+ resize the smaller edge to 224
+ 224x224 center crop
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711])
---------  ------
Dataset    UCF101
# classes  101
# train_x  1,616
# val      404
# test     3,783
---------  ------
[BayesMMRLMethod] trainable params: {'representation_learner.compound_rep_tokens_r2vproj.4.weight', 'representation_learner.compound_rep_tokens_r2vproj.1.bias'

/root/autodl-tmp/MMRL/trainers/refactor_runner.py:61: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler() if prec == "amp" else None


Loading weights to refactor_model from "/root/autodl-tmp/MMRL/output_sweeps/bayes_mmrl_rep_tokens/coarse3/confirm_stage/BayesMMRL/FS/fewshot_train/ucf101/shots_16/ViT-B-16/rep_zero_sig-diagonal_pstd-0.5_kl-5e-2/seed1/refactor_model/model.pth.tar-50" (epoch = 50)
Skipped class-dependent or shape-mismatched keys:
  - tokenized_prompts
  - prompt_embeddings
  - representation_learner.tokenized_prompts
  - representation_learner.prompt_embeddings
Missing keys after loading:
  - tokenized_prompts
  - prompt_embeddings
  - representation_learner.tokenized_prompts
  - representation_learner.prompt_embeddings
cfg.METHOD.NAME = BayesMMRL
cfg.DATASET.NAME = UCF101
cfg.DATASET.NUM_SHOTS = 16
cfg.SEED = 1
cfg.BAYES_MMRL.BAYES_TARGET = rep_tokens
cfg.BAYES_MMRL.N_MC_TEST = 10

=== Rebuild & load: case_B ===
Loading trainer: RefactorRunner
Loading dataset: UCF101
Reading split from /root/autodl-tmp/MMRL/DATASETS/ucf101/split_zhou_UCF101.json
Loading preprocessed few-shot data from /root/autodl-tmp/M

## 5. 模型对象与 posterior 审计


In [36]:
def state_dict_bayes_keys(model):
    sd = model.state_dict()
    rows = []
    keywords = ['posterior', 'prior', 'rho', 'bayes_proj_rep', 'token_tril', 'feature_diag', 'feature_lowrank']
    for k, v in sd.items():
        if any(s in k for s in keywords):
            rows.append({
                'key': k,
                'shape': tuple(v.shape) if hasattr(v, 'shape') else None,
                'dtype': str(v.dtype) if hasattr(v, 'dtype') else None,
                'mean': float(v.detach().float().mean().cpu()) if torch.is_tensor(v) and v.numel() else None,
                'std': float(v.detach().float().std(unbiased=False).cpu()) if torch.is_tensor(v) and v.numel() > 1 else None,
            })
    return pd.DataFrame(rows)


def get_bayes_block(model):
    # Scheme A/B: Bayes on representation tokens
    if hasattr(model, 'representation_learner') and hasattr(model.representation_learner, 'rep_posterior'):
        return 'rep_tokens', model.representation_learner.rep_posterior
    # Scheme C: Bayes on proj_rep
    if hasattr(model, 'image_encoder') and hasattr(model.image_encoder, 'bayes_proj_rep'):
        return 'proj_rep', model.image_encoder.bayes_proj_rep
    return None, None


def is_bayes_model_object(model):
    target, block = get_bayes_block(model)
    return block is not None


def posterior_diagnostics(model):
    target, block = get_bayes_block(model)
    if block is None:
        return pd.DataFrame([{'is_bayes': False, 'bayes_target': None, 'error': 'No Bayesian posterior block found'}])

    posterior_mean = getattr(block, 'posterior_mean', None)
    prior_mean = getattr(block, 'prior_mean', None)

    posterior_sigma = None
    if hasattr(block, 'posterior_sigma'):
        try:
            posterior_sigma = block.posterior_sigma()
        except Exception as e:
            warnings.warn(f'posterior_sigma() failed: {e}')

    kl = None
    if hasattr(block, 'kl_divergence'):
        try:
            kl = float(block.kl_divergence().detach().float().cpu())
        except Exception as e:
            warnings.warn(f'kl_divergence() failed: {e}')

    row = {
        'is_bayes': True,
        'bayes_target': target,
        'block_class': type(block).__name__,
        'KL': kl,
    }

    if posterior_mean is not None:
        pm = posterior_mean.detach().float()
        row.update({
            'posterior_mean_shape': tuple(pm.shape),
            'posterior_mean_l2': float(torch.linalg.vector_norm(pm).cpu()),
            'posterior_mean_abs_mean': float(pm.abs().mean().cpu()),
        })
    if prior_mean is not None:
        pr = prior_mean.detach().float()
        row.update({
            'prior_mean_shape': tuple(pr.shape),
            'prior_mean_l2': float(torch.linalg.vector_norm(pr).cpu()),
            'prior_mean_abs_mean': float(pr.abs().mean().cpu()),
        })
    if posterior_mean is not None and prior_mean is not None:
        delta = posterior_mean.detach().float() - prior_mean.detach().float()
        row.update({
            'mean_shift_l2': float(torch.linalg.vector_norm(delta).cpu()),
            'mean_shift_abs_mean': float(delta.abs().mean().cpu()),
            'mean_shift_abs_max': float(delta.abs().max().cpu()),
        })
    if posterior_sigma is not None:
        sig = posterior_sigma.detach().float()
        row.update({
            'posterior_sigma_shape': tuple(sig.shape),
            'sigma_mean': float(sig.mean().cpu()),
            'sigma_std': float(sig.std(unbiased=False).cpu()) if sig.numel() > 1 else 0.0,
            'sigma_min': float(sig.min().cpu()),
            'sigma_max': float(sig.max().cpu()),
            'sigma_cv': float((sig.std(unbiased=False) / sig.mean().clamp_min(1e-12)).cpu()) if sig.numel() > 1 else 0.0,
        })
    if posterior_mean is not None and prior_mean is not None and posterior_sigma is not None:
        delta = (posterior_mean.detach().float() - prior_mean.detach().float()).abs()
        sig = posterior_sigma.detach().float()
        if sig.shape != delta.shape:
            try:
                sig = sig.expand_as(delta)
            except Exception:
                pass
        if sig.shape == delta.shape:
            snr = delta / sig.clamp_min(1e-12)
            row.update({'SNR_mean': float(snr.mean().cpu()), 'SNR_max': float(snr.max().cpu())})
    return pd.DataFrame([row])

posterior_rows = []
state_key_tables = {}
for meta in case_metas:
    name = meta['name']
    trainer = trainers[name]
    model = trainer.method.model
    pdiag = posterior_diagnostics(model)
    pdiag.insert(0, 'case_name', name)
    pdiag.insert(1, 'method_from_cfg', trainer.cfg.METHOD.NAME)
    posterior_rows.append(pdiag)
    state_key_tables[name] = state_dict_bayes_keys(model)

posterior_df = pd.concat(posterior_rows, ignore_index=True)
display(posterior_df)
posterior_df.to_csv(SAVE_DIR / 'posterior_diagnostics.csv', index=False)

for name, df in state_key_tables.items():
    if len(df):
        df.to_csv(SAVE_DIR / f'state_dict_bayes_keys__{name}.csv', index=False)


,case_name,method_from_cfg,is_bayes,bayes_target,block_class,KL,posterior_mean_shape,posterior_mean_l2,posterior_mean_abs_mean,prior_mean_shape,prior_mean_l2,prior_mean_abs_mean,mean_shift_l2,mean_shift_abs_mean,mean_shift_abs_max,posterior_sigma_shape,sigma_mean,sigma_std,sigma_min,sigma_max,sigma_cv,SNR_mean,SNR_max,error
0,case_A,BayesMMRL,True,rep_tokens,BayesianTensorParameter,64.077103,"(5, 512)",5.509155,0.093204,"(5, 512)",0.0,0.0,5.509155,0.093204,0.241732,"(5, 512)",0.482468,0.004232,0.46903,0.497027,0.008771,0.1933,0.50011,NaN
1,case_B,MMRL,False,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No Bayesian posterior block found


## 6. 指标函数


In [37]:
def entropy_from_probs(p, eps=1e-12):
    p = p.clamp_min(eps)
    return -(p * p.log()).sum(dim=-1)


def probs_from_logits(logits):
    return torch.softmax(logits.float(), dim=-1)


def pred_conf_margin_entropy_from_probs(probs):
    conf, pred = probs.max(dim=-1)
    k = min(2, probs.shape[-1])
    top2 = torch.topk(probs, k=k, dim=-1).values
    margin = top2[:, 0] if k == 1 else top2[:, 0] - top2[:, 1]
    ent = entropy_from_probs(probs)
    return pred, conf, margin, ent


def metrics_from_probs(probs, labels, n_bins=15):
    labels = labels.long()
    pred, conf, margin, ent = pred_conf_margin_entropy_from_probs(probs)
    correct = pred.eq(labels)
    n = int(labels.numel())
    y = F.one_hot(labels, num_classes=probs.shape[-1]).float()
    nll = -torch.log(probs[torch.arange(n, device=labels.device), labels].clamp_min(1e-12))
    brier = ((probs - y) ** 2).sum(dim=-1)
    return {
        'n': n,
        'acc': float(correct.float().mean().cpu()),
        'macro_f1': macro_f1_from_preds(pred, labels, probs.shape[-1]),
        'ECE': float(expected_calibration_error(conf, correct, n_bins=n_bins).cpu()),
        'NLL': float(nll.mean().cpu()),
        'Brier': float(brier.mean().cpu()),
        'conf_mean': float(conf.mean().cpu()),
        'entropy_mean': float(ent.mean().cpu()),
        'margin_mean': float(margin.mean().cpu()),
    }


def expected_calibration_error(conf, correct, n_bins=15):
    conf = conf.detach().float()
    correct = correct.detach().float()
    bins = torch.linspace(0, 1, n_bins + 1, device=conf.device)
    ece = torch.zeros((), device=conf.device)
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (conf >= lo) & (conf <= hi) if i == 0 else (conf > lo) & (conf <= hi)
        if mask.any():
            ece = ece + mask.float().mean() * (correct[mask].mean() - conf[mask].mean()).abs()
    return ece


def macro_f1_from_preds(pred, labels, num_classes):
    pred_np = pred.detach().cpu().numpy()
    lab_np = labels.detach().cpu().numpy()
    vals = []
    for c in range(num_classes):
        tp = np.logical_and(pred_np == c, lab_np == c).sum()
        fp = np.logical_and(pred_np == c, lab_np != c).sum()
        fn = np.logical_and(pred_np != c, lab_np == c).sum()
        denom = 2 * tp + fp + fn
        if denom > 0:
            vals.append(2 * tp / denom)
    return float(np.mean(vals)) if vals else float('nan')


def binary_auroc(scores, is_positive):
    # Mann-Whitney U based AUROC. Higher score => more positive.
    s = np.asarray(scores, dtype=float)
    y = np.asarray(is_positive, dtype=bool)
    ok = np.isfinite(s)
    s, y = s[ok], y[ok]
    n_pos = int(y.sum())
    n_neg = int((~y).sum())
    if n_pos == 0 or n_neg == 0:
        return np.nan
    order = np.argsort(s)
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(s) + 1)
    # tie correction: average ranks for ties
    unique, inv, counts = np.unique(s, return_inverse=True, return_counts=True)
    if np.any(counts > 1):
        for idx, cnt in enumerate(counts):
            if cnt > 1:
                mask = inv == idx
                ranks[mask] = ranks[mask].mean()
    sum_ranks_pos = ranks[y].sum()
    auc = (sum_ranks_pos - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)
    return float(auc)


def binary_aupr(scores, is_positive):
    s = np.asarray(scores, dtype=float)
    y = np.asarray(is_positive, dtype=bool)
    ok = np.isfinite(s)
    s, y = s[ok], y[ok]
    n_pos = int(y.sum())
    if n_pos == 0:
        return np.nan
    order = np.argsort(-s)
    y_sorted = y[order]
    tp = np.cumsum(y_sorted)
    fp = np.cumsum(~y_sorted)
    precision = tp / np.maximum(tp + fp, 1)
    recall = tp / n_pos
    # step-wise AP
    recall_prev = np.concatenate([[0.0], recall[:-1]])
    ap = np.sum((recall - recall_prev) * precision)
    return float(ap)


def js_divergence_probs(p, q, eps=1e-12):
    p = p.clamp_min(eps)
    q = q.clamp_min(eps)
    m = 0.5 * (p + q)
    return 0.5 * (p * (p.log() - m.log())).sum(dim=-1) + 0.5 * (q * (q.log() - m.log())).sum(dim=-1)


def sym_kl_probs(p, q, eps=1e-12):
    p = p.clamp_min(eps)
    q = q.clamp_min(eps)
    return 0.5 * ((p * (p.log() - q.log())).sum(dim=-1) + (q * (q.log() - p.log())).sum(dim=-1))


def l1_probs(p, q):
    return (p - q).abs().sum(dim=-1)


def margin_y_vs_best_wrong(logits, labels):
    logits = logits.float()
    labels = labels.long()
    n, c = logits.shape
    y_logit = logits[torch.arange(n, device=logits.device), labels]
    masked = logits.clone()
    masked[torch.arange(n, device=logits.device), labels] = -float('inf')
    wrong_logit, wrong_idx = masked.max(dim=-1)
    return y_logit - wrong_logit, wrong_idx


def cosine_per_row(a, b, eps=1e-8):
    a = a.float()
    b = b.float()
    return (a * b).sum(dim=-1) / (a.norm(dim=-1) * b.norm(dim=-1)).clamp_min(eps)


## 7. 重新 forward：standard / posterior_mean / MC samples

- MMRL：一次 deterministic forward。
- BayesMMRL：
  - `standard`：按当前 cfg 的 `forward_eval` 输出；
  - `posterior_mean`：直接调用模型 `forward_eval(..., use_posterior_mean=True)`；
  - `mc_samples`：重复调用 `forward_eval(..., use_posterior_mean=False, num_samples=1)` 生成 S 个样本，然后 notebook 自己计算 `prob_mean / logit_mean / majority_vote`。


In [38]:
def make_eval_ctx(cfg):
    return SimpleNamespace(
        protocol=str(getattr(cfg.PROTOCOL, 'NAME', getattr(cfg, 'TASK', ''))),
        dataset_name=str(getattr(cfg.DATASET, 'NAME', '')),
        subsample_classes=str(getattr(cfg.DATASET, 'SUBSAMPLE_CLASSES', 'all')),
    )


def get_loader(trainer, split='test'):
    candidates = []
    if split == 'test':
        candidates = ['test_loader', 'test_loader_x']
    elif split == 'val':
        candidates = ['val_loader', 'valid_loader']
    elif split == 'train_x':
        candidates = ['train_loader_x']
    else:
        candidates = [f'{split}_loader']
    for name in candidates:
        if hasattr(trainer, name):
            loader = getattr(trainer, name)
            if loader is not None:
                return loader
    raise AttributeError(f'找不到 split={split} 的 dataloader，尝试过: {candidates}')


def batch_sample_ids(batch, offset):
    n = None
    if isinstance(batch, dict):
        if 'label' in batch:
            n = len(batch['label'])
        elif 'img' in batch:
            n = len(batch['img'])
        for key in ['impath', 'img_path', 'path', 'index', 'idx']:
            if key in batch:
                val = batch[key]
                if torch.is_tensor(val):
                    return [str(x) for x in val.detach().cpu().tolist()]
                return [str(x) for x in list(val)]
    if n is None:
        n = len(batch)
    return [f'row_{offset + i:08d}' for i in range(n)]


def unpack_output_tuple(tup):
    logits, logits_rep, logits_fusion, image_features, text_features = tup
    return {
        'main': logits.float(),
        'rep': logits_rep.float(),
        'fusion': logits_fusion.float(),
        'features_img': image_features.float(),
        'features_text': text_features.float(),
    }


def detach_cpu_dict(d):
    out = {}
    for k, v in d.items():
        if torch.is_tensor(v):
            out[k] = v.detach().float().cpu()
        else:
            out[k] = v
    return out


def standard_forward_batch(trainer, batch, eval_ctx):
    method = trainer.method
    with torch.no_grad():
        outputs = method.forward_eval(batch, eval_ctx)
    branch_logits = {'main': outputs.logits.float()}
    if hasattr(outputs, 'aux_logits') and outputs.aux_logits:
        if outputs.aux_logits.get('rep') is not None:
            branch_logits['rep'] = outputs.aux_logits['rep'].float()
        if outputs.aux_logits.get('fusion') is not None:
            branch_logits['fusion'] = outputs.aux_logits['fusion'].float()
    feats = {}
    if hasattr(outputs, 'features') and outputs.features:
        if outputs.features.get('img') is not None:
            feats['features_img'] = outputs.features['img'].float()
        if outputs.features.get('text') is not None:
            feats['features_text'] = outputs.features['text'].float()
    labels = outputs.labels if outputs.labels is not None else batch.get('label')
    return branch_logits, feats, labels


def bayes_model_forward_batch(trainer, batch, num_samples=1, use_posterior_mean=False):
    method = trainer.method
    model = method.model
    device = method.device if hasattr(method, 'device') else next(model.parameters()).device
    image = batch['img'].to(device)
    with torch.no_grad():
        tup = model.forward_eval(image, num_samples=int(num_samples), use_posterior_mean=bool(use_posterior_mean))
    return unpack_output_tuple(tup)


def collect_case_outputs(trainer, meta, split='test', bayes_n_mc=20, max_batches=None, collect_mc_features=False):
    cfg = trainer.cfg
    eval_ctx = make_eval_ctx(cfg)
    loader = get_loader(trainer, split)
    model = trainer.method.model
    is_bayes = is_bayes_model_object(model)

    labels_all = []
    sample_ids = []
    standard_branches = defaultdict(list)
    pm_branches = defaultdict(list)
    standard_features = defaultdict(list)
    pm_features = defaultdict(list)
    mc_branches_by_s = None
    mc_features_by_s = None

    global_offset = 0
    for bi, batch in enumerate(loader):
        if max_batches is not None and bi >= max_batches:
            break

        ids = batch_sample_ids(batch, global_offset)
        std_logits, std_feats, labels = standard_forward_batch(trainer, batch, eval_ctx)
        labels = labels.detach().long().cpu()
        labels_all.append(labels)
        sample_ids.extend(ids)
        global_offset += len(labels)

        for br, x in std_logits.items():
            standard_branches[br].append(x.detach().float().cpu())
        for fk, x in std_feats.items():
            standard_features[fk].append(x.detach().float().cpu())

        if is_bayes:
            pm = bayes_model_forward_batch(trainer, batch, num_samples=1, use_posterior_mean=True)
            for br in ['main', 'rep', 'fusion']:
                if br in pm:
                    pm_branches[br].append(pm[br].detach().float().cpu())
            for fk in ['features_img', 'features_text']:
                if fk in pm:
                    pm_features[fk].append(pm[fk].detach().float().cpu())

            batch_mc = {br: [] for br in ['main', 'rep', 'fusion']}
            batch_mc_feat = {fk: [] for fk in ['features_img', 'features_text']} if collect_mc_features else None
            for s in range(int(bayes_n_mc)):
                smp = bayes_model_forward_batch(trainer, batch, num_samples=1, use_posterior_mean=False)
                for br in ['main', 'rep', 'fusion']:
                    if br in smp:
                        batch_mc[br].append(smp[br].detach().float().cpu())
                if collect_mc_features:
                    for fk in ['features_img', 'features_text']:
                        if fk in smp:
                            batch_mc_feat[fk].append(smp[fk].detach().float().cpu())

            if mc_branches_by_s is None:
                mc_branches_by_s = {br: [[] for _ in range(int(bayes_n_mc))] for br in batch_mc}
            for br, xs in batch_mc.items():
                for s, x in enumerate(xs):
                    mc_branches_by_s[br][s].append(x)

            if collect_mc_features:
                if mc_features_by_s is None:
                    mc_features_by_s = {fk: [[] for _ in range(int(bayes_n_mc))] for fk in batch_mc_feat}
                for fk, xs in batch_mc_feat.items():
                    for s, x in enumerate(xs):
                        mc_features_by_s[fk][s].append(x)

    labels_all = torch.cat(labels_all, dim=0)

    def cat_branch_dict(d):
        return {k: torch.cat(v, dim=0) for k, v in d.items() if len(v)}

    result = {
        'name': meta['name'],
        'meta': meta,
        'cfg_method': trainer.cfg.METHOD.NAME,
        'is_bayes': is_bayes,
        'labels': labels_all,
        'sample_ids': sample_ids,
        'standard': cat_branch_dict(standard_branches),
        'standard_features': cat_branch_dict(standard_features),
    }
    if is_bayes:
        result['posterior_mean'] = cat_branch_dict(pm_branches)
        result['posterior_mean_features'] = cat_branch_dict(pm_features)
        result['mc_samples'] = {
            br: torch.stack([torch.cat(chunks, dim=0) for chunks in per_s], dim=0)
            for br, per_s in mc_branches_by_s.items()
        }
        if collect_mc_features and mc_features_by_s is not None:
            result['mc_sample_features'] = {
                fk: torch.stack([torch.cat(chunks, dim=0) for chunks in per_s], dim=0)
                for fk, per_s in mc_features_by_s.items()
            }
    return result

case_outputs = {}
for meta in case_metas:
    print(f"\n=== Forward collect: {meta['name']} ===")
    out = collect_case_outputs(
        trainers[meta['name']], meta,
        split=SPLIT,
        bayes_n_mc=BAYES_N_MC,
        max_batches=MAX_BATCHES,
        collect_mc_features=COLLECT_MC_FEATURES,
    )
    case_outputs[meta['name']] = out
    print('N =', len(out['labels']), 'is_bayes =', out['is_bayes'], 'branches =', list(out['standard']))
    if out['is_bayes']:
        print('MC shapes:', {k: tuple(v.shape) for k, v in out['mc_samples'].items()})



=== Forward collect: case_A ===


N = 3783 is_bayes = True branches = ['main', 'rep', 'fusion']
MC shapes: {'main': (20, 3783, 101), 'rep': (20, 3783, 101), 'fusion': (20, 3783, 101)}

=== Forward collect: case_B ===
N = 3783 is_bayes = False branches = ['main', 'rep', 'fusion']


## 8. 构造 eval modes 与主指标表


In [39]:
def eval_modes_for_case(out):
    modes = {'standard': out['standard']}
    if out['is_bayes']:
        modes['posterior_mean'] = out['posterior_mean']
        # MC aggregations
        prob_mean = {}
        logit_mean = {}
        majority_vote = {}
        for br, smp_logits in out['mc_samples'].items():
            prob_mean[br] = probs_from_logits(smp_logits).mean(dim=0)
            logit_mean[br] = smp_logits.mean(dim=0)
            maj = predictive_uncertainty_from_mc_logits(smp_logits)['majority_vote']
            majority_vote[br] = maj
        modes['mc_prob_mean_probs'] = prob_mean
        modes['mc_logit_mean'] = logit_mean
        modes['mc_majority_vote_pred'] = majority_vote
    return modes


def predictive_uncertainty_from_mc_logits(logits_samples, eps=1e-12):
    probs_samples = probs_from_logits(logits_samples)
    probs_mean = probs_samples.mean(dim=0)
    predictive_entropy = entropy_from_probs(probs_mean, eps=eps)
    expected_entropy = entropy_from_probs(probs_samples, eps=eps).mean(dim=0)
    mutual_info = predictive_entropy - expected_entropy
    pred_samples = probs_samples.argmax(dim=-1)  # [S,N]
    S, N = pred_samples.shape
    sample_agreement = []
    num_unique = []
    vote_entropy = []
    majority_vote = []
    for i in range(N):
        vals, counts = pred_samples[:, i].unique(return_counts=True)
        idx = counts.argmax()
        majority_vote.append(vals[idx])
        sample_agreement.append(counts[idx].float() / float(S))
        num_unique.append(vals.numel())
        p = counts.float() / counts.sum().float()
        vote_entropy.append(-(p * p.clamp_min(eps).log()).sum())
    return {
        'probs_samples': probs_samples,
        'probs_mean': probs_mean,
        'predictive_entropy': predictive_entropy,
        'expected_entropy': expected_entropy,
        'mutual_info': mutual_info,
        'pred_samples': pred_samples,
        'majority_vote': torch.stack(majority_vote).to(logits_samples.device),
        'sample_agreement': torch.stack(sample_agreement).to(logits_samples.device),
        'variation_ratio': 1.0 - torch.stack(sample_agreement).to(logits_samples.device),
        'num_unique': torch.tensor(num_unique, dtype=torch.float32, device=logits_samples.device),
        'vote_entropy': torch.stack(vote_entropy).to(logits_samples.device),
        'logit_var_mean': logits_samples.var(dim=0, unbiased=False).mean(dim=-1),
    }


def branch_metrics_table(case_outputs):
    rows = []
    for name, out in case_outputs.items():
        labels = out['labels']
        modes = eval_modes_for_case(out)
        for mode_name, branches in modes.items():
            for br, obj in branches.items():
                row = {'case_name': name, 'method': out['cfg_method'], 'eval_mode': mode_name, 'branch': br}
                if mode_name.endswith('_probs'):
                    row.update(metrics_from_probs(obj, labels))
                elif mode_name.endswith('_pred'):
                    pred = obj.long()
                    correct = pred.eq(labels)
                    row.update({
                        'n': int(labels.numel()),
                        'acc': float(correct.float().mean().cpu()),
                        'macro_f1': macro_f1_from_preds(pred, labels, int(max(labels.max().item() + 1, pred.max().item() + 1))),
                        'ECE': np.nan, 'NLL': np.nan, 'Brier': np.nan,
                        'conf_mean': np.nan, 'entropy_mean': np.nan, 'margin_mean': np.nan,
                    })
                else:
                    row.update(metrics_from_probs(probs_from_logits(obj), labels))
                rows.append(row)
    return pd.DataFrame(rows)

branch_metrics_df = branch_metrics_table(case_outputs)
display(branch_metrics_df)
branch_metrics_df.to_csv(SAVE_DIR / 'table_branch_main_metrics.csv', index=False)


,case_name,method,eval_mode,branch,n,acc,macro_f1,ECE,NLL,Brier,conf_mean,entropy_mean,margin_mean
0,case_A,BayesMMRL,standard,main,3783,0.841131,0.836373,0.014036,0.538075,0.224080,0.843971,0.484482,0.752462
1,case_A,BayesMMRL,standard,rep,3783,0.833201,0.825847,0.083516,0.806571,0.258951,0.916717,0.233334,0.858316
2,case_A,BayesMMRL,standard,fusion,3783,0.864922,0.860977,0.025963,0.475376,0.196878,0.888384,0.337859,0.818028
3,case_A,BayesMMRL,posterior_mean,main,3783,0.837695,0.833658,0.013065,0.558748,0.231251,0.848280,0.467957,0.757990
4,case_A,BayesMMRL,posterior_mean,rep,3783,0.816019,0.808462,0.099672,0.961561,0.292312,0.915134,0.231695,0.854526
5,case_A,BayesMMRL,posterior_mean,fusion,3783,0.854613,0.850868,0.038423,0.517263,0.212861,0.892743,0.321036,0.823744
6,case_A,BayesMMRL,mc_prob_mean_probs,main,3783,0.842717,0.838275,0.015714,0.533187,0.222596,0.832525,0.521425,0.736261
7,case_A,BayesMMRL,mc_prob_mean_probs,rep,3783,0.833994,0.826691,0.041594,0.693402,0.242555,0.875381,0.356529,0.797303
8,case_A,BayesMMRL,mc_prob_mean_probs,fusion,3783,0.869151,0.865782,0.014570,0.460725,0.193151,0.876283,0.375847,0.800326
9,case_A,BayesMMRL,mc_logit_mean,main,3783,0.841924,0.837391,0.010391,0.535798,0.222611,0.843233,0.486424,0.751217


## 9. Branch disagreement 与 fusion rescue/damage


In [40]:
def get_preds_for_mode(out, mode_name, branch):
    modes = eval_modes_for_case(out)
    obj = modes[mode_name][branch]
    if mode_name.endswith('_probs'):
        return obj.argmax(dim=-1)
    if mode_name.endswith('_pred'):
        return obj.long()
    return obj.argmax(dim=-1)


def branch_disagreement_table(case_outputs):
    rows = []
    for name, out in case_outputs.items():
        modes = eval_modes_for_case(out)
        for mode_name, branches in modes.items():
            if not all(br in branches for br in ['main', 'rep', 'fusion']):
                continue
            p_main = get_preds_for_mode(out, mode_name, 'main')
            p_rep = get_preds_for_mode(out, mode_name, 'rep')
            p_fusion = get_preds_for_mode(out, mode_name, 'fusion')
            rows.append({
                'case_name': name,
                'method': out['cfg_method'],
                'eval_mode': mode_name,
                'logits_rep_agree': float(p_main.eq(p_rep).float().mean().cpu()),
                'logits_fusion_agree': float(p_main.eq(p_fusion).float().mean().cpu()),
                'rep_fusion_agree': float(p_rep.eq(p_fusion).float().mean().cpu()),
                'all_three_agree': float((p_main.eq(p_rep) & p_main.eq(p_fusion)).float().mean().cpu()),
            })
    return pd.DataFrame(rows)


def fusion_rescue_damage_table(case_outputs):
    rows = []
    for name, out in case_outputs.items():
        labels = out['labels']
        modes = eval_modes_for_case(out)
        for mode_name, branches in modes.items():
            if not all(br in branches for br in ['main', 'rep', 'fusion']):
                continue
            p_main = get_preds_for_mode(out, mode_name, 'main')
            p_rep = get_preds_for_mode(out, mode_name, 'rep')
            p_fusion = get_preds_for_mode(out, mode_name, 'fusion')
            c_main = p_main.eq(labels)
            c_rep = p_rep.eq(labels)
            c_fusion = p_fusion.eq(labels)
            n = labels.numel()
            rows.append({
                'case_name': name,
                'method': out['cfg_method'],
                'eval_mode': mode_name,
                'n': int(n),
                'all_correct': int((c_main & c_rep & c_fusion).sum().item()),
                'all_wrong': int((~c_main & ~c_rep & ~c_fusion).sum().item()),
                'fusion_correct_others_wrong': int((c_fusion & ~c_main & ~c_rep).sum().item()),
                'fusion_wrong_logits_correct': int((~c_fusion & c_main).sum().item()),
                'fusion_wrong_rep_correct': int((~c_fusion & c_rep).sum().item()),
                'fusion_damage_any': int((~c_fusion & (c_main | c_rep)).sum().item()),
                'fusion_rescue_any': int((c_fusion & (~c_main | ~c_rep)).sum().item()),
            })
    return pd.DataFrame(rows)

branch_disagreement_df = branch_disagreement_table(case_outputs)
fusion_rescue_damage_df = fusion_rescue_damage_table(case_outputs)

display(branch_disagreement_df)
display(fusion_rescue_damage_df)
branch_disagreement_df.to_csv(SAVE_DIR / 'table_branch_disagreement.csv', index=False)
fusion_rescue_damage_df.to_csv(SAVE_DIR / 'table_fusion_rescue_damage.csv', index=False)


,case_name,method,eval_mode,logits_rep_agree,logits_fusion_agree,rep_fusion_agree,all_three_agree
0,case_A,BayesMMRL,standard,0.816019,0.918054,0.885805,0.816019
1,case_A,BayesMMRL,posterior_mean,0.816548,0.916997,0.883426,0.816548
2,case_A,BayesMMRL,mc_prob_mean_probs,0.822099,0.916468,0.891885,0.822099
3,case_A,BayesMMRL,mc_logit_mean,0.823156,0.916204,0.892413,0.823156
4,case_A,BayesMMRL,mc_majority_vote_pred,0.820513,0.915940,0.889506,0.820248
5,case_B,MMRL,standard,0.833466,0.929157,0.889770,0.833466


,case_name,method,eval_mode,n,all_correct,all_wrong,fusion_correct_others_wrong,fusion_wrong_logits_correct,fusion_wrong_rep_correct,fusion_damage_any,fusion_rescue_any
0,case_A,BayesMMRL,standard,3783,2911,344,16,71,96,167,361
1,case_A,BayesMMRL,posterior_mean,3783,2873,378,22,87,85,172,360
2,case_A,BayesMMRL,mc_prob_mean_probs,3783,2923,346,17,69,80,149,365
3,case_A,BayesMMRL,mc_logit_mean,3783,2927,346,19,69,81,150,360
4,case_A,BayesMMRL,mc_majority_vote_pred,3783,2916,345,22,72,83,154,368
5,case_B,MMRL,standard,3783,2964,352,11,76,50,126,341


## 10. 自动配对 MMRL vs BayesMMRL

按日志解析出的 `dataset / shots / seed / backbone / protocol / split` 配对。若一个组里有多个 MMRL 或多个 BayesMMRL，会取第一个；你也可以手动改 `PAIRINGS`。


In [41]:
def case_group_key(out):
    m = out['meta']
    return (
        str(m['dataset']).lower(),
        int(m['shots']),
        int(m['seed']),
        str(m['backbone']),
        str(m['protocol']),
        SPLIT,
    )


def auto_pair_cases(case_outputs):
    groups = defaultdict(list)
    for name, out in case_outputs.items():
        groups[case_group_key(out)].append(name)
    pairs = []
    for key, names in groups.items():
        det = [n for n in names if not case_outputs[n]['is_bayes']]
        bayes = [n for n in names if case_outputs[n]['is_bayes']]
        for d in det[:1]:
            for b in bayes[:1]:
                pairs.append({'pair_name': f'{d}__VS__{b}', 'mmrl': d, 'bayes': b, 'group_key': key})
    return pairs

PAIRINGS = auto_pair_cases(case_outputs)
print('PAIRINGS:')
for p in PAIRINGS:
    print(p)


PAIRINGS:
{'pair_name': 'case_B__VS__case_A', 'mmrl': 'case_B', 'bayes': 'case_A', 'group_key': ('ucf101', 16, 1, 'ViT-B/16', 'FS', 'test')}


## 11. MMRL vs Bayes paired outcome / probability distribution shift / feature shift


In [42]:
def assert_same_samples(a, b):
    if a['sample_ids'] != b['sample_ids']:
        warnings.warn('sample_ids 不完全一致，将按行号配对；请确认 dataloader 顺序一致。')
    if not torch.equal(a['labels'], b['labels']):
        raise ValueError('paired cases 的 labels 不一致，不能安全做样本级 paired outcome。')


def probs_for_case_mode_branch(out, mode_name='standard', branch='fusion'):
    modes = eval_modes_for_case(out)
    obj = modes[mode_name][branch]
    if mode_name.endswith('_probs'):
        return obj
    if mode_name.endswith('_pred'):
        raise ValueError('prediction-only mode has no probabilities')
    return probs_from_logits(obj)


def logits_for_case_mode_branch(out, mode_name='standard', branch='fusion'):
    modes = eval_modes_for_case(out)
    obj = modes[mode_name][branch]
    if mode_name.endswith('_probs'):
        return obj.clamp_min(1e-12).log()
    if mode_name.endswith('_pred'):
        raise ValueError('prediction-only mode has no logits')
    return obj


def paired_outcome_for_pair(mmrl_out, bayes_out, bayes_mode='mc_prob_mean_probs', branch='fusion'):
    assert_same_samples(mmrl_out, bayes_out)
    labels = mmrl_out['labels']
    p_m = probs_for_case_mode_branch(mmrl_out, 'standard', branch)
    p_b = probs_for_case_mode_branch(bayes_out, bayes_mode, branch)
    pred_m = p_m.argmax(dim=-1)
    pred_b = p_b.argmax(dim=-1)
    c_m = pred_m.eq(labels)
    c_b = pred_b.eq(labels)
    case_type = np.empty(labels.numel(), dtype=object)
    case_type[np.logical_and(c_m.numpy(), c_b.numpy())] = 'both_correct'
    case_type[np.logical_and(c_m.numpy(), ~c_b.numpy())] = 'MMRL_correct_Bayes_wrong'
    case_type[np.logical_and(~c_m.numpy(), c_b.numpy())] = 'MMRL_wrong_Bayes_correct'
    case_type[np.logical_and(~c_m.numpy(), ~c_b.numpy())] = 'both_wrong'

    pred_m2, conf_m, margin_m, ent_m = pred_conf_margin_entropy_from_probs(p_m)
    pred_b2, conf_b, margin_b, ent_b = pred_conf_margin_entropy_from_probs(p_b)
    js = js_divergence_probs(p_m, p_b)
    skl = sym_kl_probs(p_m, p_b)
    l1 = l1_probs(p_m, p_b)

    rows = []
    for ct in ['both_correct', 'MMRL_correct_Bayes_wrong', 'MMRL_wrong_Bayes_correct', 'both_wrong']:
        mask_np = case_type == ct
        mask = torch.tensor(mask_np, dtype=torch.bool)
        if mask.sum() == 0:
            rows.append({'case_type': ct, 'count': 0, 'fraction': 0.0})
            continue
        rows.append({
            'case_type': ct,
            'count': int(mask.sum().item()),
            'fraction': float(mask.float().mean().item()),
            'MMRL_conf': float(conf_m[mask].mean().item()),
            'Bayes_conf': float(conf_b[mask].mean().item()),
            'MMRL_entropy': float(ent_m[mask].mean().item()),
            'Bayes_entropy': float(ent_b[mask].mean().item()),
            'MMRL_margin': float(margin_m[mask].mean().item()),
            'Bayes_margin': float(margin_b[mask].mean().item()),
            'JS': float(js[mask].mean().item()),
            'sym_KL': float(skl[mask].mean().item()),
            'L1': float(l1[mask].mean().item()),
        })
    sample_df = pd.DataFrame({
        'sample_id': mmrl_out['sample_ids'],
        'label': labels.numpy(),
        'MMRL_pred': pred_m.numpy(),
        'Bayes_pred': pred_b.numpy(),
        'case_type': case_type,
        'MMRL_conf': conf_m.numpy(),
        'Bayes_conf': conf_b.numpy(),
        'MMRL_entropy': ent_m.numpy(),
        'Bayes_entropy': ent_b.numpy(),
        'JS': js.numpy(),
        'sym_KL': skl.numpy(),
        'L1': l1.numpy(),
    })
    return pd.DataFrame(rows), sample_df

paired_summary_tables = []
paired_sample_tables = {}
for pair in PAIRINGS:
    mmrl_out = case_outputs[pair['mmrl']]
    bayes_out = case_outputs[pair['bayes']]
    for bayes_mode in ['posterior_mean', 'mc_prob_mean_probs', 'mc_logit_mean']:
        if bayes_mode not in eval_modes_for_case(bayes_out):
            continue
        summ, sample_df = paired_outcome_for_pair(mmrl_out, bayes_out, bayes_mode=bayes_mode, branch='fusion')
        summ.insert(0, 'pair_name', pair['pair_name'])
        summ.insert(1, 'bayes_mode', bayes_mode)
        paired_summary_tables.append(summ)
        paired_sample_tables[(pair['pair_name'], bayes_mode)] = sample_df
        sample_df.to_csv(SAVE_DIR / f"sample_paired_outcome__{pair['pair_name']}__{bayes_mode}.csv", index=False)

paired_outcome_df = pd.concat(paired_summary_tables, ignore_index=True) if paired_summary_tables else pd.DataFrame()
display(paired_outcome_df)
paired_outcome_df.to_csv(SAVE_DIR / 'table_mmrl_vs_bayes_paired_outcome.csv', index=False)


,pair_name,bayes_mode,case_type,count,fraction,MMRL_conf,Bayes_conf,MMRL_entropy,Bayes_entropy,MMRL_margin,Bayes_margin,JS,sym_KL,L1
0,case_B__VS__case_A,posterior_mean,both_correct,3144,0.831086,0.943744,0.941577,0.197586,0.187769,0.905436,0.899562,0.012129,0.063706,0.096201
1,case_B__VS__case_A,posterior_mean,MMRL_correct_Bayes_wrong,161,0.042559,0.651450,0.610955,0.961927,0.991473,0.428263,0.347111,0.168656,0.826739,0.982062
2,case_B__VS__case_A,posterior_mean,MMRL_wrong_Bayes_correct,89,0.023526,0.512647,0.616499,1.273754,1.047774,0.231337,0.396437,0.153565,0.762559,0.906068
3,case_B__VS__case_A,posterior_mean,both_wrong,389,0.102828,0.633171,0.677886,1.116328,0.954379,0.450723,0.505995,0.096039,0.505310,0.576933
4,case_B__VS__case_A,mc_prob_mean_probs,both_correct,3190,0.843246,0.940610,0.926971,0.207345,0.237320,0.900696,0.877128,0.011619,0.056732,0.100249
5,case_B__VS__case_A,mc_prob_mean_probs,MMRL_correct_Bayes_wrong,115,0.030399,0.621487,0.569120,0.996963,1.073672,0.368886,0.277750,0.115392,0.525012,0.808143
6,case_B__VS__case_A,mc_prob_mean_probs,MMRL_wrong_Bayes_correct,98,0.025905,0.520417,0.548755,1.287406,1.249799,0.248527,0.308995,0.125569,0.587227,0.832078
7,case_B__VS__case_A,mc_prob_mean_probs,both_wrong,380,0.100449,0.634022,0.628200,1.109079,1.102176,0.451486,0.440453,0.074012,0.361588,0.503580
8,case_B__VS__case_A,mc_logit_mean,both_correct,3187,0.842453,0.940750,0.936967,0.206523,0.204628,0.900816,0.892428,0.010674,0.052472,0.092625
9,case_B__VS__case_A,mc_logit_mean,MMRL_correct_Bayes_wrong,118,0.031192,0.625802,0.589269,0.999102,1.027778,0.379167,0.306866,0.127573,0.589667,0.851472


In [43]:
def feature_shift_tables_for_pairs(pairings, case_outputs):
    rows = []
    for pair in pairings:
        m = case_outputs[pair['mmrl']]
        b = case_outputs[pair['bayes']]
        assert_same_samples(m, b)
        labels = m['labels']
        # Need paired case type from prob_mean if available
        _, sample_df = paired_outcome_for_pair(m, b, bayes_mode='mc_prob_mean_probs', branch='fusion')
        for feat_key, pretty in [('features_img', 'image'), ('features_text', 'text')]:
            if feat_key not in m['standard_features'] or feat_key not in b.get('posterior_mean_features', {}):
                continue
            fm = m['standard_features'][feat_key]
            fb = b['posterior_mean_features'][feat_key]
            # text features are [C,D], not sample-level. Treat separately.
            if fm.shape[0] != labels.shape[0]:
                rows.append({
                    'pair_name': pair['pair_name'],
                    'feature': pretty,
                    'case_type': 'class_level_or_global',
                    'n': int(fm.shape[0]),
                    'cosine_mean': float(cosine_per_row(fm, fb).mean().item()),
                    'shift_norm_mean': float((fb - fm).norm(dim=-1).mean().item()),
                })
                continue
            cos = cosine_per_row(fm, fb)
            shift = (fb - fm).norm(dim=-1)
            for ct in sorted(sample_df['case_type'].unique()):
                mask = torch.tensor((sample_df['case_type'].values == ct), dtype=torch.bool)
                rows.append({
                    'pair_name': pair['pair_name'],
                    'feature': pretty,
                    'case_type': ct,
                    'n': int(mask.sum().item()),
                    'cosine_mean': float(cos[mask].mean().item()) if mask.any() else np.nan,
                    'shift_norm_mean': float(shift[mask].mean().item()) if mask.any() else np.nan,
                })
    return pd.DataFrame(rows)

feature_shift_df = feature_shift_tables_for_pairs(PAIRINGS, case_outputs)
display(feature_shift_df)
feature_shift_df.to_csv(SAVE_DIR / 'table_feature_shift.csv', index=False)


,pair_name,feature,case_type,n,cosine_mean,shift_norm_mean
0,case_B__VS__case_A,image,MMRL_correct_Bayes_wrong,115,0.945435,0.326927
1,case_B__VS__case_A,image,MMRL_wrong_Bayes_correct,98,0.944975,0.329152
2,case_B__VS__case_A,image,both_correct,3190,0.951148,0.308697
3,case_B__VS__case_A,image,both_wrong,380,0.947035,0.322024
4,case_B__VS__case_A,text,class_level_or_global,1616,0.946649,0.322012


## 8A. 特征差异增强分析：BayesMMRL 与 MMRL 的表示是否真的不同

原始 `feature_shift_df` 只给出不同样本类型下的平均余弦相似度和平均偏移范数，不能充分回答“BayesMMRL 与 MMRL 的特征是否有区别”。

这里新增更完整的特征差异分析：

1. `feature_difference_sample_df`：逐样本特征差异表，包含余弦相似度、角度、偏移范数、预测变化、概率分布差异。
2. `feature_shift_distribution_df`：按样本类型统计特征偏移的均值、标准差、分位数。
3. `feature_shift_effect_df`：相对 `both_correct` 的差异量、比例、Cohen's d，用于判断不同样本类型的偏移是否更大。
4. `feature_shift_error_detection_df`：检验特征偏移能否识别 Bayes 错误、有害翻转、有益翻转。
5. `feature_shift_correlation_df`：检验特征偏移与 JS / L1 / 熵 / 互信息等是否相关。
6. `top_feature_shift_samples_df`：列出特征偏移最大的样本，方便做 case study。

注意：这部分回答的是“BayesMMRL 是否改变了表示、改变集中在哪里”，而不是直接证明 Bayes 一定更好。

## 8B/15C. 采样特征、各分支类别打分、最终融合类别打分与冲突分析

这一节专门回答四个问题：

1. **采样得到的特征是否有差异？** 不同蒙特卡洛采样得到的图像/文本特征是否彼此不同、是否偏离后验均值、是否偏离 MMRL 特征。
2. **各分支是否有差异？** 主分支、表示增强分支、融合分支的类别打分和概率分布相对 MMRL 是否改变。
3. **最后的融合类别打分是否有差异？** BayesMMRL 的最终融合类别打分与 MMRL 的最终融合类别打分有多大差异，是否导致预测改变。
4. **是否有冲突？** 包括采样内部冲突、分支之间冲突、最终融合在冲突样本上跟随哪一侧，以及哪些样本发生了冲突。

注意：要分析采样特征，配置区需要 `COLLECT_MC_FEATURES = True`，否则只能分析采样类别打分，不能分析采样特征。本 v9 已默认改成 True。

In [44]:

# ============================================================
# 8B/15C. 采样特征、分支类别打分、最终融合类别打分与冲突分析
# ============================================================

def _to_np(x):
    if torch.is_tensor(x):
        return x.detach().float().cpu().numpy()
    return np.asarray(x)

def _safe_mean(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    return float(np.mean(x)) if len(x) else np.nan

def _safe_std(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    return float(np.std(x, ddof=0)) if len(x) else np.nan

def _safe_quantile(x, q):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    return float(np.quantile(x, q)) if len(x) else np.nan

def _summ(v, prefix):
    return {
        f"{prefix}_mean": _safe_mean(v),
        f"{prefix}_std": _safe_std(v),
        f"{prefix}_q10": _safe_quantile(v, 0.10),
        f"{prefix}_median": _safe_quantile(v, 0.50),
        f"{prefix}_q90": _safe_quantile(v, 0.90),
        f"{prefix}_max": float(np.nanmax(v)) if np.asarray(v).size else np.nan,
    }

def _probs_from_logits(x):
    return torch.softmax(x.float(), dim=-1)

def _prob_js_l1(p, q, eps=1e-12):
    p = p.float().clamp_min(eps)
    q = q.float().clamp_min(eps)
    m = 0.5 * (p + q)
    kl_pm = (p * (p.log() - m.log())).sum(dim=-1)
    kl_qm = (q * (q.log() - m.log())).sum(dim=-1)
    js = 0.5 * (kl_pm + kl_qm)
    l1 = (p - q).abs().sum(dim=-1)
    return js, l1

def _logit_stats_between(a, b, labels=None):
    # a, b: [N,C]
    out = {}
    a = a.float()
    b = b.float()
    out.update(_summ((b - a).norm(dim=-1).detach().cpu().numpy(), "logit_l2"))
    out.update(_summ(torch.nn.functional.cosine_similarity(a, b, dim=-1).detach().cpu().numpy(), "logit_cosine"))

    pa = _probs_from_logits(a)
    pb = _probs_from_logits(b)
    js, l1 = _prob_js_l1(pa, pb)
    out.update(_summ(js.detach().cpu().numpy(), "prob_JS"))
    out.update(_summ(l1.detach().cpu().numpy(), "prob_L1"))

    pred_a = a.argmax(dim=-1)
    pred_b = b.argmax(dim=-1)
    out["pred_changed_rate"] = float(pred_a.ne(pred_b).float().mean().item())

    conf_a = pa.max(dim=-1).values
    conf_b = pb.max(dim=-1).values
    out.update(_summ((conf_b - conf_a).detach().cpu().numpy(), "confidence_delta"))

    if labels is not None:
        labels = labels.long()
        top2_a = pa.topk(k=2, dim=-1).values
        top2_b = pb.topk(k=2, dim=-1).values
        margin_a = top2_a[:, 0] - top2_a[:, 1]
        margin_b = top2_b[:, 0] - top2_b[:, 1]
        out.update(_summ((margin_b - margin_a).detach().cpu().numpy(), "prob_margin_delta"))
        out["acc_a"] = float(pred_a.eq(labels).float().mean().item())
        out["acc_b"] = float(pred_b.eq(labels).float().mean().item())
        out["acc_delta_b_minus_a"] = out["acc_b"] - out["acc_a"]

    return out

def _paired_case_type(mmrl_pred, bayes_pred, labels):
    m_ok = mmrl_pred.eq(labels)
    b_ok = bayes_pred.eq(labels)
    case = np.full(labels.numel(), "both_wrong", dtype=object)
    case[m_ok.cpu().numpy() & b_ok.cpu().numpy()] = "both_correct"
    case[m_ok.cpu().numpy() & (~b_ok).cpu().numpy()] = "MMRL_correct_Bayes_wrong"
    case[(~m_ok).cpu().numpy() & b_ok.cpu().numpy()] = "MMRL_wrong_Bayes_correct"
    return case

def build_mc_sampled_feature_difference_tables(case_outputs, pairings, feature_keys=("features_img", "features_text")):
    """
    分析 BayesMMRL 不同采样得到的特征是否有差异。
    需要 case_outputs[bayes]['mc_sample_features'] 存在，即 COLLECT_MC_FEATURES=True。
    """
    group_rows = []
    sample_rows = []

    for pair in pairings:
        pair_name = pair["pair_name"]
        m = case_outputs[pair["mmrl"]]
        b = case_outputs[pair["bayes"]]
        labels = m["labels"]
        assert_same_samples(m, b)

        if "mc_sample_features" not in b:
            group_rows.append({
                "pair_name": pair_name,
                "feature": "NA",
                "case_type": "NA",
                "n": 0,
                "note": "缺少 mc_sample_features。请把配置区 COLLECT_MC_FEATURES=True，并从 forward collect 开始重新运行。",
            })
            continue

        # 用融合分支的 MMRL vs Bayes MC 概率均值来定义 case_type
        p_m = probs_for_case_mode_branch(m, "standard", "fusion")
        p_b = probs_for_case_mode_branch(b, "mc_prob_mean_probs", "fusion")
        pred_m = p_m.argmax(dim=-1)
        pred_b = p_b.argmax(dim=-1)
        case_types = _paired_case_type(pred_m, pred_b, labels)

        for fk in feature_keys:
            if fk not in b["mc_sample_features"]:
                continue

            fs = b["mc_sample_features"][fk].float()  # [S,N,D] or [S,K,D]
            S = fs.shape[0]
            is_sample_level = fs.shape[1] == labels.numel()

            f_mean = fs.mean(dim=0)
            sample_dev = (fs - f_mean.unsqueeze(0)).norm(dim=-1)  # [S,N]
            sample_dev_mean = sample_dev.mean(dim=0)
            sample_dev_std = sample_dev.std(dim=0, unbiased=False)

            # cosine between each sampled feature and MC mean feature
            cos_to_mean = torch.nn.functional.cosine_similarity(
                fs.reshape(-1, fs.shape[-1]),
                f_mean.unsqueeze(0).expand(S, *f_mean.shape).reshape(-1, fs.shape[-1]),
                dim=-1,
            ).reshape(S, fs.shape[1])
            cos_to_mean_mean = cos_to_mean.mean(dim=0)

            # distance to posterior mean feature if available
            dist_to_pm = None
            cos_to_pm = None
            if "posterior_mean_features" in b and fk in b["posterior_mean_features"]:
                fpm = b["posterior_mean_features"][fk].float()
                if fpm.shape == f_mean.shape:
                    dist_to_pm = (f_mean - fpm).norm(dim=-1)
                    cos_to_pm = torch.nn.functional.cosine_similarity(f_mean, fpm, dim=-1)

            # distance to MMRL standard feature if available
            dist_to_mmrl = None
            cos_to_mmrl = None
            if "standard_features" in m and fk in m["standard_features"]:
                fm = m["standard_features"][fk].float()
                if fm.shape == f_mean.shape:
                    dist_to_mmrl = (f_mean - fm).norm(dim=-1)
                    cos_to_mmrl = torch.nn.functional.cosine_similarity(f_mean, fm, dim=-1)

            if is_sample_level:
                for i in range(labels.numel()):
                    sample_rows.append({
                        "pair_name": pair_name,
                        "feature": fk,
                        "sample_index": i,
                        "sample_id": m["sample_ids"][i],
                        "label": int(labels[i].item()),
                        "case_type": case_types[i],
                        "mc_feature_dev_mean": float(sample_dev_mean[i].item()),
                        "mc_feature_dev_std": float(sample_dev_std[i].item()),
                        "mc_feature_cos_to_mc_mean": float(cos_to_mean_mean[i].item()),
                        "mc_mean_dist_to_posterior_mean": float(dist_to_pm[i].item()) if dist_to_pm is not None else np.nan,
                        "mc_mean_cos_to_posterior_mean": float(cos_to_pm[i].item()) if cos_to_pm is not None else np.nan,
                        "mc_mean_dist_to_MMRL": float(dist_to_mmrl[i].item()) if dist_to_mmrl is not None else np.nan,
                        "mc_mean_cos_to_MMRL": float(cos_to_mmrl[i].item()) if cos_to_mmrl is not None else np.nan,
                    })

                df_tmp = pd.DataFrame(sample_rows)
                # group only latest feature/pair rows
                gdf = df_tmp[(df_tmp["pair_name"] == pair_name) & (df_tmp["feature"] == fk)]
                for ct, g in gdf.groupby("case_type", dropna=False):
                    row = {
                        "pair_name": pair_name,
                        "feature": fk,
                        "case_type": ct,
                        "n": len(g),
                        "note": "",
                    }
                    for col in [
                        "mc_feature_dev_mean",
                        "mc_feature_dev_std",
                        "mc_feature_cos_to_mc_mean",
                        "mc_mean_dist_to_posterior_mean",
                        "mc_mean_cos_to_posterior_mean",
                        "mc_mean_dist_to_MMRL",
                        "mc_mean_cos_to_MMRL",
                    ]:
                        row.update(_summ(g[col].values, col))
                    group_rows.append(row)

            else:
                # text/class-level features are not per test sample.
                row = {
                    "pair_name": pair_name,
                    "feature": fk,
                    "case_type": "class_or_text_level",
                    "n": int(fs.shape[1]),
                    "note": "该特征不是逐测试样本维度，按类别/文本原型维度统计。",
                }
                row.update(_summ(sample_dev_mean.detach().cpu().numpy(), "mc_feature_dev_mean"))
                row.update(_summ(sample_dev_std.detach().cpu().numpy(), "mc_feature_dev_std"))
                row.update(_summ(cos_to_mean_mean.detach().cpu().numpy(), "mc_feature_cos_to_mc_mean"))
                if dist_to_pm is not None:
                    row.update(_summ(dist_to_pm.detach().cpu().numpy(), "mc_mean_dist_to_posterior_mean"))
                    row.update(_summ(cos_to_pm.detach().cpu().numpy(), "mc_mean_cos_to_posterior_mean"))
                if dist_to_mmrl is not None:
                    row.update(_summ(dist_to_mmrl.detach().cpu().numpy(), "mc_mean_dist_to_MMRL"))
                    row.update(_summ(cos_to_mmrl.detach().cpu().numpy(), "mc_mean_cos_to_MMRL"))
                group_rows.append(row)

    return pd.DataFrame(group_rows), pd.DataFrame(sample_rows)


def build_branch_logit_difference_tables(case_outputs, pairings, bayes_modes=("posterior_mean", "mc_prob_mean_logits", "mc_logit_mean"), branches=("main", "rep", "fusion")):
    """
    比较 MMRL 与 BayesMMRL 在各分支的类别打分/概率分布差异。
    """
    rows = []
    sample_rows = []

    for pair in pairings:
        pair_name = pair["pair_name"]
        m = case_outputs[pair["mmrl"]]
        b = case_outputs[pair["bayes"]]
        labels = m["labels"]
        assert_same_samples(m, b)

        for br in branches:
            if br not in m["standard"]:
                continue
            lm = m["standard"][br].float()
            pm = _probs_from_logits(lm)
            pred_m = lm.argmax(dim=-1)

            # Build Bayes logits for each requested mode.
            bayes_logits_by_mode = {}
            if "posterior_mean" in bayes_modes and br in b.get("posterior_mean", {}):
                bayes_logits_by_mode["posterior_mean"] = b["posterior_mean"][br].float()
            if "mc_logit_mean" in bayes_modes and br in b.get("mc_samples", {}):
                bayes_logits_by_mode["mc_logit_mean"] = b["mc_samples"][br].float().mean(dim=0)
            if "mc_prob_mean_logits" in bayes_modes and br in b.get("mc_samples", {}):
                # 注意：概率均值本身不是 logit，这里保存 log(prob_mean) 作为可比较的概率空间代表。
                prob_mean = torch.softmax(b["mc_samples"][br].float(), dim=-1).mean(dim=0).clamp_min(1e-12)
                bayes_logits_by_mode["mc_prob_mean_logits"] = prob_mean.log()

            for mode, lb in bayes_logits_by_mode.items():
                stats = _logit_stats_between(lm, lb, labels=labels)
                stats.update({
                    "pair_name": pair_name,
                    "bayes_mode": mode,
                    "branch": br,
                    "branch_cn": {"main": "主分支", "rep": "表示增强分支", "fusion": "融合分支"}.get(br, br),
                })
                rows.append(stats)

                pb = _probs_from_logits(lb)
                pred_b = lb.argmax(dim=-1)
                case_types = _paired_case_type(pred_m, pred_b, labels)
                js, l1 = _prob_js_l1(pm, pb)

                logit_l2 = (lb - lm).norm(dim=-1)
                logit_cos = torch.nn.functional.cosine_similarity(lm, lb, dim=-1)
                for i in range(labels.numel()):
                    sample_rows.append({
                        "pair_name": pair_name,
                        "bayes_mode": mode,
                        "branch": br,
                        "sample_index": i,
                        "sample_id": m["sample_ids"][i],
                        "label": int(labels[i].item()),
                        "MMRL_pred": int(pred_m[i].item()),
                        "Bayes_pred": int(pred_b[i].item()),
                        "case_type": case_types[i],
                        "pred_changed": bool(pred_m[i].ne(pred_b[i]).item()),
                        "logit_l2": float(logit_l2[i].item()),
                        "logit_cosine": float(logit_cos[i].item()),
                        "prob_JS": float(js[i].item()),
                        "prob_L1": float(l1[i].item()),
                        "MMRL_conf": float(pm[i].max().item()),
                        "Bayes_conf": float(pb[i].max().item()),
                    })

    return pd.DataFrame(rows), pd.DataFrame(sample_rows)


def build_mc_logit_variation_by_branch(case_outputs, pairings, branches=("main", "rep", "fusion")):
    """
    分析 BayesMMRL 内部不同采样得到的各分支类别打分是否有差异。
    """
    rows = []
    sample_rows = []

    for pair in pairings:
        pair_name = pair["pair_name"]
        b = case_outputs[pair["bayes"]]
        labels = b["labels"]

        for br in branches:
            if br not in b.get("mc_samples", {}):
                continue
            ls = b["mc_samples"][br].float()  # [S,N,C]
            S, N, C = ls.shape
            lm = ls.mean(dim=0)
            ps = torch.softmax(ls, dim=-1)
            pm = ps.mean(dim=0)
            pred_s = ls.argmax(dim=-1)  # [S,N]
            pred_mode = torch.mode(pred_s, dim=0).values
            pred_pm = pm.argmax(dim=-1)

            # logit std across samples
            logit_std_mean_per_sample = ls.std(dim=0, unbiased=False).mean(dim=-1)  # [N]
            logit_l2_to_mean = (ls - lm.unsqueeze(0)).norm(dim=-1).mean(dim=0)       # [N]
            prob_l1_to_mean = (ps - pm.unsqueeze(0)).abs().sum(dim=-1).mean(dim=0)   # [N]

            # sample agreement
            agreements = []
            unique_counts = []
            for i in range(N):
                vals, cnts = torch.unique(pred_s[:, i], return_counts=True)
                agreements.append(float(cnts.max().item()) / float(S))
                unique_counts.append(int(vals.numel()))
            agreements = np.asarray(agreements)
            unique_counts = np.asarray(unique_counts)

            # uncertainty
            u = predictive_uncertainty_from_mc_logits(ls)

            row = {
                "pair_name": pair_name,
                "branch": br,
                "branch_cn": {"main": "主分支", "rep": "表示增强分支", "fusion": "融合分支"}.get(br, br),
                "n_samples": int(S),
                "n_test": int(N),
            }
            row.update(_summ(logit_std_mean_per_sample.detach().cpu().numpy(), "mc_logit_std_mean"))
            row.update(_summ(logit_l2_to_mean.detach().cpu().numpy(), "mc_logit_l2_to_mean"))
            row.update(_summ(prob_l1_to_mean.detach().cpu().numpy(), "mc_prob_l1_to_mean"))
            row.update(_summ(agreements, "sample_agreement"))
            row.update(_summ(unique_counts, "num_unique_preds"))
            row.update(_summ(u["mutual_info"].detach().cpu().numpy(), "mutual_information"))
            row.update(_summ(u["variation_ratio"].detach().cpu().numpy(), "variation_ratio"))
            rows.append(row)

            correct = pred_pm.eq(labels)
            for i in range(N):
                sample_rows.append({
                    "pair_name": pair_name,
                    "branch": br,
                    "sample_index": i,
                    "sample_id": b["sample_ids"][i],
                    "label": int(labels[i].item()),
                    "pred_mc_prob_mean": int(pred_pm[i].item()),
                    "correct_mc_prob_mean": bool(correct[i].item()),
                    "mc_logit_std_mean": float(logit_std_mean_per_sample[i].item()),
                    "mc_logit_l2_to_mean": float(logit_l2_to_mean[i].item()),
                    "mc_prob_l1_to_mean": float(prob_l1_to_mean[i].item()),
                    "sample_agreement": float(agreements[i]),
                    "num_unique_preds": int(unique_counts[i]),
                    "mutual_information": float(u["mutual_info"][i].item()),
                    "variation_ratio": float(u["variation_ratio"][i].item()),
                })

    return pd.DataFrame(rows), pd.DataFrame(sample_rows)


def build_conflict_sample_table(case_outputs, pairings):
    """
    输出具体冲突样本：主分支、表示增强分支、融合分支的预测是否一致；
    MMRL 和 Bayes 的最终预测是否改变；Bayes MC 采样是否冲突。
    """
    rows = []
    for pair in pairings:
        pair_name = pair["pair_name"]
        m = case_outputs[pair["mmrl"]]
        b = case_outputs[pair["bayes"]]
        labels = m["labels"]
        assert_same_samples(m, b)

        # Bayes MC prob mean probs for branches
        probs = {}
        preds = {}
        confs = {}
        for br in ["main", "rep", "fusion"]:
            probs[br] = probs_for_case_mode_branch(b, "mc_prob_mean_probs", br)
            preds[br] = probs[br].argmax(dim=-1)
            confs[br] = probs[br].max(dim=-1).values

        mmrl_fusion_probs = probs_for_case_mode_branch(m, "standard", "fusion")
        mmrl_fusion_pred = mmrl_fusion_probs.argmax(dim=-1)

        # MC sample info for fusion
        if "fusion" in b.get("mc_samples", {}):
            u_fusion = predictive_uncertainty_from_mc_logits(b["mc_samples"]["fusion"])
            pred_s = b["mc_samples"]["fusion"].argmax(dim=-1)
        else:
            u_fusion = None
            pred_s = None

        for i in range(labels.numel()):
            main_rep_conflict = preds["main"][i].item() != preds["rep"][i].item()
            main_fusion_conflict = preds["main"][i].item() != preds["fusion"][i].item()
            rep_fusion_conflict = preds["rep"][i].item() != preds["fusion"][i].item()
            all_three_agree = (not main_rep_conflict) and (not main_fusion_conflict) and (not rep_fusion_conflict)

            if pred_s is not None:
                vals, cnts = torch.unique(pred_s[:, i], return_counts=True)
                sample_agreement = float(cnts.max().item()) / float(pred_s.shape[0])
                num_unique_preds = int(vals.numel())
                mi = float(u_fusion["mutual_info"][i].item())
                vr = float(u_fusion["variation_ratio"][i].item())
            else:
                sample_agreement = np.nan
                num_unique_preds = np.nan
                mi = np.nan
                vr = np.nan

            rows.append({
                "pair_name": pair_name,
                "sample_index": i,
                "sample_id": m["sample_ids"][i],
                "label": int(labels[i].item()),
                "MMRL_fusion_pred": int(mmrl_fusion_pred[i].item()),
                "Bayes_main_pred": int(preds["main"][i].item()),
                "Bayes_rep_pred": int(preds["rep"][i].item()),
                "Bayes_fusion_pred": int(preds["fusion"][i].item()),
                "Bayes_fusion_correct": bool(preds["fusion"][i].eq(labels[i]).item()),
                "MMRL_fusion_correct": bool(mmrl_fusion_pred[i].eq(labels[i]).item()),
                "MMRL_vs_Bayes_fusion_changed": bool(mmrl_fusion_pred[i].ne(preds["fusion"][i]).item()),
                "main_rep_conflict": bool(main_rep_conflict),
                "main_fusion_conflict": bool(main_fusion_conflict),
                "rep_fusion_conflict": bool(rep_fusion_conflict),
                "all_three_agree": bool(all_three_agree),
                "Bayes_main_conf": float(confs["main"][i].item()),
                "Bayes_rep_conf": float(confs["rep"][i].item()),
                "Bayes_fusion_conf": float(confs["fusion"][i].item()),
                "fusion_sample_agreement": sample_agreement,
                "fusion_num_unique_mc_preds": num_unique_preds,
                "fusion_MI": mi,
                "fusion_variation_ratio": vr,
            })

    return pd.DataFrame(rows)


mc_sampled_feature_difference_df, mc_sampled_feature_sample_df = build_mc_sampled_feature_difference_tables(
    case_outputs, PAIRINGS
)
branch_logit_difference_df, branch_logit_difference_sample_df = build_branch_logit_difference_tables(
    case_outputs, PAIRINGS
)
mc_logit_variation_by_branch_df, mc_logit_variation_sample_df = build_mc_logit_variation_by_branch(
    case_outputs, PAIRINGS
)
conflict_sample_df = build_conflict_sample_table(case_outputs, PAIRINGS)

display(mc_sampled_feature_difference_df)
display(branch_logit_difference_df)
display(mc_logit_variation_by_branch_df)
display(conflict_sample_df.head(20))

# 保存 CSV
mc_sampled_feature_difference_df.to_csv(SAVE_DIR / "table_mc_sampled_feature_difference.csv", index=False)
mc_sampled_feature_sample_df.to_csv(SAVE_DIR / "table_mc_sampled_feature_sample.csv", index=False)
branch_logit_difference_df.to_csv(SAVE_DIR / "table_branch_logit_difference.csv", index=False)
branch_logit_difference_sample_df.to_csv(SAVE_DIR / "table_branch_logit_difference_sample.csv", index=False)
mc_logit_variation_by_branch_df.to_csv(SAVE_DIR / "table_mc_logit_variation_by_branch.csv", index=False)
mc_logit_variation_sample_df.to_csv(SAVE_DIR / "table_mc_logit_variation_sample.csv", index=False)
conflict_sample_df.to_csv(SAVE_DIR / "table_conflict_sample.csv", index=False)


,pair_name,feature,case_type,n,note,mc_feature_dev_mean_mean,mc_feature_dev_mean_std,mc_feature_dev_mean_q10,mc_feature_dev_mean_median,mc_feature_dev_mean_q90,mc_feature_dev_mean_max,mc_feature_dev_std_mean,mc_feature_dev_std_std,mc_feature_dev_std_q10,mc_feature_dev_std_median,mc_feature_dev_std_q90,mc_feature_dev_std_max,mc_feature_cos_to_mc_mean_mean,mc_feature_cos_to_mc_mean_std,mc_feature_cos_to_mc_mean_q10,mc_feature_cos_to_mc_mean_median,mc_feature_cos_to_mc_mean_q90,mc_feature_cos_to_mc_mean_max,mc_mean_dist_to_posterior_mean_mean,mc_mean_dist_to_posterior_mean_std,mc_mean_dist_to_posterior_mean_q10,mc_mean_dist_to_posterior_mean_median,mc_mean_dist_to_posterior_mean_q90,mc_mean_dist_to_posterior_mean_max,mc_mean_cos_to_posterior_mean_mean,mc_mean_cos_to_posterior_mean_std,mc_mean_cos_to_posterior_mean_q10,mc_mean_cos_to_posterior_mean_median,mc_mean_cos_to_posterior_mean_q90,mc_mean_cos_to_posterior_mean_max,mc_mean_dist_to_MMRL_mean,mc_mean_dist_to_MMRL_std,mc_mean_dist_to_MMRL_q10,mc_mean_dist_to_MMRL_median,mc_mean_dist_to_MMRL_q90,mc_mean_dist_to_MMRL_max,mc_mean_cos_to_MMRL_mean,mc_mean_cos_to_MMRL_std,mc_mean_cos_to_MMRL_q10,mc_mean_cos_to_MMRL_median,mc_mean_cos_to_MMRL_q90,mc_mean_cos_to_MMRL_max
0,case_B__VS__case_A,features_img,MMRL_correct_Bayes_wrong,115,,0.104357,0.012322,0.087929,0.103776,0.118990,0.139847,0.014951,0.003604,0.011236,0.014140,0.019482,0.027622,0.994344,0.001351,0.992602,0.994514,0.996041,0.996713,0.075830,0.014069,0.059999,0.075222,0.094360,0.117800,0.997025,0.001124,0.995545,0.997172,0.998204,0.998773,0.283762,0.041260,0.237480,0.275681,0.334773,0.410403,0.958662,0.012278,0.943563,0.961828,0.971692,0.979719
1,case_B__VS__case_A,features_img,MMRL_wrong_Bayes_correct,98,,0.105436,0.011828,0.090699,0.105910,0.118807,0.140450,0.015092,0.003789,0.010929,0.015035,0.018666,0.031890,0.994233,0.001307,0.992761,0.994284,0.995789,0.997165,0.076416,0.011721,0.064735,0.074389,0.091676,0.110793,0.997011,0.000958,0.995788,0.997232,0.997905,0.998545,0.285485,0.036888,0.240258,0.281836,0.341185,0.375593,0.958338,0.010903,0.941388,0.960058,0.971002,0.976541
2,case_B__VS__case_A,features_img,both_correct,3190,,0.094934,0.014345,0.076675,0.094354,0.113848,0.158323,0.013878,0.004279,0.009420,0.013249,0.018777,0.046630,0.995273,0.001436,0.993341,0.995440,0.997007,0.998665,0.069671,0.014788,0.052745,0.067832,0.088457,0.157249,0.997463,0.001163,0.996082,0.997700,0.998612,0.999423,0.268324,0.042291,0.217287,0.264313,0.322815,0.479565,0.962933,0.012021,0.947575,0.964914,0.976326,0.986996
3,case_B__VS__case_A,features_img,both_wrong,380,,0.103241,0.013870,0.086371,0.101969,0.121843,0.142393,0.015008,0.003752,0.010884,0.014544,0.020511,0.026028,0.994438,0.001498,0.992367,0.994679,0.996172,0.997506,0.074570,0.014087,0.057493,0.073321,0.092823,0.122485,0.997120,0.001108,0.995683,0.997316,0.998350,0.999037,0.279793,0.040258,0.229231,0.275459,0.334060,0.424714,0.959829,0.011856,0.943941,0.961877,0.973654,0.979285
4,case_B__VS__case_A,features_text,class_or_text_level,1616,该特征不是逐测试样本维度，按类别/文本原型维度统计。,0.119090,0.015411,0.102130,0.117263,0.138937,0.191353,0.013692,0.003056,0.009961,0.013405,0.017717,0.025341,0.992663,0.002043,0.990197,0.993027,0.994696,0.996206,0.055363,0.012156,0.043011,0.053134,0.069293,0.115428,0.998410,0.000797,0.997623,0.998606,0.999086,0.999373,0.299262,0.048620,0.239934,0.294017,0.367314,0.447129,0.953706,0.015694,0.932019,0.956517,0.971048,0.978163


,logit_l2_mean,logit_l2_std,logit_l2_q10,logit_l2_median,logit_l2_q90,logit_l2_max,logit_cosine_mean,logit_cosine_std,logit_cosine_q10,logit_cosine_median,logit_cosine_q90,logit_cosine_max,prob_JS_mean,prob_JS_std,prob_JS_q10,prob_JS_median,prob_JS_q90,prob_JS_max,prob_L1_mean,prob_L1_std,prob_L1_q10,prob_L1_median,prob_L1_q90,prob_L1_max,pred_changed_rate,confidence_delta_mean,confidence_delta_std,confidence_delta_q10,confidence_delta_median,confidence_delta_q90,confidence_delta_max,prob_margin_delta_mean,prob_margin_delta_std,prob_margin_delta_q10,prob_margin_delta_median,prob_margin_delta_q90,prob_margin_delta_max,acc_a,acc_b,acc_delta_b_minus_a,pair_name,bayes_mode,branch,branch_cn
0,34.453558,9.009121,23.179973,33.822453,46.193755,71.007889,0.992592,0.004087,0.987590,0.993703,0.996253,0.998618,0.033303,0.060077,0.000332,0.006686,0.103061,0.504408,0.239715,0.327741,0.003473,0.077180,0.737400,1.747907,0.108115,0.006404,0.133346,-0.131287,0.002421,0.144378,0.676946,-0.001615,0.209085,-0.229325,0.002589,0.204910,0.946379,0.863865,0.837695,-0.026170,case_B__VS__case_A,posterior_mean,main,主分支
1,31.637597,8.368404,21.077600,31.094587,42.649728,65.648727,0.994001,0.003422,0.989858,0.994950,0.997026,0.999022,0.027635,0.049468,0.000281,0.005821,0.085236,0.446757,0.219160,0.297286,0.003021,0.072938,0.675273,1.692477,0.099921,0.001357,0.122748,-0.129762,0.001695,0.122822,0.640404,-0.008388,0.194468,-0.232427,0.001681,0.180977,0.896251,0.863865,0.841924,-0.021940,case_B__VS__case_A,mc_logit_mean,main,主分支
2,331.439262,26.934828,297.732092,330.726257,366.454590,438.235626,-0.898054,0.028510,-0.931835,-0.902299,-0.856285,-0.793511,0.027243,0.047123,0.000281,0.006114,0.085638,0.443252,0.220700,0.291281,0.003224,0.079269,0.668037,1.690058,0.100449,-0.009351,0.121870,-0.148681,0.000510,0.105451,0.613255,-0.023345,0.193311,-0.257814,0.000218,0.155989,0.874397,0.863865,0.842717,-0.021147,case_B__VS__case_A,mc_prob_mean_logits,main,主分支
3,52.951870,14.650874,36.070476,51.110985,72.949390,121.452164,0.861230,0.097129,0.749574,0.887374,0.948900,0.980808,0.068930,0.139349,0.000005,0.001728,0.267060,0.690949,0.307001,0.524627,0.000039,0.014471,1.242688,1.999006,0.142744,-0.001934,0.155066,-0.160446,0.000011,0.139541,0.736375,-0.005065,0.262676,-0.285315,0.000019,0.225644,0.992684,0.823685,0.816019,-0.007666,case_B__VS__case_A,posterior_mean,rep,表示增强分支
4,49.109546,14.141462,32.758789,47.007523,68.456870,120.429161,0.886154,0.084086,0.786386,0.910769,0.958155,0.984318,0.054436,0.118068,0.000004,0.001221,0.198618,0.687710,0.261320,0.470503,0.000035,0.011426,1.045636,1.997140,0.122125,-0.000736,0.144304,-0.133694,0.000003,0.123582,0.723783,-0.001585,0.243820,-0.220729,0.000006,0.210731,0.975323,0.823685,0.835316,0.011631,case_B__VS__case_A,mc_logit_mean,rep,表示增强分支
5,293.621440,47.831367,230.276172,297.053894,352.537457,453.512115,-0.536718,0.284933,-0.777017,-0.635247,-0.121560,0.815179,0.055451,0.106779,0.000007,0.003424,0.196379,0.682702,0.282330,0.449774,0.000062,0.029575,1.043006,1.993041,0.123183,-0.041686,0.151752,-0.243347,-0.000738,0.043986,0.655596,-0.062288,0.248192,-0.396423,-0.001152,0.088388,0.954666,0.823685,0.833994,0.010309,case_B__VS__case_A,mc_prob_mean_logits,rep,表示增强分支
6,36.862061,7.964390,26.738567,36.367626,47.515105,68.685257,0.985227,0.008328,0.975282,0.987539,0.992420,0.997310,0.030746,0.067293,0.000052,0.002230,0.098063,0.611163,0.202389,0.341528,0.000583,0.026770,0.717683,1.924925,0.095691,0.003516,0.123449,-0.108153,0.000406,0.115282,0.709947,0.001232,0.202202,-0.191594,0.000544,0.194030,0.928357,0.873645,0.854613,-0.019032,case_B__VS__case_A,posterior_mean,fusion,融合分支
7,34.131857,7.550046,24.505966,33.687561,44.213435,62.062805,0.988232,0.007025,0.979533,0.990300,0.994252,0.997979,0.024203,0.053521,0.000046,0.001854,0.077177,0.584391,0.178384,0.299428,0.000528,0.023847,0.617398,1.888265,0.081153,-0.000616,0.112885,-0.106426,0.000237,0.100031,0.662496,-0.004271,0.185647,-0.185210,0.000258,0.163828,0.911003,0.873645,0.868887,

,pair_name,branch,branch_cn,n_samples,n_test,mc_logit_std_mean_mean,mc_logit_std_mean_std,mc_logit_std_mean_q10,mc_logit_std_mean_median,mc_logit_std_mean_q90,mc_logit_std_mean_max,mc_logit_l2_to_mean_mean,mc_logit_l2_to_mean_std,mc_logit_l2_to_mean_q10,mc_logit_l2_to_mean_median,mc_logit_l2_to_mean_q90,mc_logit_l2_to_mean_max,mc_prob_l1_to_mean_mean,mc_prob_l1_to_mean_std,mc_prob_l1_to_mean_q10,mc_prob_l1_to_mean_median,mc_prob_l1_to_mean_q90,mc_prob_l1_to_mean_max,sample_agreement_mean,sample_agreement_std,sample_agreement_q10,sample_agreement_median,sample_agreement_q90,sample_agreement_max,num_unique_preds_mean,num_unique_preds_std,num_unique_preds_q10,num_unique_preds_median,num_unique_preds_q90,num_unique_preds_max,mutual_information_mean,mutual_information_std,mutual_information_q10,mutual_information_median,mutual_information_q90,mutual_information_max,variation_ratio_mean,variation_ratio_std,variation_ratio_q10,variation_ratio_median,variation_ratio_q90,variation_ratio_max
0,case_B__VS__case_A,main,主分支,20,3783,0.720721,0.094490,0.614029,0.708081,0.848100,1.117532,7.014778,0.854192,6.038402,6.883706,8.163708,10.616076,0.106650,0.123405,0.001259,0.046603,0.305246,0.573691,0.947423,0.124802,0.75,1.0,1.0,1.0,1.314830,0.697800,1.0,1.0,2.0,7.0,0.027746,0.038377,0.000271,0.009835,0.083914,0.248007,0.052577,0.124802,0.0,0.0,0.25,0.7
1,case_B__VS__case_A,rep,表示增强分支,20,3783,1.934216,0.244449,1.627866,1.920669,2.261442,2.845012,19.200492,2.325792,16.263325,19.134899,22.288320,27.926392,0.189966,0.266651,0.000050,0.029462,0.637669,1.163085,0.913878,0.164043,0.65,1.0,1.0,1.0,1.586043,1.075002,1.0,1.0,3.0,9.0,0.107571,0.166772,0.000022,0.013716,0.357310,1.022849,0.086122,0.164043,0.0,0.0,0.35,0.8
2,case_B__VS__case_A,fusion,融合分支,20,3783,0.846970,0.102048,0.721438,0.840154,0.980595,1.266080,8.336911,0.946585,7.161981,8.293864,9.566224,12.300798,0.099661,0.139343,0.000254,0.021521,0.336424,0.657131,0.954071,0.117566,0.80,1.0,1.0,1.0,1.280730,0.669147,1.0,1.0,2.0,7.0,0.030934,0.049937,0.000060,0.005254,0.107611,0.326539,0.045929,0.117566,0.0,0.0,0.20,0.7


,pair_name,sample_index,sample_id,label,MMRL_fusion_pred,Bayes_main_pred,Bayes_rep_pred,Bayes_fusion_pred,Bayes_fusion_correct,MMRL_fusion_correct,MMRL_vs_Bayes_fusion_changed,main_rep_conflict,main_fusion_conflict,rep_fusion_conflict,all_three_agree,Bayes_main_conf,Bayes_rep_conf,Bayes_fusion_conf,fusion_sample_agreement,fusion_num_unique_mc_preds,fusion_MI,fusion_variation_ratio
0,case_B__VS__case_A,0,/root/autodl-tmp/MMRL/DATASETS/ucf101/UCF-101-...,0,0,0,0,0,True,True,False,False,False,False,True,0.987790,0.936584,0.987993,1.00,1,0.002135,0.00
1,case_B__VS__case_A,1,/root/autodl-tmp/MMRL/DATASETS/ucf101/UCF-101-...,0,0,0,0,0,True,True,False,False,False,False,True,0.948651,0.928055,0.953381,1.00,1,0.006320,0.00
2,case_B__VS__case_A,2,/root/autodl-tmp/MMRL/DATASETS/ucf101/UCF-101-...,0,0,0,0,0,True,True,False,False,False,False,True,0.999780,0.981722,0.999291,1.00,1,0.000081,0.00
3,case_B__VS__case_A,3,/root/autodl-tmp/MMRL/DATASETS/ucf101/UCF-101-...,0,0,0,0,0,True,True,False,False,False,False,True,0.980834,0.941813,0.976443,1.00,1,0.001886,0.00
4,case_B__VS__case_A,4,/root/autodl-tmp/MMRL/DATASETS/ucf101/UCF-101-...,0,0,0,0,0,True,True,False,False,False,False,True,0.999747,0.988516,0.999358,1.00,1,0.000063,0.00
5,case_B__VS__case_A,5,/root/autodl-tmp/MMRL/DATASETS/ucf101/UCF-101-...,0,0,0,1,0,True,True,False,True,False,True,False,0.986397,0.773016,0.922957,1.00,1,0.008220,0.00
6,case_B__VS__case_A,6,/root/autodl-tmp/MMRL/DATASETS/ucf101/UCF-101-...,0,1,0,1,1,False,False,False,True,True,False,False,0.671443,0.774256,0.507670,0.55,2,0.034203,0.45
7,case_B__VS__case_A,7,/root/autodl-tmp/MMRL/DATASETS/ucf101/UCF-101-...,0,0,0,0,0,True,True,False,False,False,False,True,0.940728,0.955586,0.959509,1.00,1,0.006792,0.00
8,case_B__VS__case_A,8,/root/autodl-tmp/MMRL/DATASETS/ucf101/UCF-101-...,0,0,0,0,0,True,True,False,False,False,False,True,0.803461,0.957757,0.951767,1.00,1,0.006969,0.00
9,case_B__VS__case_A,9,/root/autodl-tmp/MMRL/DATASETS/ucf101/UCF-101-...,0,1,1,0,0,True,False,True,True,True,False,False,0.597529,0.729129,0.571732,0.65,2,0.041814,0.35


## 15D. 分支冲突消歧：区分“预测不一致”和“真实类别方向冲突”

之前的“冲突”容易产生歧义：

- **预测不一致**：主分支和表示增强分支预测类别不同，即 `main_pred != rep_pred`。这种情况下两个分支可能一个对一个错，也可能两个都错。
- **真实类别方向冲突**：主分支和表示增强分支对真实类别的 margin 符号相反，即一个分支把真实类别排在所有错误类别前面，另一个分支没有。这种情况下才可以说一个分支站在真实类别一侧、另一个分支站在错误侧。

本节新增：

- `branch_disagreement_outcome_sample_df`：逐样本冲突/不一致明细。
- `branch_disagreement_outcome_summary_df`：按冲突类型汇总数量、准确率、融合跟随关系。
- `branch_margin_conflict_summary_df`：只针对真实类别 margin 方向冲突的严格统计。

后续报告中不要再把 `main_pred != rep_pred` 简单解释成“一方对、一方错”。必须先看这里的 `disagreement_type`。

In [45]:

# ============================================================
# 15D. 分支冲突消歧：预测不一致 vs 真实类别 margin 方向冲突
# ============================================================

def _true_vs_max_wrong_margin_from_probs(probs, labels):
    """返回 true class probability - max wrong class probability。
    因为 softmax 保持类别排序，所以这个 margin 的符号与 logit margin 符号一致。
    """
    probs = probs.float()
    labels = labels.long()
    n = probs.shape[0]
    idx = torch.arange(n, device=probs.device)

    true_score = probs[idx, labels]
    wrong_probs = probs.clone()
    wrong_probs[idx, labels] = -float("inf")
    max_wrong_score, max_wrong_class = wrong_probs.max(dim=-1)
    margin = true_score - max_wrong_score
    return margin, max_wrong_class, true_score, max_wrong_score


def _followed_branch(fusion_pred, main_pred, rep_pred):
    """融合预测跟随哪个分支。
    如果主分支和表示增强分支预测相同，则 both。
    如果融合等于其中一个分支，则返回 main/rep。
    如果融合不等于两者，返回 neither。
    """
    if int(main_pred) == int(rep_pred):
        return "both_same_prediction"
    if int(fusion_pred) == int(main_pred):
        return "main"
    if int(fusion_pred) == int(rep_pred):
        return "rep"
    return "neither"


def _margin_followed_side(fusion_margin, main_margin, rep_margin):
    """在 margin 符号冲突时，判断融合 margin 跟随哪一侧。
    注意：只有 main_margin 和 rep_margin 符号相反时，这个字段最有意义。
    """
    fm = float(fusion_margin)
    mm = float(main_margin)
    rm = float(rep_margin)

    if fm == 0:
        return "zero_margin"
    if mm == 0 or rm == 0:
        return "undefined_due_to_zero_branch_margin"

    fsign = 1 if fm > 0 else -1
    msign = 1 if mm > 0 else -1
    rsign = 1 if rm > 0 else -1

    if msign == rsign:
        return "branches_same_margin_side"
    if fsign == msign:
        return "main"
    if fsign == rsign:
        return "rep"
    return "neither"


def _disagreement_type(main_correct, rep_correct, main_pred, rep_pred):
    if int(main_pred) == int(rep_pred):
        if bool(main_correct) and bool(rep_correct):
            return "same_pred_both_correct"
        if (not bool(main_correct)) and (not bool(rep_correct)):
            return "same_pred_both_wrong"
        # 理论上 same_pred 时不可能一个对一个错，因为 label 相同；保留防御。
        return "same_pred_inconsistent_correctness"
    if bool(main_correct) and (not bool(rep_correct)):
        return "main_correct_rep_wrong"
    if (not bool(main_correct)) and bool(rep_correct):
        return "main_wrong_rep_correct"
    return "both_wrong_different_predictions"


def build_branch_disagreement_outcome_tables(case_outputs, pairings, bayes_mode="mc_prob_mean_probs"):
    """构建预测不一致与真实类别 margin 方向冲突的逐样本表和汇总表。"""
    sample_rows = []

    for pair in pairings:
        pair_name = pair["pair_name"]
        b = case_outputs[pair["bayes"]]
        labels = b["labels"].long()

        p_main = probs_for_case_mode_branch(b, bayes_mode, "main")
        p_rep = probs_for_case_mode_branch(b, bayes_mode, "rep")
        p_fusion = probs_for_case_mode_branch(b, bayes_mode, "fusion")

        pred_main = p_main.argmax(dim=-1)
        pred_rep = p_rep.argmax(dim=-1)
        pred_fusion = p_fusion.argmax(dim=-1)

        correct_main = pred_main.eq(labels)
        correct_rep = pred_rep.eq(labels)
        correct_fusion = pred_fusion.eq(labels)

        margin_main, max_wrong_main, true_main, wrong_main = _true_vs_max_wrong_margin_from_probs(p_main, labels)
        margin_rep, max_wrong_rep, true_rep, wrong_rep = _true_vs_max_wrong_margin_from_probs(p_rep, labels)
        margin_fusion, max_wrong_fusion, true_fusion, wrong_fusion = _true_vs_max_wrong_margin_from_probs(p_fusion, labels)

        pred_disagree = pred_main.ne(pred_rep)
        margin_sign_conflict = margin_main.mul(margin_rep).lt(0)

        for i in range(labels.numel()):
            main_ok = bool(correct_main[i].item())
            rep_ok = bool(correct_rep[i].item())
            fusion_ok = bool(correct_fusion[i].item())

            dt = _disagreement_type(
                main_ok,
                rep_ok,
                int(pred_main[i].item()),
                int(pred_rep[i].item()),
            )
            followed_pred = _followed_branch(
                int(pred_fusion[i].item()),
                int(pred_main[i].item()),
                int(pred_rep[i].item()),
            )
            followed_margin = _margin_followed_side(
                float(margin_fusion[i].item()),
                float(margin_main[i].item()),
                float(margin_rep[i].item()),
            )

            if bool(margin_sign_conflict[i].item()):
                if float(margin_main[i].item()) > 0 and float(margin_rep[i].item()) < 0:
                    margin_conflict_type = "main_positive_rep_negative"
                elif float(margin_main[i].item()) < 0 and float(margin_rep[i].item()) > 0:
                    margin_conflict_type = "main_negative_rep_positive"
                else:
                    margin_conflict_type = "unexpected"
            else:
                if float(margin_main[i].item()) > 0 and float(margin_rep[i].item()) > 0:
                    margin_conflict_type = "both_positive"
                elif float(margin_main[i].item()) < 0 and float(margin_rep[i].item()) < 0:
                    margin_conflict_type = "both_negative"
                else:
                    margin_conflict_type = "zero_or_boundary"

            sample_rows.append({
                "pair_name": pair_name,
                "bayes_mode": bayes_mode,
                "sample_index": i,
                "sample_id": b["sample_ids"][i],
                "label": int(labels[i].item()),

                "main_pred": int(pred_main[i].item()),
                "rep_pred": int(pred_rep[i].item()),
                "fusion_pred": int(pred_fusion[i].item()),

                "main_correct": main_ok,
                "rep_correct": rep_ok,
                "fusion_correct": fusion_ok,

                "main_rep_pred_disagree": bool(pred_disagree[i].item()),
                "main_rep_margin_sign_conflict": bool(margin_sign_conflict[i].item()),

                "disagreement_type": dt,
                "fusion_followed_prediction": followed_pred,

                "main_true_margin": float(margin_main[i].item()),
                "rep_true_margin": float(margin_rep[i].item()),
                "fusion_true_margin": float(margin_fusion[i].item()),
                "main_max_wrong_class": int(max_wrong_main[i].item()),
                "rep_max_wrong_class": int(max_wrong_rep[i].item()),
                "fusion_max_wrong_class": int(max_wrong_fusion[i].item()),

                "margin_conflict_type": margin_conflict_type,
                "fusion_followed_margin_side": followed_margin,
                "fusion_margin_negative": bool(float(margin_fusion[i].item()) < 0),

                "main_true_prob": float(true_main[i].item()),
                "rep_true_prob": float(true_rep[i].item()),
                "fusion_true_prob": float(true_fusion[i].item()),
                "main_max_wrong_prob": float(wrong_main[i].item()),
                "rep_max_wrong_prob": float(wrong_rep[i].item()),
                "fusion_max_wrong_prob": float(wrong_fusion[i].item()),
            })

    sample_df = pd.DataFrame(sample_rows)

    if sample_df.empty:
        return sample_df, pd.DataFrame(), pd.DataFrame()

    # 1) 预测不一致视角的汇总
    summary_rows = []
    group_cols = [
        "pair_name",
        "bayes_mode",
        "main_rep_pred_disagree",
        "disagreement_type",
        "fusion_followed_prediction",
    ]
    for keys, g in sample_df.groupby(group_cols, dropna=False):
        pair_name, mode, pred_disagree, disagreement_type, followed = keys
        summary_rows.append({
            "pair_name": pair_name,
            "bayes_mode": mode,
            "main_rep_pred_disagree": bool(pred_disagree),
            "disagreement_type": disagreement_type,
            "fusion_followed_prediction": followed,
            "n": len(g),
            "rate_over_all": len(g) / len(sample_df),
            "main_acc": g["main_correct"].mean(),
            "rep_acc": g["rep_correct"].mean(),
            "fusion_acc": g["fusion_correct"].mean(),
            "fusion_wrong_count": int((~g["fusion_correct"]).sum()),
            "fusion_wrong_rate": float((~g["fusion_correct"]).mean()),
            "margin_sign_conflict_rate": g["main_rep_margin_sign_conflict"].mean(),
        })
    summary_df = pd.DataFrame(summary_rows).sort_values(
        ["pair_name", "main_rep_pred_disagree", "disagreement_type", "fusion_followed_prediction"],
        ascending=[True, False, True, True],
    ).reset_index(drop=True)

    # 2) 真实类别 margin 符号冲突视角的严格汇总
    margin_rows = []
    strict = sample_df[sample_df["main_rep_margin_sign_conflict"]].copy()
    if not strict.empty:
        group_cols2 = [
            "pair_name",
            "bayes_mode",
            "margin_conflict_type",
            "fusion_followed_margin_side",
        ]
        for keys, g in strict.groupby(group_cols2, dropna=False):
            pair_name, mode, margin_conflict_type, followed_margin = keys
            margin_rows.append({
                "pair_name": pair_name,
                "bayes_mode": mode,
                "margin_conflict_type": margin_conflict_type,
                "fusion_followed_margin_side": followed_margin,
                "n": len(g),
                "rate_over_margin_conflict": len(g) / len(strict),
                "rate_over_all": len(g) / len(sample_df),
                "main_acc": g["main_correct"].mean(),
                "rep_acc": g["rep_correct"].mean(),
                "fusion_acc": g["fusion_correct"].mean(),
                "fusion_margin_negative_count": int(g["fusion_margin_negative"].sum()),
                "fusion_margin_negative_rate": float(g["fusion_margin_negative"].mean()),
                "fusion_wrong_count": int((~g["fusion_correct"]).sum()),
                "fusion_wrong_rate": float((~g["fusion_correct"]).mean()),
            })
    margin_summary_df = pd.DataFrame(margin_rows)
    if not margin_summary_df.empty:
        margin_summary_df = margin_summary_df.sort_values(
            ["pair_name", "margin_conflict_type", "fusion_followed_margin_side"]
        ).reset_index(drop=True)

    return sample_df, summary_df, margin_summary_df


branch_disagreement_outcome_sample_df, branch_disagreement_outcome_summary_df, branch_margin_conflict_summary_df = build_branch_disagreement_outcome_tables(
    case_outputs,
    PAIRINGS,
    bayes_mode="mc_prob_mean_probs",
)

display(branch_disagreement_outcome_summary_df)
display(branch_margin_conflict_summary_df)

# 方便直接查看关键样本：预测不一致且融合错误
display(
    branch_disagreement_outcome_sample_df[
        branch_disagreement_outcome_sample_df["main_rep_pred_disagree"]
        & (~branch_disagreement_outcome_sample_df["fusion_correct"])
    ].head(30)
)

branch_disagreement_outcome_sample_df.to_csv(SAVE_DIR / "table_branch_disagreement_outcome_sample.csv", index=False)
branch_disagreement_outcome_summary_df.to_csv(SAVE_DIR / "table_branch_disagreement_outcome_summary.csv", index=False)
branch_margin_conflict_summary_df.to_csv(SAVE_DIR / "table_branch_margin_conflict_summary.csv", index=False)


,pair_name,bayes_mode,main_rep_pred_disagree,disagreement_type,fusion_followed_prediction,n,rate_over_all,main_acc,rep_acc,fusion_acc,fusion_wrong_count,fusion_wrong_rate,margin_sign_conflict_rate
0,case_B__VS__case_A,mc_prob_mean_probs,True,both_wrong_different_predictions,main,91,0.024055,0.0,0.0,0.0,91,1.0,0.0
1,case_B__VS__case_A,mc_prob_mean_probs,True,both_wrong_different_predictions,neither,34,0.008988,0.0,0.0,0.5,17,0.5,0.0
2,case_B__VS__case_A,mc_prob_mean_probs,True,both_wrong_different_predictions,rep,51,0.013481,0.0,0.0,0.0,51,1.0,0.0
3,case_B__VS__case_A,mc_prob_mean_probs,True,main_correct_rep_wrong,main,196,0.051811,1.0,0.0,1.0,0,0.0,1.0
4,case_B__VS__case_A,mc_prob_mean_probs,True,main_correct_rep_wrong,neither,8,0.002115,1.0,0.0,0.0,8,1.0,1.0
5,case_B__VS__case_A,mc_prob_mean_probs,True,main_correct_rep_wrong,rep,61,0.016125,1.0,0.0,0.0,61,1.0,1.0
6,case_B__VS__case_A,mc_prob_mean_probs,True,main_wrong_rep_correct,main,70,0.018504,0.0,1.0,0.0,70,1.0,1.0
7,case_B__VS__case_A,mc_prob_mean_probs,True,main_wrong_rep_correct,neither,10,0.002643,0.0,1.0,0.0,10,1.0,1.0
8,case_B__VS__case_A,mc_prob_mean_probs,True,main_wrong_rep_correct,rep,152,0.040180,0.0,1.0,1.0,0,0.0,1.0
9,case_B__VS__case_A,mc_prob_mean_probs,False,same_pred_both_correct,both_same_prediction,2923,0.772667,1.0,1.0,1.0,0,0.0,0.0


,pair_name,bayes_mode,margin_conflict_type,fusion_followed_margin_side,n,rate_over_margin_conflict,rate_over_all,main_acc,rep_acc,fusion_acc,fusion_margin_negative_count,fusion_margin_negative_rate,fusion_wrong_count,fusion_wrong_rate
0,case_B__VS__case_A,mc_prob_mean_probs,main_negative_rep_positive,main,80,0.160966,0.021147,0.0,1.0,0.0,80,1.0,80,1.0
1,case_B__VS__case_A,mc_prob_mean_probs,main_negative_rep_positive,rep,152,0.305835,0.040180,0.0,1.0,1.0,0,0.0,0,0.0
2,case_B__VS__case_A,mc_prob_mean_probs,main_positive_rep_negative,main,196,0.394366,0.051811,1.0,0.0,1.0,0,0.0,0,0.0
3,case_B__VS__case_A,mc_prob_mean_probs,main_positive_rep_negative,rep,69,0.138833,0.018239,1.0,0.0,0.0,69,1.0,69,1.0


,pair_name,bayes_mode,sample_index,sample_id,label,main_pred,rep_pred,fusion_pred,main_correct,rep_correct,fusion_correct,main_rep_pred_disagree,main_rep_margin_sign_conflict,disagreement_type,fusion_followed_prediction,main_true_margin,rep_true_margin,fusion_true_margin,main_max_wrong_class,rep_max_wrong_class,fusion_max_wrong_class,margin_conflict_type,fusion_followed_margin_side,fusion_margin_negative,main_true_prob,rep_true_prob,fusion_true_prob,main_max_wrong_prob,rep_max_wrong_prob,fusion_max_wrong_prob
6,case_B__VS__case_A,mc_prob_mean_probs,6,/root/autodl-tmp/MMRL/DATASETS/ucf101/UCF-101-...,0,0,1,1,True,False,False,True,True,main_correct_rep_wrong,rep,0.343550,-0.551329,-0.015976,1,1,1,main_positive_rep_negative,rep,True,0.671443,2.229266e-01,0.491694,0.327893,0.774256,0.507670
23,case_B__VS__case_A,mc_prob_mean_probs,23,/root/autodl-tmp/MMRL/DATASETS/ucf101/UCF-101-...,0,1,0,1,False,True,False,True,True,main_wrong_rep_correct,main,-0.147345,0.081822,-0.063494,1,1,1,main_negative_rep_positive,main,True,0.422983,4.395145e-01,0.461598,0.570328,0.357693,0.525092
57,case_B__VS__case_A,mc_prob_mean_probs,57,/root/autodl-tmp/MMRL/DATASETS/ucf101/UCF-101-...,1,1,0,0,True,False,False,True,True,main_correct_rep_wrong,rep,0.372562,-0.867690,-0.199941,0,0,0,main_positive_rep_negative,rep,True,0.627895,6.583720e-02,0.391975,0.255334,0.933528,0.591917
66,case_B__VS__case_A,mc_prob_mean_probs,66,/root/autodl-tmp/MMRL/DATASETS/ucf101/UCF-101-...,1,19,1,19,False,True,False,True,True,main_wrong_rep_correct,main,-0.275232,0.162506,-0.127847,19,19,19,main_negative_rep_positive,main,True,0.356880,5.786105e-01,0.433180,0.632112,0.416104,0.561027
79,case_B__VS__case_A,mc_prob_mean_probs,79,/root/autodl-tmp/MMRL/DATASETS/ucf101/UCF-101-...,2,2,42,42,True,False,False,True,True,main_correct_rep_wrong,rep,0.280467,-0.563298,-0.288181,42,42,42,main_positive_rep_negative,rep,True,0.639420,3.618490e-02,0.351522,0.358953,0.599483,0.639704
101,case_B__VS__case_A,mc_prob_mean_probs,101,/root/autodl-tmp/MMRL/DATASETS/ucf101/UCF-101-...,2,2,42,42,True,False,False,True,True,main_correct_rep_wrong,rep,0.102239,-0.483653,-0.209593,47,42,42,main_positive_rep_negative,rep,True,0.297058,7.156967e-03,0.134921,0.194819,0.490810,0.344514
105,case_B__VS__case_A,mc_prob_mean_probs,105,/root/autodl-tmp/MMRL/DATASETS/ucf101/UCF-101-...,2,55,2,55,False,True,False,True,True,main_wrong_rep_correct,main,-0.865708,0.317011,-0.730642,55,55,55,main_negative_rep_positive,main,True,0.009930,5.320815e-01,0.064610,0.875638,0.215071,0.795252
154,case_B__VS__case_A,mc_prob_mean_probs,154,/root/autodl-tmp/MMRL/DATASETS/ucf101/UCF-101-...,4,4,56,56,True,False,False,True,True,main_correct_rep_wrong,rep,0.059126,-0.339523,-0.106401,56,56,56,main_positive_rep_negative,rep,True,0.463573,3.136977e-01,0.424864,0.404447,0.653221,0.531265
161,case_B__VS__case_A,mc_prob_mean_probs,161,/root/autodl-tmp/MMRL/DATASETS/ucf101/UCF-101-...,4,68,56,68,False,False,False,True,False,both_wrong_different_predictions,main,-0.329558,-0.090721,-0.518154,68,56,68,both_negative,branches_same_margin_side,True,0.012293,2.865725e-01,0.073372,0.341851,0.377294,0.591526
172,case_B__VS__case_A,mc_prob_mean_probs,172,/root/autodl-tmp/MMRL/DATASETS/ucf101/UCF-101-...,4,39,56,56,False,False,False,True,False,both_wrong_different_predictions,rep,-0.392960,-0.771432,-0.670498,39,56,56,both_negative,branches_same_margin_side,True,0.023043,1.785920e-02,0.029807,0.416003,0.789292,0.700305


In [46]:

# ============================================================
# 8A. 特征差异增强分析：BayesMMRL 与 MMRL 的表示是否真的不同
# ============================================================


# 本 cell 自包含的安全除法函数，避免依赖后续不确定性模块里的 helper。
def _feature_safe_ratio(num, den):
    try:
        num = float(num)
        den = float(den)
    except Exception:
        return np.nan
    if not np.isfinite(num) or not np.isfinite(den) or abs(den) < 1e-12:
        return np.nan
    return num / den



# 本 cell 自包含的安全除法函数，避免依赖其他 cell。
def _feature_safe_ratio(num, den):
    try:
        num = float(num)
        den = float(den)
    except Exception:
        return np.nan
    if not np.isfinite(num) or not np.isfinite(den) or abs(den) < 1e-12:
        return np.nan
    return num / den

def _torch_to_np(x):
    if torch.is_tensor(x):
        return x.detach().float().cpu().numpy()
    return np.asarray(x)


def _angle_deg_from_cosine(cos):
    cos_np = np.clip(_torch_to_np(cos).astype(float), -1.0, 1.0)
    return np.degrees(np.arccos(cos_np))


def _cohen_d(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    a = a[np.isfinite(a)]
    b = b[np.isfinite(b)]
    if len(a) < 2 or len(b) < 2:
        return np.nan
    va = a.var(ddof=1)
    vb = b.var(ddof=1)
    pooled = np.sqrt(((len(a) - 1) * va + (len(b) - 1) * vb) / max(len(a) + len(b) - 2, 1))
    if pooled <= 1e-12:
        return np.nan
    return float((a.mean() - b.mean()) / pooled)



def _feature_safe_ratio(num, den):
    try:
        num = float(num)
        den = float(den)
    except Exception:
        return np.nan
    if not np.isfinite(num) or not np.isfinite(den) or abs(den) < 1e-12:
        return np.nan
    return num / den


def _summary_stats(values, prefix):
    v = np.asarray(values, dtype=float)
    v = v[np.isfinite(v)]
    if len(v) == 0:
        return {
            f"{prefix}_mean": np.nan,
            f"{prefix}_std": np.nan,
            f"{prefix}_min": np.nan,
            f"{prefix}_q10": np.nan,
            f"{prefix}_q25": np.nan,
            f"{prefix}_median": np.nan,
            f"{prefix}_q75": np.nan,
            f"{prefix}_q90": np.nan,
            f"{prefix}_max": np.nan,
        }
    return {
        f"{prefix}_mean": float(np.mean(v)),
        f"{prefix}_std": float(np.std(v, ddof=0)),
        f"{prefix}_min": float(np.min(v)),
        f"{prefix}_q10": float(np.quantile(v, 0.10)),
        f"{prefix}_q25": float(np.quantile(v, 0.25)),
        f"{prefix}_median": float(np.quantile(v, 0.50)),
        f"{prefix}_q75": float(np.quantile(v, 0.75)),
        f"{prefix}_q90": float(np.quantile(v, 0.90)),
        f"{prefix}_max": float(np.max(v)),
    }


def build_feature_difference_sample_table(pairings, case_outputs, bayes_mode="mc_prob_mean_probs", branch="fusion"):
    rows = []

    for pair in pairings:
        pair_name = pair["pair_name"]
        m = case_outputs[pair["mmrl"]]
        b = case_outputs[pair["bayes"]]
        assert_same_samples(m, b)

        _, sample_df = paired_outcome_for_pair(m, b, bayes_mode=bayes_mode, branch=branch)
        labels = m["labels"]

        if "features_img" not in m.get("standard_features", {}) or "features_img" not in b.get("posterior_mean_features", {}):
            print(f"[WARN] {pair_name}: 缺少 features_img，跳过逐样本图像特征差异。")
            continue

        fm = m["standard_features"]["features_img"].float()
        fb = b["posterior_mean_features"]["features_img"].float()

        if fm.shape[0] != labels.shape[0] or fb.shape[0] != labels.shape[0]:
            print(f"[WARN] {pair_name}: features_img 不是逐样本形状，跳过。", fm.shape, fb.shape)
            continue

        cos = cosine_per_row(fm, fb)
        shift = (fb - fm).norm(dim=-1)
        angle = _angle_deg_from_cosine(cos)

        # MC feature statistics, only if collect_mc_features=True when collecting outputs
        mc_shift_mean = np.full(len(labels), np.nan)
        mc_shift_std = np.full(len(labels), np.nan)
        mc_feature_var = np.full(len(labels), np.nan)
        if "mc_sample_features" in b and "features_img" in b["mc_sample_features"]:
            fs = b["mc_sample_features"]["features_img"].float()  # [S,N,D]
            # 每个采样特征相对 MMRL 特征的偏移
            mc_shift = (fs - fm.unsqueeze(0)).norm(dim=-1)  # [S,N]
            mc_shift_mean = _torch_to_np(mc_shift.mean(dim=0))
            mc_shift_std = _torch_to_np(mc_shift.std(dim=0, unbiased=False))
            # 采样特征围绕自身均值的方差
            fs_center = fs - fs.mean(dim=0, keepdim=True)
            mc_feature_var = _torch_to_np(fs_center.pow(2).sum(dim=-1).mean(dim=0))

        # Bayes MC uncertainty for correlation
        bayes_mi = np.full(len(labels), np.nan)
        bayes_var_ratio = np.full(len(labels), np.nan)
        if b.get("is_bayes", False) and branch in b.get("mc_samples", {}):
            u = predictive_uncertainty_from_mc_logits(b["mc_samples"][branch])
            bayes_mi = _torch_to_np(u["mutual_info"])
            bayes_var_ratio = _torch_to_np(u["variation_ratio"])

        for i in range(len(labels)):
            base = sample_df.iloc[i].to_dict()
            rows.append({
                "pair_name": pair_name,
                "sample_index": i,
                "sample_id": m["sample_ids"][i],
                "label": int(labels[i].item()),
                "case_type": base.get("case_type"),
                "MMRL_pred": int(base.get("MMRL_pred")),
                "Bayes_pred": int(base.get("Bayes_pred")),
                "MMRL_conf": float(base.get("MMRL_conf")),
                "Bayes_conf": float(base.get("Bayes_conf")),
                "MMRL_entropy": float(base.get("MMRL_entropy")),
                "Bayes_entropy": float(base.get("Bayes_entropy")),
                "JS": float(base.get("JS")),
                "sym_KL": float(base.get("sym_KL")),
                "L1": float(base.get("L1")),
                "image_cosine": float(cos[i].item()),
                "image_angle_deg": float(angle[i]),
                "image_shift_norm": float(shift[i].item()),
                "mc_image_shift_mean": float(mc_shift_mean[i]),
                "mc_image_shift_std": float(mc_shift_std[i]),
                "mc_image_feature_var": float(mc_feature_var[i]),
                "Bayes_MI": float(bayes_mi[i]),
                "Bayes_variation_ratio": float(bayes_var_ratio[i]),
            })

    return pd.DataFrame(rows)


def build_text_feature_difference_table(pairings, case_outputs):
    rows = []
    for pair in pairings:
        pair_name = pair["pair_name"]
        m = case_outputs[pair["mmrl"]]
        b = case_outputs[pair["bayes"]]
        if "features_text" not in m.get("standard_features", {}) or "features_text" not in b.get("posterior_mean_features", {}):
            continue
        tm = m["standard_features"]["features_text"].float()
        tb = b["posterior_mean_features"]["features_text"].float()
        n = min(tm.shape[0], tb.shape[0])
        tm, tb = tm[:n], tb[:n]
        cos = cosine_per_row(tm, tb)
        shift = (tb - tm).norm(dim=-1)
        angle = _angle_deg_from_cosine(cos)
        for i in range(n):
            rows.append({
                "pair_name": pair_name,
                "class_or_text_index": i,
                "text_cosine": float(cos[i].item()),
                "text_angle_deg": float(angle[i]),
                "text_shift_norm": float(shift[i].item()),
            })
    return pd.DataFrame(rows)


def build_feature_shift_distribution(sample_table):
    if sample_table.empty:
        return pd.DataFrame()

    rows = []
    for (pair_name, case_type), g in sample_table.groupby(["pair_name", "case_type"], dropna=False):
        row = {
            "pair_name": pair_name,
            "case_type": case_type,
            "n": len(g),
        }
        row.update(_summary_stats(g["image_cosine"], "image_cosine"))
        row.update(_summary_stats(g["image_angle_deg"], "image_angle_deg"))
        row.update(_summary_stats(g["image_shift_norm"], "image_shift_norm"))
        if "mc_image_shift_mean" in g and g["mc_image_shift_mean"].notna().any():
            row.update(_summary_stats(g["mc_image_shift_mean"], "mc_image_shift_mean"))
            row.update(_summary_stats(g["mc_image_feature_var"], "mc_image_feature_var"))
        rows.append(row)

    # overall
    for pair_name, g in sample_table.groupby("pair_name", dropna=False):
        row = {
            "pair_name": pair_name,
            "case_type": "all",
            "n": len(g),
        }
        row.update(_summary_stats(g["image_cosine"], "image_cosine"))
        row.update(_summary_stats(g["image_angle_deg"], "image_angle_deg"))
        row.update(_summary_stats(g["image_shift_norm"], "image_shift_norm"))
        if "mc_image_shift_mean" in g and g["mc_image_shift_mean"].notna().any():
            row.update(_summary_stats(g["mc_image_shift_mean"], "mc_image_shift_mean"))
            row.update(_summary_stats(g["mc_image_feature_var"], "mc_image_feature_var"))
        rows.append(row)

    return pd.DataFrame(rows)


def build_feature_shift_effect_table(sample_table, reference_case_type="both_correct"):
    if sample_table.empty:
        return pd.DataFrame()

    rows = []
    metrics = ["image_cosine", "image_angle_deg", "image_shift_norm"]

    for pair_name, gpair in sample_table.groupby("pair_name", dropna=False):
        ref = gpair[gpair["case_type"] == reference_case_type]
        if ref.empty:
            continue

        for case_type, g in gpair.groupby("case_type", dropna=False):
            if case_type == reference_case_type:
                continue

            row = {
                "pair_name": pair_name,
                "case_type": case_type,
                "reference_case_type": reference_case_type,
                "n": len(g),
                "n_reference": len(ref),
            }

            for metric in metrics:
                a = g[metric].astype(float).values
                b = ref[metric].astype(float).values
                row[f"{metric}_mean"] = float(np.nanmean(a)) if len(a) else np.nan
                row[f"{metric}_reference_mean"] = float(np.nanmean(b)) if len(b) else np.nan
                row[f"{metric}_delta_vs_reference"] = row[f"{metric}_mean"] - row[f"{metric}_reference_mean"]
                row[f"{metric}_ratio_vs_reference"] = _feature_safe_ratio(row[f"{metric}_mean"], row[f"{metric}_reference_mean"])
                row[f"{metric}_cohen_d_vs_reference"] = _cohen_d(a, b)

            rows.append(row)

    return pd.DataFrame(rows)


def build_feature_shift_error_detection_table(sample_table):
    if sample_table.empty:
        return pd.DataFrame()

    rows = []
    score_defs = {
        "image_shift_norm": "图像特征偏移范数",
        "one_minus_image_cosine": "1-图像特征余弦相似度",
        "image_angle_deg": "图像特征夹角",
    }

    df = sample_table.copy()
    df["one_minus_image_cosine"] = 1.0 - df["image_cosine"]

    targets = {
        "Bayes_error": df["Bayes_pred"].values != df["label"].values,
        "MMRL_error": df["MMRL_pred"].values != df["label"].values,
        "prediction_changed": df["MMRL_pred"].values != df["Bayes_pred"].values,
        "harmful_flip": df["case_type"].values == "MMRL_correct_Bayes_wrong",
        "beneficial_flip": df["case_type"].values == "MMRL_wrong_Bayes_correct",
    }

    for pair_name, g in df.groupby("pair_name", dropna=False):
        idx = g.index
        for target_name, y_all in targets.items():
            y = np.asarray(y_all)[idx]
            for score_name, score_cn in score_defs.items():
                s = g[score_name].astype(float).values
                rows.append({
                    "pair_name": pair_name,
                    "target": target_name,
                    "score": score_name,
                    "score_cn": score_cn,
                    "n": len(g),
                    "n_positive": int(np.sum(y)),
                    "positive_rate": float(np.mean(y)) if len(y) else np.nan,
                    "AUROC": binary_auroc(s, y),
                    "AUPR": binary_aupr(s, y),
                    "positive_score_mean": float(np.nanmean(s[y])) if np.any(y) else np.nan,
                    "negative_score_mean": float(np.nanmean(s[~y])) if np.any(~y) else np.nan,
                    "positive_over_negative_score_mean": _feature_safe_ratio(
                        float(np.nanmean(s[y])) if np.any(y) else np.nan,
                        float(np.nanmean(s[~y])) if np.any(~y) else np.nan,
                    ),
                })

    return pd.DataFrame(rows)


def build_feature_shift_correlation_table(sample_table):
    if sample_table.empty:
        return pd.DataFrame()

    rows = []
    df = sample_table.copy()
    candidate_cols = [
        "JS", "sym_KL", "L1",
        "MMRL_conf", "Bayes_conf",
        "MMRL_entropy", "Bayes_entropy",
        "Bayes_MI", "Bayes_variation_ratio",
        "mc_image_shift_mean", "mc_image_feature_var",
    ]

    for pair_name, g in df.groupby("pair_name", dropna=False):
        for xcol in ["image_shift_norm", "image_angle_deg", "image_cosine"]:
            for ycol in candidate_cols:
                if ycol not in g.columns:
                    continue
                sub = g[[xcol, ycol]].replace([np.inf, -np.inf], np.nan).dropna()
                if len(sub) < 3:
                    continue
                rows.append({
                    "pair_name": pair_name,
                    "x": xcol,
                    "y": ycol,
                    "n": len(sub),
                    "spearman_corr": sub[xcol].corr(sub[ycol], method="spearman"),
                    "pearson_corr": sub[xcol].corr(sub[ycol], method="pearson"),
                    "interpretation_hint": "相关性越高，说明特征偏移与概率变化/不确定性变化越同步；相关性低说明特征变化和该指标关系弱。",
                })

    return pd.DataFrame(rows)


def build_top_feature_shift_samples(sample_table, top_k=50):
    if sample_table.empty:
        return pd.DataFrame()
    cols = [
        "pair_name", "sample_index", "sample_id", "label",
        "case_type", "MMRL_pred", "Bayes_pred",
        "MMRL_conf", "Bayes_conf", "MMRL_entropy", "Bayes_entropy",
        "image_cosine", "image_angle_deg", "image_shift_norm",
        "JS", "L1", "Bayes_MI", "Bayes_variation_ratio",
    ]
    cols = [c for c in cols if c in sample_table.columns]
    return sample_table.sort_values("image_shift_norm", ascending=False)[cols].head(top_k).reset_index(drop=True)


feature_difference_sample_df = build_feature_difference_sample_table(
    PAIRINGS,
    case_outputs,
    bayes_mode="mc_prob_mean_probs",
    branch="fusion",
)

text_feature_difference_df = build_text_feature_difference_table(PAIRINGS, case_outputs)
feature_shift_distribution_df = build_feature_shift_distribution(feature_difference_sample_df)
feature_shift_effect_df = build_feature_shift_effect_table(feature_difference_sample_df)
feature_shift_error_detection_df = build_feature_shift_error_detection_table(feature_difference_sample_df)
feature_shift_correlation_df = build_feature_shift_correlation_table(feature_difference_sample_df)
top_feature_shift_samples_df = build_top_feature_shift_samples(feature_difference_sample_df, top_k=50)

display(feature_shift_distribution_df)
display(feature_shift_effect_df)
display(feature_shift_error_detection_df)
display(feature_shift_correlation_df)
display(text_feature_difference_df.describe() if not text_feature_difference_df.empty else text_feature_difference_df)
display(top_feature_shift_samples_df)

feature_difference_sample_df.to_csv(SAVE_DIR / "table_feature_difference_sample.csv", index=False)
text_feature_difference_df.to_csv(SAVE_DIR / "table_text_feature_difference.csv", index=False)
feature_shift_distribution_df.to_csv(SAVE_DIR / "table_feature_shift_distribution.csv", index=False)
feature_shift_effect_df.to_csv(SAVE_DIR / "table_feature_shift_effect.csv", index=False)
feature_shift_error_detection_df.to_csv(SAVE_DIR / "table_feature_shift_error_detection.csv", index=False)
feature_shift_correlation_df.to_csv(SAVE_DIR / "table_feature_shift_correlation.csv", index=False)
top_feature_shift_samples_df.to_csv(SAVE_DIR / "table_top_feature_shift_samples.csv", index=False)


,pair_name,case_type,n,image_cosine_mean,image_cosine_std,image_cosine_min,image_cosine_q10,image_cosine_q25,image_cosine_median,image_cosine_q75,image_cosine_q90,image_cosine_max,image_angle_deg_mean,image_angle_deg_std,image_angle_deg_min,image_angle_deg_q10,image_angle_deg_q25,image_angle_deg_median,image_angle_deg_q75,image_angle_deg_q90,image_angle_deg_max,image_shift_norm_mean,image_shift_norm_std,image_shift_norm_min,image_shift_norm_q10,image_shift_norm_q25,image_shift_norm_median,image_shift_norm_q75,image_shift_norm_q90,image_shift_norm_max,mc_image_shift_mean_mean,mc_image_shift_mean_std,mc_image_shift_mean_min,mc_image_shift_mean_q10,mc_image_shift_mean_q25,mc_image_shift_mean_median,mc_image_shift_mean_q75,mc_image_shift_mean_q90,mc_image_shift_mean_max,mc_image_feature_var_mean,mc_image_feature_var_std,mc_image_feature_var_min,mc_image_feature_var_q10,mc_image_feature_var_q25,mc_image_feature_var_median,mc_image_feature_var_q75,mc_image_feature_var_q90,mc_image_feature_var_max
0,case_B__VS__case_A,MMRL_correct_Bayes_wrong,115,0.945435,0.016030,0.892371,0.923924,0.936738,0.947225,0.957517,0.963302,0.972726,18.821587,2.757386,13.412373,15.569867,16.760742,18.697384,20.489036,22.492623,26.827278,0.326927,0.047425,0.233556,0.270910,0.291488,0.324885,0.355699,0.390054,0.463959,0.302367,0.041257,0.218674,0.254380,0.270446,0.298068,0.327856,0.354941,0.431231,0.011279,0.002685,0.006564,0.007902,0.009491,0.010942,0.012864,0.014741,0.020070
1,case_B__VS__case_A,MMRL_wrong_Bayes_correct,98,0.944975,0.013891,0.909012,0.927199,0.938211,0.946565,0.953310,0.961889,0.970311,18.949484,2.403192,13.996385,15.868636,17.577353,18.814910,20.246698,21.997616,24.630792,0.329152,0.041342,0.243676,0.276075,0.305581,0.326909,0.351536,0.381577,0.426586,0.304396,0.037051,0.236917,0.260605,0.278277,0.302245,0.324478,0.360773,0.394666,0.011499,0.002597,0.005662,0.008404,0.009833,0.011399,0.012657,0.014426,0.020224
2,case_B__VS__case_A,both_correct,3190,0.951148,0.015920,0.854222,0.931180,0.942496,0.953728,0.962221,0.968919,0.983864,17.763739,2.851309,10.306794,14.322382,15.799431,17.497959,19.524883,21.380421,31.326155,0.308697,0.049096,0.179645,0.249323,0.274879,0.304212,0.339127,0.370997,0.539960,0.284642,0.043240,0.169243,0.232462,0.254790,0.280466,0.312094,0.340753,0.493584,0.009429,0.002858,0.002669,0.005977,0.007429,0.009099,0.011112,0.013274,0.026059
3,case_B__VS__case_A,both_wrong,380,0.947035,0.015681,0.889944,0.925978,0.938173,0.949160,0.958435,0.965878,0.974538,18.536750,2.744769,12.957089,15.010623,16.577338,18.348288,20.253042,22.183780,27.133750,0.322024,0.047233,0.225662,0.261236,0.288321,0.318872,0.351645,0.384766,0.469160,0.298250,0.041087,0.215323,0.246687,0.269030,0.293791,0.322896,0.354643,0.438981,0.011090,0.002978,0.004981,0.007641,0.008953,0.010613,0.012948,0.015209,0.020839
4,case_B__VS__case_A,all,3783,0.950401,0.015948,0.854222,0.929822,0.941570,0.952990,0.961636,0.968282,0.983864,17.904263,2.846714,10.306794,14.469121,15.922070,17.638031,19.683046,21.592903,31.326155,0.311120,0.049013,0.179645,0.251864,0.276999,0.306628,0.341847,0.374641,0.539960,0.287060,0.043196,0.169243,0.234375,0.257056,0.283252,0.313808,0.343377,0.493584,0.009706,0.002931,0.002669,0.006152,0.007657,0.009415,0.011440,0.013566,0.026059


,pair_name,case_type,reference_case_type,n,n_reference,image_cosine_mean,image_cosine_reference_mean,image_cosine_delta_vs_reference,image_cosine_ratio_vs_reference,image_cosine_cohen_d_vs_reference,image_angle_deg_mean,image_angle_deg_reference_mean,image_angle_deg_delta_vs_reference,image_angle_deg_ratio_vs_reference,image_angle_deg_cohen_d_vs_reference,image_shift_norm_mean,image_shift_norm_reference_mean,image_shift_norm_delta_vs_reference,image_shift_norm_ratio_vs_reference,image_shift_norm_cohen_d_vs_reference
0,case_B__VS__case_A,MMRL_correct_Bayes_wrong,both_correct,115,3190,0.945435,0.951148,-0.005713,0.993994,-0.358675,18.821587,17.763739,1.057848,1.059551,0.371311,0.326927,0.308697,0.018230,1.059054,0.371631
1,case_B__VS__case_A,MMRL_wrong_Bayes_correct,both_correct,98,3190,0.944975,0.951148,-0.006173,0.993510,-0.389035,18.949484,17.763739,1.185745,1.066751,0.417539,0.329152,0.308697,0.020455,1.066263,0.418328
2,case_B__VS__case_A,both_wrong,both_correct,380,3190,0.947035,0.951148,-0.004113,0.995676,-0.258704,18.536750,17.763739,0.773011,1.043516,0.272096,0.322024,0.308697,0.013327,1.043172,0.272453


,pair_name,target,score,score_cn,n,n_positive,positive_rate,AUROC,AUPR,positive_score_mean,negative_score_mean,positive_over_negative_score_mean
0,case_B__VS__case_A,Bayes_error,image_shift_norm,图像特征偏移范数,3783,495,0.130849,0.584074,0.163247,0.323163,0.309307,1.044798
1,case_B__VS__case_A,Bayes_error,one_minus_image_cosine,1-图像特征余弦相似度,3783,495,0.130849,0.584074,0.163247,0.053337,0.049036,1.087707
2,case_B__VS__case_A,Bayes_error,image_angle_deg,图像特征夹角,3783,495,0.130849,0.584074,0.163247,18.602924,17.799081,1.045162
3,case_B__VS__case_A,MMRL_error,image_shift_norm,图像特征偏移范数,3783,478,0.126355,0.588904,0.157057,0.323486,0.309332,1.045757
4,case_B__VS__case_A,MMRL_error,one_minus_image_cosine,1-图像特征余弦相似度,3783,478,0.126355,0.588904,0.157057,0.053388,0.049051,1.088411
5,case_B__VS__case_A,MMRL_error,image_angle_deg,图像特征夹角,3783,478,0.126355,0.588904,0.157057,18.621369,17.800548,1.046112
6,case_B__VS__case_A,prediction_changed,image_shift_norm,图像特征偏移范数,3783,303,0.080095,0.641407,0.121913,0.332587,0.309251,1.075461
7,case_B__VS__case_A,prediction_changed,one_minus_image_cosine,1-图像特征余弦相似度,3783,303,0.080095,0.641405,0.121912,0.056372,0.049009,1.150227
8,case_B__VS__case_A,prediction_changed,image_angle_deg,图像特征夹角,3783,303,0.080095,0.641405,0.121912,19.150137,17.795786,1.076105
9,case_B__VS__case_A,harmful_flip,image_shift_norm,图像特征偏移范数,3783,115,0.030399,0.596309,0.041958,0.326927,0.310624,1.052484


,pair_name,x,y,n,spearman_corr,pearson_corr,interpretation_hint
0,case_B__VS__case_A,image_shift_norm,JS,3783,0.285430,0.244366,相关性越高，说明特征偏移与概率变化/不确定性变化越同步；相关性低说明特征变化和该指标关系弱。
1,case_B__VS__case_A,image_shift_norm,sym_KL,3783,0.294907,0.250433,相关性越高，说明特征偏移与概率变化/不确定性变化越同步；相关性低说明特征变化和该指标关系弱。
2,case_B__VS__case_A,image_shift_norm,L1,3783,0.257954,0.228879,相关性越高，说明特征偏移与概率变化/不确定性变化越同步；相关性低说明特征变化和该指标关系弱。
3,case_B__VS__case_A,image_shift_norm,MMRL_conf,3783,-0.217345,-0.170849,相关性越高，说明特征偏移与概率变化/不确定性变化越同步；相关性低说明特征变化和该指标关系弱。
4,case_B__VS__case_A,image_shift_norm,Bayes_conf,3783,-0.209428,-0.164241,相关性越高，说明特征偏移与概率变化/不确定性变化越同步；相关性低说明特征变化和该指标关系弱。
5,case_B__VS__case_A,image_shift_norm,MMRL_entropy,3783,0.228068,0.203585,相关性越高，说明特征偏移与概率变化/不确定性变化越同步；相关性低说明特征变化和该指标关系弱。
6,case_B__VS__case_A,image_shift_norm,Bayes_entropy,3783,0.223196,0.193799,相关性越高，说明特征偏移与概率变化/不确定性变化越同步；相关性低说明特征变化和该指标关系弱。
7,case_B__VS__case_A,image_shift_norm,Bayes_MI,3783,0.275269,0.250049,相关性越高，说明特征偏移与概率变化/不确定性变化越同步；相关性低说明特征变化和该指标关系弱。
8,case_B__VS__case_A,image_shift_norm,Bayes_variation_ratio,3783,0.158400,0.138588,相关性越高，说明特征偏移与概率变化/不确定性变化越同步；相关性低说明特征变化和该指标关系弱。
9,case_B__VS__case_A,image_shift_norm,mc_image_shift_mean,3783,0.975664,0.978010,相关性越高，说明特征偏移与概率变化/不确定性变化越同步；相关性低说明特征变化和该指标关系弱。


,class_or_text_index,text_cosine,text_angle_deg,text_shift_norm
count,1616.000000,1616.000000,1616.000000,1616.000000
mean,807.500000,0.946649,18.538158,0.322012
std,466.643333,0.018925,3.193209,0.054884
min,0.000000,0.892753,12.836057,0.223563
25%,403.750000,0.940091,16.337189,0.284173
50%,807.500000,0.949809,18.229939,0.316832
75%,1211.250000,0.959623,19.933193,0.346148
max,1615.000000,0.975010,26.778807,0.463136


,pair_name,sample_index,sample_id,label,case_type,MMRL_pred,Bayes_pred,MMRL_conf,Bayes_conf,MMRL_entropy,Bayes_entropy,image_cosine,image_angle_deg,image_shift_norm,JS,L1,Bayes_MI,Bayes_variation_ratio
0,case_B__VS__case_A,394,/root/autodl-tmp/MMRL/DATASETS/ucf101/UCF-101-...,10,both_correct,10,10,0.993271,0.999514,0.055997,0.005212,0.854222,31.326155,0.539960,0.001843,0.012607,0.000142,0.00
1,case_B__VS__case_A,2248,/root/autodl-tmp/MMRL/DATASETS/ucf101/UCF-101-...,59,both_correct,59,59,0.998920,0.990507,0.010423,0.065848,0.854742,31.268738,0.538995,0.002170,0.017090,0.005552,0.00
2,case_B__VS__case_A,3222,/root/autodl-tmp/MMRL/DATASETS/ucf101/UCF-101-...,85,both_correct,85,85,0.520905,0.937769,0.942448,0.276875,0.870580,29.473840,0.508762,0.162486,0.882513,0.041331,0.00
3,case_B__VS__case_A,2247,/root/autodl-tmp/MMRL/DATASETS/ucf101/UCF-101-...,59,both_correct,59,59,0.997063,0.980517,0.025095,0.123207,0.875827,28.856968,0.498343,0.003935,0.033332,0.014284,0.00
4,case_B__VS__case_A,1794,/root/autodl-tmp/MMRL/DATASETS/ucf101/UCF-101-...,47,both_correct,47,47,0.726058,0.784763,0.961048,0.914797,0.877994,28.598716,0.493977,0.043111,0.300771,0.083179,0.00
5,case_B__VS__case_A,2287,/root/autodl-tmp/MMRL/DATASETS/ucf101/UCF-101-...,60,both_correct,60,60,0.377908,0.950309,1.881795,0.269414,0.878051,28.591852,0.493860,0.240736,1.157747,0.069549,0.00
6,case_B__VS__case_A,2244,/root/autodl-tmp/MMRL/DATASETS/ucf101/UCF-101-...,59,both_correct,59,59,0.998406,0.997482,0.014581,0.020781,0.883641,27.915212,0.482408,0.000241,0.002515,0.001266,0.00
7,case_B__VS__case_A,2290,/root/autodl-tmp/MMRL/DATASETS/ucf101/UCF-101-...,60,both_correct,60,60,0.953165,0.993106,0.262573,0.046869,0.884998,27.748729,0.479587,0.010109,0.079880,0.001771,0.00
8,case_B__VS__case_A,2260,/root/autodl-tmp/MMRL/DATASETS/ucf101/UCF-101-...,59,both_correct,59,59,0.945154,0.996811,0.233263,0.024756,0.886461,27.568182,0.476528,0.018264,0.107877,0.001427,0.00
9,case_B__VS__case_A,2246,/root/autodl-tmp/MMRL/DATASETS/ucf101/UCF-101-...,59,both_correct,59,59,0.999021,0.994279,0.009638,0.045699,0.887654,27.420100,0.474017,0.001370,0.009878,0.003447,0.00


## 12. Fusion margin decomposition 与 counterfactual branch swap


In [47]:
def fusion_margin_decomposition_table(case_outputs):
    rows = []
    for name, out in case_outputs.items():
        labels = out['labels']
        modes = eval_modes_for_case(out)
        for mode_name, branches in modes.items():
            if mode_name.endswith('_pred') or not all(br in branches for br in ['main', 'rep', 'fusion']):
                continue
            logits_main = logits_for_case_mode_branch(out, mode_name, 'main')
            logits_rep = logits_for_case_mode_branch(out, mode_name, 'rep')
            logits_fusion = logits_for_case_mode_branch(out, mode_name, 'fusion')
            for br_name, logits in [('main', logits_main), ('rep', logits_rep), ('fusion', logits_fusion)]:
                margin, _ = margin_y_vs_best_wrong(logits, labels)
                pred = logits.argmax(dim=-1)
                correct = pred.eq(labels)
                rows.append({
                    'case_name': name,
                    'method': out['cfg_method'],
                    'eval_mode': mode_name,
                    'branch': br_name,
                    'margin_true_vs_best_wrong_mean': float(margin.mean().item()),
                    'margin_true_vs_best_wrong_median': float(margin.median().item()),
                    'frac_positive_margin': float((margin > 0).float().mean().item()),
                    'acc_from_margin': float(correct.float().mean().item()),
                })
    return pd.DataFrame(rows)


def get_alpha_for_case(out):
    cfg = cfgs[out['name']]
    if hasattr(cfg, 'BAYES_MMRL') and out['is_bayes']:
        return float(cfg.BAYES_MMRL.ALPHA)
    if hasattr(cfg, 'MMRL'):
        return float(cfg.MMRL.ALPHA)
    return 0.7


def counterfactual_branch_swap_table(pairings, case_outputs):
    rows = []
    for pair in pairings:
        m = case_outputs[pair['mmrl']]
        b = case_outputs[pair['bayes']]
        assert_same_samples(m, b)
        labels = m['labels']
        alpha = get_alpha_for_case(b)
        m_main = logits_for_case_mode_branch(m, 'standard', 'main')
        m_rep = logits_for_case_mode_branch(m, 'standard', 'rep')
        for bayes_mode in ['posterior_mean', 'mc_logit_mean']:
            if bayes_mode not in eval_modes_for_case(b):
                continue
            b_main = logits_for_case_mode_branch(b, bayes_mode, 'main')
            b_rep = logits_for_case_mode_branch(b, bayes_mode, 'rep')
            variants = {
                'MMRL_main+MMRL_rep': alpha * m_main + (1.0 - alpha) * m_rep,
                'Bayes_main+Bayes_rep': alpha * b_main + (1.0 - alpha) * b_rep,
                'MMRL_main+Bayes_rep': alpha * m_main + (1.0 - alpha) * b_rep,
                'Bayes_main+MMRL_rep': alpha * b_main + (1.0 - alpha) * m_rep,
            }
            for var, logits in variants.items():
                row = {
                    'pair_name': pair['pair_name'],
                    'bayes_mode': bayes_mode,
                    'variant': var,
                    'alpha_used': alpha,
                }
                row.update(metrics_from_probs(probs_from_logits(logits), labels))
                rows.append(row)
    return pd.DataFrame(rows)

fusion_margin_df = fusion_margin_decomposition_table(case_outputs)
counterfactual_swap_df = counterfactual_branch_swap_table(PAIRINGS, case_outputs)
display(fusion_margin_df)
display(counterfactual_swap_df)
fusion_margin_df.to_csv(SAVE_DIR / 'table_fusion_margin_decomposition.csv', index=False)
counterfactual_swap_df.to_csv(SAVE_DIR / 'table_counterfactual_branch_swap.csv', index=False)


,case_name,method,eval_mode,branch,margin_true_vs_best_wrong_mean,margin_true_vs_best_wrong_median,frac_positive_margin,acc_from_margin
0,case_A,BayesMMRL,standard,main,3.358770,3.464470,0.841131,0.841131
1,case_A,BayesMMRL,standard,rep,5.362066,5.918343,0.833201,0.833201
2,case_A,BayesMMRL,standard,fusion,4.351710,4.614033,0.864922,0.864922
3,case_A,BayesMMRL,posterior_mean,main,3.377007,3.554512,0.837695,0.837695
4,case_A,BayesMMRL,posterior_mean,rep,5.282821,6.066895,0.816019,0.816019
5,case_A,BayesMMRL,posterior_mean,fusion,4.385847,4.714674,0.854613,0.854613
6,case_A,BayesMMRL,mc_prob_mean_probs,main,3.214597,3.242666,0.842717,0.842717
7,case_A,BayesMMRL,mc_prob_mean_probs,rep,4.530201,4.514752,0.833994,0.833994
8,case_A,BayesMMRL,mc_prob_mean_probs,fusion,4.150700,4.270417,0.869151,0.869151
9,case_A,BayesMMRL,mc_logit_mean,main,3.347224,3.424814,0.841924,0.841924


,pair_name,bayes_mode,variant,alpha_used,n,acc,macro_f1,ECE,NLL,Brier,conf_mean,entropy_mean,margin_mean
0,case_B__VS__case_A,posterior_mean,MMRL_main+MMRL_rep,0.7,3783,0.873645,0.869241,0.016849,0.440715,0.181814,0.889227,0.349907,0.822512
1,case_B__VS__case_A,posterior_mean,Bayes_main+Bayes_rep,0.7,3783,0.854613,0.850868,0.038423,0.517263,0.212861,0.892743,0.321036,0.823744
2,case_B__VS__case_A,posterior_mean,MMRL_main+Bayes_rep,0.7,3783,0.875496,0.871991,0.018668,0.444306,0.184712,0.892270,0.337820,0.826876
3,case_B__VS__case_A,posterior_mean,Bayes_main+MMRL_rep,0.7,3783,0.865715,0.862098,0.029183,0.473696,0.195083,0.892465,0.323347,0.823447
4,case_B__VS__case_A,mc_logit_mean,MMRL_main+MMRL_rep,0.7,3783,0.873645,0.869241,0.016849,0.440715,0.181814,0.889227,0.349907,0.822512
5,case_B__VS__case_A,mc_logit_mean,Bayes_main+Bayes_rep,0.7,3783,0.868887,0.865462,0.022364,0.469039,0.194644,0.888611,0.337046,0.818241
6,case_B__VS__case_A,mc_logit_mean,MMRL_main+Bayes_rep,0.7,3783,0.880518,0.877171,0.016583,0.421131,0.175747,0.890554,0.348164,0.825118
7,case_B__VS__case_A,mc_logit_mean,Bayes_main+MMRL_rep,0.7,3783,0.869945,0.866485,0.023364,0.459676,0.189102,0.891055,0.330934,0.821860


## 13. MC uncertainty / sample conflict / aggregation conflict / error detection


In [48]:
def mc_uncertainty_tables(case_outputs):
    unc_rows = []
    err_rows = []
    conflict_rows = []
    agg_rows = []
    margin_sign_rows = []
    branch_conflict_rows = []
    fusion_conflict_rows = []

    for name, out in case_outputs.items():
        if not out['is_bayes']:
            continue
        labels = out['labels']
        modes = eval_modes_for_case(out)
        pm_preds = {br: logits.argmax(dim=-1) for br, logits in out['posterior_mean'].items()}

        # aggregation conflict per branch
        for br, smp_logits in out['mc_samples'].items():
            u = predictive_uncertainty_from_mc_logits(smp_logits)
            prob_pred = u['probs_mean'].argmax(dim=-1)
            logit_pred = smp_logits.mean(dim=0).argmax(dim=-1)
            vote_pred = u['majority_vote']
            pm_pred = pm_preds.get(br)
            row = {
                'case_name': name,
                'branch': br,
                'prob_logit_agree': float(prob_pred.eq(logit_pred).float().mean().item()),
                'prob_vote_agree': float(prob_pred.eq(vote_pred).float().mean().item()),
                'logit_vote_agree': float(logit_pred.eq(vote_pred).float().mean().item()),
                'prob_correct': float(prob_pred.eq(labels).float().mean().item()),
                'logit_correct': float(logit_pred.eq(labels).float().mean().item()),
                'vote_correct': float(vote_pred.eq(labels).float().mean().item()),
            }
            if pm_pred is not None:
                row.update({
                    'PM_prob_agree': float(pm_pred.eq(prob_pred).float().mean().item()),
                    'PM_logit_agree': float(pm_pred.eq(logit_pred).float().mean().item()),
                    'PM_correct': float(pm_pred.eq(labels).float().mean().item()),
                })
            agg_rows.append(row)

            pred_prob = prob_pred
            correct = pred_prob.eq(labels)
            for correctness, mask in [('correct', correct), ('wrong', ~correct), ('all', torch.ones_like(correct, dtype=torch.bool))]:
                if not mask.any():
                    continue
                unc_rows.append({
                    'case_name': name,
                    'branch': br,
                    'group': correctness,
                    'n': int(mask.sum().item()),
                    'predictive_entropy': float(u['predictive_entropy'][mask].mean().item()),
                    'expected_entropy': float(u['expected_entropy'][mask].mean().item()),
                    'MI': float(u['mutual_info'][mask].mean().item()),
                    'variation_ratio': float(u['variation_ratio'][mask].mean().item()),
                    'sample_agreement': float(u['sample_agreement'][mask].mean().item()),
                    'num_unique_preds': float(u['num_unique'][mask].mean().item()),
                    'vote_entropy': float(u['vote_entropy'][mask].mean().item()),
                    'logit_var': float(u['logit_var_mean'][mask].mean().item()),
                })

            is_error = (~correct).cpu().numpy().astype(bool)
            scores = {
                'negative_confidence': -u['probs_mean'].max(dim=-1).values.cpu().numpy(),
                'predictive_entropy': u['predictive_entropy'].cpu().numpy(),
                'expected_entropy': u['expected_entropy'].cpu().numpy(),
                'MI': u['mutual_info'].cpu().numpy(),
                'variation_ratio': u['variation_ratio'].cpu().numpy(),
                'num_unique_preds': u['num_unique'].cpu().numpy(),
                'vote_entropy': u['vote_entropy'].cpu().numpy(),
                'logit_var': u['logit_var_mean'].cpu().numpy(),
            }
            for score_name, score in scores.items():
                err_rows.append({
                    'case_name': name,
                    'branch': br,
                    'score': score_name,
                    'AUROC_error': binary_auroc(score, is_error),
                    'AUPR_error': binary_aupr(score, is_error),
                    'error_rate': float(is_error.mean()),
                })

            # prediction conflict
            conflict_rows.append({
                'case_name': name,
                'branch': br,
                'sample_agreement': float(u['sample_agreement'].mean().item()),
                'variation_ratio': float(u['variation_ratio'].mean().item()),
                'num_unique_preds': float(u['num_unique'].mean().item()),
                'vote_entropy': float(u['vote_entropy'].mean().item()),
                'pred_flip_rate_vs_PM': float(u['pred_samples'].ne(pm_preds[br].unsqueeze(0)).float().mean().item()) if br in pm_preds else np.nan,
            })

            # margin sign conflict
            margins = []
            for s in range(smp_logits.shape[0]):
                margin_s, _ = margin_y_vs_best_wrong(smp_logits[s], labels)
                margins.append(margin_s)
            margins = torch.stack(margins, dim=0)  # [S,N]
            frac_pos = (margins > 0).float().mean(dim=0)
            sign_conflict_rate = 1.0 - torch.maximum(frac_pos, 1.0 - frac_pos)
            margin_sign_rows.append({
                'case_name': name,
                'branch': br,
                'margin_mean': float(margins.mean().item()),
                'margin_std': float(margins.std(unbiased=False).item()),
                'sign_conflict_rate': float(sign_conflict_rate.mean().item()),
                'frac_positive_margin': float(frac_pos.mean().item()),
            })

        # branch conflict inside each MC sample
        if all(br in out['mc_samples'] for br in ['main', 'rep', 'fusion']):
            pred_main = out['mc_samples']['main'].argmax(dim=-1)
            pred_rep = out['mc_samples']['rep'].argmax(dim=-1)
            pred_fusion = out['mc_samples']['fusion'].argmax(dim=-1)
            branch_conflict_rows.append({
                'case_name': name,
                'logits_rep_agree': float(pred_main.eq(pred_rep).float().mean().item()),
                'logits_fusion_agree': float(pred_main.eq(pred_fusion).float().mean().item()),
                'rep_fusion_agree': float(pred_rep.eq(pred_fusion).float().mean().item()),
                'all_three_agree': float((pred_main.eq(pred_rep) & pred_main.eq(pred_fusion)).float().mean().item()),
            })

            margin_main = torch.stack([margin_y_vs_best_wrong(out['mc_samples']['main'][s], labels)[0] for s in range(out['mc_samples']['main'].shape[0])])
            margin_rep = torch.stack([margin_y_vs_best_wrong(out['mc_samples']['rep'][s], labels)[0] for s in range(out['mc_samples']['rep'].shape[0])])
            margin_fusion = torch.stack([margin_y_vs_best_wrong(out['mc_samples']['fusion'][s], labels)[0] for s in range(out['mc_samples']['fusion'].shape[0])])
            sign_main = margin_main > 0
            sign_rep = margin_rep > 0
            sign_fusion = margin_fusion > 0
            conflict = sign_main.ne(sign_rep)
            if conflict.any():
                fusion_conflict_rows.append({
                    'case_name': name,
                    'main_rep_margin_conflict_rate': float(conflict.float().mean().item()),
                    'fusion_follow_main_when_conflict': float(sign_fusion[conflict].eq(sign_main[conflict]).float().mean().item()),
                    'fusion_follow_rep_when_conflict': float(sign_fusion[conflict].eq(sign_rep[conflict]).float().mean().item()),
                    'fusion_negative_when_conflict': float((~sign_fusion[conflict]).float().mean().item()),
                })
            else:
                fusion_conflict_rows.append({
                    'case_name': name,
                    'main_rep_margin_conflict_rate': 0.0,
                    'fusion_follow_main_when_conflict': np.nan,
                    'fusion_follow_rep_when_conflict': np.nan,
                    'fusion_negative_when_conflict': np.nan,
                })

    return {
        'mc_uncertainty': pd.DataFrame(unc_rows),
        'error_detection': pd.DataFrame(err_rows),
        'sample_conflict': pd.DataFrame(conflict_rows),
        'aggregation_conflict': pd.DataFrame(agg_rows),
        'margin_sign_conflict': pd.DataFrame(margin_sign_rows),
        'branch_conflict_inside_mc': pd.DataFrame(branch_conflict_rows),
        'fusion_conflict_under_sampling': pd.DataFrame(fusion_conflict_rows),
    }

mc_tables = mc_uncertainty_tables(case_outputs)
for k, df in mc_tables.items():
    print('\n###', k)
    display(df)
    df.to_csv(SAVE_DIR / f'table_{k}.csv', index=False)



### mc_uncertainty


,case_name,branch,group,n,predictive_entropy,expected_entropy,MI,variation_ratio,sample_agreement,num_unique_preds,vote_entropy,logit_var
0,case_A,main,correct,3188,0.380948,0.360797,0.020152,0.028388,0.971612,1.175031,0.066818,0.538611
1,case_A,main,wrong,595,1.274096,1.205660,0.068436,0.182185,0.817815,2.063866,0.414518,0.590361
2,case_A,main,all,3783,0.521425,0.493679,0.027746,0.052577,0.947423,1.314829,0.121505,0.546750
3,case_A,rep,correct,3155,0.230147,0.162959,0.067188,0.046735,0.953265,1.345483,0.117878,3.839903
4,case_A,rep,wrong,628,0.991457,0.681004,0.310453,0.283997,0.716003,2.794586,0.658362,4.347959
5,case_A,rep,all,3783,0.356529,0.248958,0.107571,0.086122,0.913878,1.586043,0.207601,3.924243
6,case_A,fusion,correct,3288,0.267498,0.246225,0.021273,0.025669,0.974331,1.164538,0.061425,0.738746
7,case_A,fusion,wrong,495,1.095554,1.000446,0.095108,0.180505,0.819495,2.052525,0.413916,0.838276
8,case_A,fusion,all,3783,0.375847,0.344913,0.030934,0.045929,0.954071,1.280730,0.107548,0.751770



### error_detection


,case_name,branch,score,AUROC_error,AUPR_error,error_rate
0,case_A,main,negative_confidence,0.892116,0.577263,0.157283
1,case_A,main,predictive_entropy,0.885540,0.562597,0.157283
2,case_A,main,expected_entropy,0.886107,0.564053,0.157283
3,case_A,main,MI,0.858021,0.458462,0.157283
4,case_A,main,variation_ratio,0.780693,0.483055,0.157283
5,case_A,main,num_unique_preds,0.777843,0.495576,0.157283
6,case_A,main,vote_entropy,0.781591,0.499965,0.157283
7,case_A,main,logit_var,0.604754,0.209961,0.157283
8,case_A,rep,negative_confidence,0.888736,0.587344,0.166006
9,case_A,rep,predictive_entropy,0.888731,0.587135,0.166006



### sample_conflict


...

,case_name,branch,margin_mean,margin_std,sign_conflict_rate,frac_positive_margin
0,case_A,main,3.273575,3.494088,0.037708,0.833188
1,case_A,rep,5.051735,6.040455,0.053939,0.811988
2,case_A,fusion,4.268212,4.022345,0.032554,0.857639



### branch_conflict_inside_mc


,case_name,logits_rep_agree,logits_fusion_agree,rep_fusion_agree,all_three_agree
0,case_A,0.801798,0.907917,0.876104,0.801798



### fusion_conflict_under_sampling


,case_name,main_rep_margin_conflict_rate,fusion_follow_main_when_conflict,fusion_follow_rep_when_conflict,fusion_negative_when_conflict
0,case_A,0.145149,0.565744,0.434256,0.298488


## 14A. 不确定性增益对比：BayesMMRL 必须和 MMRL 基线比较

注意：仅看到 BayesMMRL 的互信息能够识别自身错误，并不能证明 BayesMMRL 的不确定性优于 MMRL。

因此这里新增三类对比：

1. **错误识别能力对比**：MMRL 使用低置信度、预测熵、低决策间隔；BayesMMRL 除这些分数外，再加入互信息、投票分歧率、投票熵等采样不确定性。
2. **风险-覆盖曲线**：按不确定性从高到低拒绝样本，比较相同覆盖率下保留样本的准确率。
3. **不确定性增益摘要**：比较 BayesMMRL 的最佳不确定性分数、Bayes 互信息、以及 MMRL 最佳基线分数，判断 Bayes 的不确定性是否真的提供额外收益。

这部分会生成以下表：

- `uncertainty_baseline_comparison_df`
- `risk_coverage_df`
- `uncertainty_gain_summary_df`
- `uncertainty_score_correlation_df`


In [49]:

# ============================================================
# 14A. 不确定性增益对比：必须把 BayesMMRL 与 MMRL 基线放在一起比较
# ============================================================
# 目的：
#   不能只用 “BayesMMRL 的互信息 AUROC 很高” 来证明 BayesMMRL uncertainty 更好。
#   必须比较：
#     - MMRL 的低置信度 / 预测熵 / 低决策间隔
#     - BayesMMRL 的低置信度 / 预测熵 / 低决策间隔
#     - BayesMMRL 的互信息 / 投票分歧率 / 投票熵
#
# 输出：
#   uncertainty_baseline_comparison_df:
#       每个模型、每个分支、每个风险分数的错误识别 AUROC / AUPR
#
#   risk_coverage_df:
#       相同 coverage 下，按不同风险分数拒绝高风险样本后的保留准确率
#
#   uncertainty_gain_summary_df:
#       MMRL 最佳基线 vs Bayes 互信息 vs Bayes 最佳分数
#
#   uncertainty_score_correlation_df:
#       Bayes 互信息与 Bayes 自身低置信度/熵/低间隔的 Spearman 相关性，
#       用于判断 MI 是否只是重复已有置信度信息。

BRANCH_CN = {
    "main": "主分支",
    "rep": "表示增强分支",
    "fusion": "融合分支",
}

SCORE_CN = {
    "one_minus_confidence": "低置信度",
    "predictive_entropy": "预测熵",
    "one_minus_probability_margin": "低决策间隔",
    "mutual_information": "互信息",
    "variation_ratio": "投票分歧率",
    "vote_entropy": "投票熵",
    "logit_variance": "类别打分方差",
}

SCORE_FAMILY = {
    "one_minus_confidence": "普通概率分数",
    "predictive_entropy": "普通概率分数",
    "one_minus_probability_margin": "普通概率分数",
    "mutual_information": "Bayes采样分数",
    "variation_ratio": "Bayes采样分数",
    "vote_entropy": "Bayes采样分数",
    "logit_variance": "Bayes采样分数",
}


def uncertainty_scores_from_probs(probs):
    """从普通概率分布构造所有模型都能使用的风险分数。
    分数越高，表示越可能错误。
    """
    pred, conf, margin, ent = pred_conf_margin_entropy_from_probs(probs)
    return {
        "one_minus_confidence": 1.0 - conf,
        "predictive_entropy": ent,
        "one_minus_probability_margin": 1.0 - margin,
    }


def bayes_mc_uncertainty_scores(logits_samples):
    """从 Bayes MC samples 构造采样不确定性分数。
    分数越高，表示越可能错误。
    """
    u = predictive_uncertainty_from_mc_logits(logits_samples)
    return {
        "mutual_information": u["mutual_info"],
        "variation_ratio": u["variation_ratio"],
        "vote_entropy": u["vote_entropy"],
        "logit_variance": u["logit_var_mean"],
    }


def _np_float(x):
    if torch.is_tensor(x):
        return x.detach().float().cpu().numpy()
    return np.asarray(x, dtype=float)


def _safe_ratio(num, den):
    if den is None or not np.isfinite(den) or abs(float(den)) < 1e-12:
        return np.nan
    return float(num) / float(den)


def _error_detection_row(pair_name, model_label, method, eval_mode, branch, probs, labels, score_name, score_values):
    pred, conf, margin, ent = pred_conf_margin_entropy_from_probs(probs)
    correct = pred.eq(labels)
    is_error = ~correct

    s = _np_float(score_values)
    y = is_error.detach().cpu().numpy().astype(bool)

    correct_mask = correct.detach().cpu().numpy().astype(bool)
    wrong_mask = ~correct_mask

    correct_mean = float(np.nanmean(s[correct_mask])) if correct_mask.any() else np.nan
    wrong_mean = float(np.nanmean(s[wrong_mask])) if wrong_mask.any() else np.nan

    return {
        "pair_name": pair_name,
        "model_label": model_label,
        "method": method,
        "eval_mode": eval_mode,
        "branch": branch,
        "branch_cn": BRANCH_CN.get(branch, branch),
        "score": score_name,
        "score_cn": SCORE_CN.get(score_name, score_name),
        "score_family": SCORE_FAMILY.get(score_name, "unknown"),
        "n": int(labels.numel()),
        "n_error": int(is_error.sum().item()),
        "error_rate": float(is_error.float().mean().item()),
        "accuracy": float(correct.float().mean().item()),
        "AUROC_error": binary_auroc(s, y),
        "AUPR_error": binary_aupr(s, y),
        "correct_score_mean": correct_mean,
        "wrong_score_mean": wrong_mean,
        "wrong_over_correct_score_mean": _safe_ratio(wrong_mean, correct_mean),
    }


def uncertainty_baseline_comparison_for_pairs(case_outputs, pairings, bayes_mode="mc_prob_mean_probs", branches=("main", "rep", "fusion")):
    rows = []

    for pair in pairings:
        pair_name = pair["pair_name"]
        mmrl_out = case_outputs[pair["mmrl"]]
        bayes_out = case_outputs[pair["bayes"]]
        assert_same_samples(mmrl_out, bayes_out)

        labels = mmrl_out["labels"]

        for br in branches:
            # ---------- MMRL: 只能使用普通概率分数 ----------
            p_m = probs_for_case_mode_branch(mmrl_out, "standard", br)
            mmrl_scores = uncertainty_scores_from_probs(p_m)
            for score_name, score_values in mmrl_scores.items():
                rows.append(_error_detection_row(
                    pair_name=pair_name,
                    model_label="MMRL",
                    method=mmrl_out["cfg_method"],
                    eval_mode="standard",
                    branch=br,
                    probs=p_m,
                    labels=labels,
                    score_name=score_name,
                    score_values=score_values,
                ))

            # ---------- BayesMMRL: 普通概率分数 + MC 采样分数 ----------
            p_b = probs_for_case_mode_branch(bayes_out, bayes_mode, br)
            bayes_scores = uncertainty_scores_from_probs(p_b)

            # 从 MC logits 构造 Bayes 特有分数
            if bayes_out.get("is_bayes", False) and br in bayes_out.get("mc_samples", {}):
                bayes_scores.update(bayes_mc_uncertainty_scores(bayes_out["mc_samples"][br]))

            for score_name, score_values in bayes_scores.items():
                rows.append(_error_detection_row(
                    pair_name=pair_name,
                    model_label="BayesMMRL",
                    method=bayes_out["cfg_method"],
                    eval_mode=bayes_mode,
                    branch=br,
                    probs=p_b,
                    labels=labels,
                    score_name=score_name,
                    score_values=score_values,
                ))

    return pd.DataFrame(rows)


def risk_coverage_curve_for_score(pair_name, model_label, method, eval_mode, branch, probs, labels, score_name, score_values, coverages):
    pred = probs.argmax(dim=-1)
    correct = pred.eq(labels).detach().cpu().numpy().astype(bool)
    is_error = ~correct
    score = _np_float(score_values)

    ok = np.isfinite(score)
    if not ok.all():
        correct = correct[ok]
        is_error = is_error[ok]
        score = score[ok]

    n = len(score)
    order_high_risk = np.argsort(-score)  # 高风险优先拒绝
    rows = []
    full_acc = float(correct.mean()) if n else np.nan
    full_error = float(is_error.mean()) if n else np.nan

    for cov in coverages:
        cov = float(cov)
        keep_n = int(round(n * cov))
        keep_n = max(1, min(n, keep_n))
        reject_n = n - keep_n

        keep_idx = order_high_risk[reject_n:]
        reject_idx = order_high_risk[:reject_n]

        retained_acc = float(correct[keep_idx].mean()) if keep_n > 0 else np.nan
        retained_error = float(is_error[keep_idx].mean()) if keep_n > 0 else np.nan
        rejected_error = float(is_error[reject_idx].mean()) if reject_n > 0 else np.nan

        rows.append({
            "pair_name": pair_name,
            "model_label": model_label,
            "method": method,
            "eval_mode": eval_mode,
            "branch": branch,
            "branch_cn": BRANCH_CN.get(branch, branch),
            "score": score_name,
            "score_cn": SCORE_CN.get(score_name, score_name),
            "score_family": SCORE_FAMILY.get(score_name, "unknown"),
            "coverage": cov,
            "n_total": int(n),
            "n_keep": int(keep_n),
            "n_reject": int(reject_n),
            "full_accuracy": full_acc,
            "retained_accuracy": retained_acc,
            "accuracy_gain_vs_full": retained_acc - full_acc if np.isfinite(retained_acc) and np.isfinite(full_acc) else np.nan,
            "full_error_rate": full_error,
            "retained_error_rate": retained_error,
            "rejected_error_rate": rejected_error,
        })

    return rows


def risk_coverage_for_pairs(case_outputs, pairings, bayes_mode="mc_prob_mean_probs", branches=("fusion",), coverages=(1.0, 0.98, 0.95, 0.90, 0.85, 0.80, 0.70, 0.60)):
    rows = []

    for pair in pairings:
        pair_name = pair["pair_name"]
        mmrl_out = case_outputs[pair["mmrl"]]
        bayes_out = case_outputs[pair["bayes"]]
        assert_same_samples(mmrl_out, bayes_out)

        labels = mmrl_out["labels"]

        for br in branches:
            # MMRL
            p_m = probs_for_case_mode_branch(mmrl_out, "standard", br)
            for score_name, score_values in uncertainty_scores_from_probs(p_m).items():
                rows.extend(risk_coverage_curve_for_score(
                    pair_name, "MMRL", mmrl_out["cfg_method"], "standard", br, p_m, labels,
                    score_name, score_values, coverages
                ))

            # BayesMMRL
            p_b = probs_for_case_mode_branch(bayes_out, bayes_mode, br)
            bayes_scores = uncertainty_scores_from_probs(p_b)
            if bayes_out.get("is_bayes", False) and br in bayes_out.get("mc_samples", {}):
                bayes_scores.update(bayes_mc_uncertainty_scores(bayes_out["mc_samples"][br]))

            for score_name, score_values in bayes_scores.items():
                rows.extend(risk_coverage_curve_for_score(
                    pair_name, "BayesMMRL", bayes_out["cfg_method"], bayes_mode, br, p_b, labels,
                    score_name, score_values, coverages
                ))

    return pd.DataFrame(rows)


def uncertainty_gain_summary(error_df, rc_df, target_branch="fusion", target_coverage=0.90):
    rows = []

    if error_df.empty:
        return pd.DataFrame()

    for pair_name in sorted(error_df["pair_name"].dropna().unique()):
        for br in sorted(error_df.loc[error_df["pair_name"] == pair_name, "branch"].dropna().unique()):
            sub = error_df[(error_df["pair_name"] == pair_name) & (error_df["branch"] == br)].copy()
            if sub.empty:
                continue

            mmrl = sub[sub["model_label"] == "MMRL"]
            bayes = sub[sub["model_label"] == "BayesMMRL"]
            if mmrl.empty or bayes.empty:
                continue

            mmrl_best = mmrl.loc[mmrl["AUROC_error"].idxmax()]
            bayes_best = bayes.loc[bayes["AUROC_error"].idxmax()]

            bayes_mi = bayes[bayes["score"] == "mutual_information"]
            if not bayes_mi.empty:
                bayes_mi_row = bayes_mi.iloc[0]
                bayes_mi_auc = float(bayes_mi_row["AUROC_error"])
                bayes_mi_aupr = float(bayes_mi_row["AUPR_error"])
            else:
                bayes_mi_auc = np.nan
                bayes_mi_aupr = np.nan

            # 风险-覆盖曲线上的 90% coverage 对比
            rc_pair = rc_df[(rc_df["pair_name"] == pair_name) & (rc_df["branch"] == br) & (np.isclose(rc_df["coverage"], target_coverage))]
            mmrl_rc = rc_pair[rc_pair["model_label"] == "MMRL"]
            bayes_rc = rc_pair[rc_pair["model_label"] == "BayesMMRL"]

            mmrl_best_retained = np.nan
            mmrl_best_retained_score = None
            bayes_best_retained = np.nan
            bayes_best_retained_score = None
            bayes_mi_retained = np.nan

            if not mmrl_rc.empty:
                idx = mmrl_rc["retained_accuracy"].idxmax()
                mmrl_best_retained = float(mmrl_rc.loc[idx, "retained_accuracy"])
                mmrl_best_retained_score = mmrl_rc.loc[idx, "score"]

            if not bayes_rc.empty:
                idx = bayes_rc["retained_accuracy"].idxmax()
                bayes_best_retained = float(bayes_rc.loc[idx, "retained_accuracy"])
                bayes_best_retained_score = bayes_rc.loc[idx, "score"]

                mi_rc = bayes_rc[bayes_rc["score"] == "mutual_information"]
                if not mi_rc.empty:
                    bayes_mi_retained = float(mi_rc.iloc[0]["retained_accuracy"])

            bayes_best_minus_mmrl_best_auc = float(bayes_best["AUROC_error"]) - float(mmrl_best["AUROC_error"])
            bayes_mi_minus_mmrl_best_auc = bayes_mi_auc - float(mmrl_best["AUROC_error"]) if np.isfinite(bayes_mi_auc) else np.nan

            if np.isfinite(bayes_best_minus_mmrl_best_auc):
                if bayes_best_minus_mmrl_best_auc > 0.01:
                    conclusion = "BayesMMRL 不确定性优于 MMRL 基线"
                elif bayes_best_minus_mmrl_best_auc >= -0.005:
                    conclusion = "BayesMMRL 与 MMRL 基线接近"
                else:
                    conclusion = "BayesMMRL 未超过 MMRL 基线"
            else:
                conclusion = "无法判断"

            rows.append({
                "pair_name": pair_name,
                "branch": br,
                "branch_cn": BRANCH_CN.get(br, br),

                "MMRL_best_score": mmrl_best["score"],
                "MMRL_best_score_cn": mmrl_best["score_cn"],
                "MMRL_best_AUROC_error": float(mmrl_best["AUROC_error"]),
                "MMRL_best_AUPR_error": float(mmrl_best["AUPR_error"]),

                "Bayes_best_score": bayes_best["score"],
                "Bayes_best_score_cn": bayes_best["score_cn"],
                "Bayes_best_AUROC_error": float(bayes_best["AUROC_error"]),
                "Bayes_best_AUPR_error": float(bayes_best["AUPR_error"]),

                "Bayes_MI_AUROC_error": bayes_mi_auc,
                "Bayes_MI_AUPR_error": bayes_mi_aupr,

                "Bayes_best_minus_MMRL_best_AUROC": bayes_best_minus_mmrl_best_auc,
                "Bayes_MI_minus_MMRL_best_AUROC": bayes_mi_minus_mmrl_best_auc,

                f"MMRL_best_retained_acc_at_{target_coverage:.2f}_coverage": mmrl_best_retained,
                f"MMRL_best_retained_score_at_{target_coverage:.2f}_coverage": mmrl_best_retained_score,
                f"Bayes_best_retained_acc_at_{target_coverage:.2f}_coverage": bayes_best_retained,
                f"Bayes_best_retained_score_at_{target_coverage:.2f}_coverage": bayes_best_retained_score,
                f"Bayes_MI_retained_acc_at_{target_coverage:.2f}_coverage": bayes_mi_retained,

                "conclusion_hint": conclusion,
            })

    return pd.DataFrame(rows)


def uncertainty_score_correlations(case_outputs, pairings, bayes_mode="mc_prob_mean_probs", branches=("fusion", "main", "rep")):
    rows = []

    for pair in pairings:
        pair_name = pair["pair_name"]
        bayes_out = case_outputs[pair["bayes"]]
        if not bayes_out.get("is_bayes", False):
            continue

        for br in branches:
            if br not in bayes_out.get("mc_samples", {}):
                continue

            probs = probs_for_case_mode_branch(bayes_out, bayes_mode, br)
            prob_scores = uncertainty_scores_from_probs(probs)
            mc_scores = bayes_mc_uncertainty_scores(bayes_out["mc_samples"][br])
            if "mutual_information" not in mc_scores:
                continue

            mi = pd.Series(_np_float(mc_scores["mutual_information"]))
            for score_name, score_values in prob_scores.items():
                s = pd.Series(_np_float(score_values))
                rows.append({
                    "pair_name": pair_name,
                    "branch": br,
                    "branch_cn": BRANCH_CN.get(br, br),
                    "score_a": "mutual_information",
                    "score_a_cn": "互信息",
                    "score_b": score_name,
                    "score_b_cn": SCORE_CN.get(score_name, score_name),
                    "spearman_corr": mi.corr(s, method="spearman"),
                    "pearson_corr": mi.corr(s, method="pearson"),
                    "note": "相关性高说明互信息可能与普通置信度/熵信息重叠；相关性低说明互信息可能提供额外排序信息。",
                })

    return pd.DataFrame(rows)


# 生成新增表
uncertainty_baseline_comparison_df = uncertainty_baseline_comparison_for_pairs(
    case_outputs,
    PAIRINGS,
    bayes_mode="mc_prob_mean_probs",
    branches=("main", "rep", "fusion"),
)

risk_coverage_df = risk_coverage_for_pairs(
    case_outputs,
    PAIRINGS,
    bayes_mode="mc_prob_mean_probs",
    branches=("main", "rep", "fusion"),
    coverages=(1.0, 0.98, 0.95, 0.90, 0.85, 0.80, 0.70, 0.60),
)

uncertainty_gain_summary_df = uncertainty_gain_summary(
    uncertainty_baseline_comparison_df,
    risk_coverage_df,
    target_coverage=0.90,
)

uncertainty_score_correlation_df = uncertainty_score_correlations(
    case_outputs,
    PAIRINGS,
    bayes_mode="mc_prob_mean_probs",
    branches=("main", "rep", "fusion"),
)

display(uncertainty_baseline_comparison_df.sort_values(["pair_name", "branch", "model_label", "AUROC_error"], ascending=[True, True, True, False]))
display(uncertainty_gain_summary_df)
display(risk_coverage_df[(risk_coverage_df["branch"] == "fusion") & (risk_coverage_df["coverage"].isin([1.0, 0.95, 0.90, 0.80]))].sort_values(["pair_name", "branch", "coverage", "model_label", "retained_accuracy"], ascending=[True, True, False, True, False]))
display(uncertainty_score_correlation_df)

# 保存 CSV
uncertainty_baseline_comparison_df.to_csv(SAVE_DIR / "table_uncertainty_baseline_comparison.csv", index=False)
risk_coverage_df.to_csv(SAVE_DIR / "table_risk_coverage.csv", index=False)
uncertainty_gain_summary_df.to_csv(SAVE_DIR / "table_uncertainty_gain_summary.csv", index=False)
uncertainty_score_correlation_df.to_csv(SAVE_DIR / "table_uncertainty_score_correlation.csv", index=False)


,pair_name,model_label,method,eval_mode,branch,branch_cn,score,score_cn,score_family,n,n_error,error_rate,accuracy,AUROC_error,AUPR_error,correct_score_mean,wrong_score_mean,wrong_over_correct_score_mean
24,case_B__VS__case_A,BayesMMRL,BayesMMRL,mc_prob_mean_probs,fusion,融合分支,predictive_entropy,预测熵,普通概率分数,3783,495,0.130849,0.869151,0.898350,0.520132,0.267498,1.095554,4.095566
23,case_B__VS__case_A,BayesMMRL,BayesMMRL,mc_prob_mean_probs,fusion,融合分支,one_minus_confidence,低置信度,普通概率分数,3783,495,0.130849,0.869151,0.898133,0.519331,0.084302,0.385525,4.573148
25,case_B__VS__case_A,BayesMMRL,BayesMMRL,mc_prob_mean_probs,fusion,融合分支,one_minus_probability_margin,低决策间隔,普通概率分数,3783,495,0.130849,0.869151,0.893264,0.496205,0.139805,0.597346,4.272695
26,case_B__VS__case_A,BayesMMRL,BayesMMRL,mc_prob_mean_probs,fusion,融合分支,mutual_information,互信息,Bayes采样分数,3783,495,0.130849,0.869151,0.884976,0.481142,0.021273,0.095108,4.470898
28,case_B__VS__case_A,BayesMMRL,BayesMMRL,mc_prob_mean_probs,fusion,融合分支,vote_entropy,投票熵,Bayes采样分数,3783,495,0.130849,0.869151,0.784042,0.452991,0.061425,0.413916,6.738602
27,case_B__VS__case_A,BayesMMRL,BayesMMRL,mc_prob_mean_probs,fusion,融合分支,variation_ratio,投票分歧率,Bayes采样分数,3783,495,0.130849,0.869151,0.783570,0.446030,0.025669,0.180505,7.031998
29,case_B__VS__case_A,BayesMMRL,BayesMMRL,mc_prob_mean_probs,fusion,融合分支,logit_variance,类别打分方差,Bayes采样分数,3783,495,0.130849,0.869151,0.661963,0.206781,0.738747,0.838276,1.134728
21,case_B__VS__case_A,MMRL,MMRL,standard,fusion,融合分支,predictive_entropy,预测熵,普通概率分数,3783,478,0.126355,0.873645,0.912799,0.605232,0.234820,1.145640,4.878790
20,case_B__VS__case_A,MMRL,MMRL,standard,fusion,融合分支,one_minus_confidence,低置信度,普通概率分数,3783,478,0.126355,0.873645,0.912157,0.604776,0.070495,0.389270,5.521984
22,case_B__VS__case_A,MMRL,MMRL,standard,fusion,融合分支,one_minus_probability_margin,低决策间隔,普通概率分数,3783,478,0.126355,0.873645,0.906052,0.538871,0.117809,0.590125,5.009164


,pair_name,branch,branch_cn,MMRL_best_score,MMRL_best_score_cn,MMRL_best_AUROC_error,MMRL_best_AUPR_error,Bayes_best_score,Bayes_best_score_cn,Bayes_best_AUROC_error,Bayes_best_AUPR_error,Bayes_MI_AUROC_error,Bayes_MI_AUPR_error,Bayes_best_minus_MMRL_best_AUROC,Bayes_MI_minus_MMRL_best_AUROC,MMRL_best_retained_acc_at_0.90_coverage,MMRL_best_retained_score_at_0.90_coverage,Bayes_best_retained_acc_at_0.90_coverage,Bayes_best_retained_score_at_0.90_coverage,Bayes_MI_retained_acc_at_0.90_coverage,conclusion_hint
0,case_B__VS__case_A,fusion,融合分支,predictive_entropy,预测熵,0.912799,0.605232,predictive_entropy,预测熵,0.898350,0.520132,0.884976,0.481142,-0.014449,-0.027823,0.928341,one_minus_confidence,0.917474,predictive_entropy,0.912775,BayesMMRL 未超过 MMRL 基线
1,case_B__VS__case_A,main,主分支,one_minus_confidence,低置信度,0.906457,0.606662,one_minus_confidence,低置信度,0.892116,0.577263,0.858021,0.458462,-0.014340,-0.048436,0.918943,one_minus_confidence,0.893979,predictive_entropy,0.881938,BayesMMRL 未超过 MMRL 基线
2,case_B__VS__case_A,rep,表示增强分支,predictive_entropy,预测熵,0.889681,0.612405,one_minus_confidence,低置信度,0.888736,0.587344,0.880505,0.565541,-0.000945,-0.009176,0.879589,predictive_entropy,0.887518,predictive_entropy,0.887225,BayesMMRL 与 MMRL 基线接近


,pair_name,model_label,method,eval_mode,branch,branch_cn,score,score_cn,score_family,coverage,n_total,n_keep,n_reject,full_accuracy,retained_accuracy,accuracy_gain_vs_full,full_error_rate,retained_error_rate,rejected_error_rate
184,case_B__VS__case_A,BayesMMRL,BayesMMRL,mc_prob_mean_probs,fusion,融合分支,one_minus_confidence,低置信度,普通概率分数,1.00,3783,3783,0,0.869151,0.869151,0.000000,0.130849,0.130849,NaN
192,case_B__VS__case_A,BayesMMRL,BayesMMRL,mc_prob_mean_probs,fusion,融合分支,predictive_entropy,预测熵,普通概率分数,1.00,3783,3783,0,0.869151,0.869151,0.000000,0.130849,0.130849,NaN
200,case_B__VS__case_A,BayesMMRL,BayesMMRL,mc_prob_mean_probs,fusion,融合分支,one_minus_probability_margin,低决策间隔,普通概率分数,1.00,3783,3783,0,0.869151,0.869151,0.000000,0.130849,0.130849,NaN
208,case_B__VS__case_A,BayesMMRL,BayesMMRL,mc_prob_mean_probs,fusion,融合分支,mutual_information,互信息,Bayes采样分数,1.00,3783,3783,0,0.869151,0.869151,0.000000,0.130849,0.130849,NaN
216,case_B__VS__case_A,BayesMMRL,BayesMMRL,mc_prob_mean_probs,fusion,融合分支,variation_ratio,投票分歧率,Bayes采样分数,1.00,3783,3783,0,0.869151,0.869151,0.000000,0.130849,0.130849,NaN
224,case_B__VS__case_A,BayesMMRL,BayesMMRL,mc_prob_mean_probs,fusion,融合分支,vote_entropy,投票熵,Bayes采样分数,1.00,3783,3783,0,0.869151,0.869151,0.000000,0.130849,0.130849,NaN
232,case_B__VS__case_A,BayesMMRL,BayesMMRL,mc_prob_mean_probs,fusion,融合分支,logit_variance,类别打分方差,Bayes采样分数,1.00,3783,3783,0,0.869151,0.869151,0.000000,0.130849,0.130849,NaN
160,case_B__VS__case_A,MMRL,MMRL,standard,fusion,融合分支,one_minus_confidence,低置信度,普通概率分数,1.00,3783,3783,0,0.873645,0.873645,0.000000,0.126355,0.126355,NaN
168,case_B__VS__case_A,MMRL,MMRL,standard,fusion,融合分支,predictive_entropy,预测熵,普通概率分数,1.00,3783,3783,0,0.873645,0.873645,0.000000,0.126355,0.126355,NaN
176,case_B__VS__case_A,MMRL,MMRL,standard,fusion,融合分支,one_minus_probability_margin,低决策间隔,普通概率分数,1.00,3783,3783,0,0.873645,0.873645,0.000000,0.126355,0.126355,NaN


,pair_name,branch,branch_cn,score_a,score_a_cn,score_b,score_b_cn,spearman_corr,pearson_corr,note
0,case_B__VS__case_A,main,主分支,mutual_information,互信息,one_minus_confidence,低置信度,0.951863,0.833190,相关性高说明互信息可能与普通置信度/熵信息重叠；相关性低说明互信息可能提供额外排序信息。
1,case_B__VS__case_A,main,主分支,mutual_information,互信息,predictive_entropy,预测熵,0.967222,0.874750,相关性高说明互信息可能与普通置信度/熵信息重叠；相关性低说明互信息可能提供额外排序信息。
2,case_B__VS__case_A,main,主分支,mutual_information,互信息,one_minus_probability_margin,低决策间隔,0.934379,0.774658,相关性高说明互信息可能与普通置信度/熵信息重叠；相关性低说明互信息可能提供额外排序信息。
3,case_B__VS__case_A,rep,表示增强分支,mutual_information,互信息,one_minus_confidence,低置信度,0.985292,0.925748,相关性高说明互信息可能与普通置信度/熵信息重叠；相关性低说明互信息可能提供额外排序信息。
4,case_B__VS__case_A,rep,表示增强分支,mutual_information,互信息,predictive_entropy,预测熵,0.991135,0.965330,相关性高说明互信息可能与普通置信度/熵信息重叠；相关性低说明互信息可能提供额外排序信息。
5,case_B__VS__case_A,rep,表示增强分支,mutual_information,互信息,one_minus_probability_margin,低决策间隔,0.978305,0.874854,相关性高说明互信息可能与普通置信度/熵信息重叠；相关性低说明互信息可能提供额外排序信息。
6,case_B__VS__case_A,fusion,融合分支,mutual_information,互信息,one_minus_confidence,低置信度,0.974965,0.883148,相关性高说明互信息可能与普通置信度/熵信息重叠；相关性低说明互信息可能提供额外排序信息。
7,case_B__VS__case_A,fusion,融合分支,mutual_information,互信息,predictive_entropy,预测熵,0.983222,0.927048,相关性高说明互信息可能与普通置信度/熵信息重叠；相关性低说明互信息可能提供额外排序信息。
8,case_B__VS__case_A,fusion,融合分支,mutual_information,互信息,one_minus_probability_margin,低决策间隔,0.965421,0.825744,相关性高说明互信息可能与普通置信度/熵信息重叠；相关性低说明互信息可能提供额外排序信息。


## 14B. 同分数公平比较：MMRL 与 BayesMMRL 的风险分数必须先逐项对比

`uncertainty_gain_summary_df` 给的是各模型的最优风险分数，但 MMRL 最优可能是低决策间隔，BayesMMRL 最优可能是低置信度，这不是同一指标下的公平比较。

本节新增四张表：

- `same_score_uncertainty_comparison_df`：同分数比较，低置信度对低置信度、预测熵对预测熵、低决策间隔对低决策间隔。
- `best_common_uncertainty_comparison_df`：只在共有风险分数集合内，各模型的最佳结果。
- `bayes_specific_uncertainty_value_df`：互信息、投票分歧率等 Bayes 特有采样分数的相对价值。
- `risk_coverage_same_score_df`：风险-覆盖曲线的同分数比较。


In [50]:

# ============================================================
# 14B. 同分数公平比较
# ============================================================

COMMON_RISK_SCORES = [
    "one_minus_confidence",
    "predictive_entropy",
    "one_minus_probability_margin",
]

BAYES_ONLY_RISK_SCORES = [
    "mutual_information",
    "variation_ratio",
    "vote_entropy",
    "logit_variance",
]


def build_same_score_uncertainty_comparison(error_df):
    """共有风险分数的同分数比较。"""
    if error_df is None or error_df.empty:
        return pd.DataFrame()

    d = error_df[
        error_df["score"].isin(COMMON_RISK_SCORES)
        & error_df["model_label"].isin(["MMRL", "BayesMMRL"])
    ].copy()

    keys = ["pair_name", "branch", "branch_cn", "score", "score_cn"]
    value_cols = [
        "AUROC_error",
        "AUPR_error",
        "accuracy",
        "error_rate",
        "correct_score_mean",
        "wrong_score_mean",
        "wrong_over_correct_score_mean",
    ]

    piv = d.pivot_table(index=keys, columns="model_label", values=value_cols, aggfunc="first")
    if piv.empty:
        return pd.DataFrame()

    piv.columns = [f"{metric}_{model}" for metric, model in piv.columns.to_flat_index()]
    out = piv.reset_index()

    for metric in value_cols:
        b = f"{metric}_BayesMMRL"
        m = f"{metric}_MMRL"
        if b in out.columns and m in out.columns:
            out[f"delta_{metric}_Bayes_minus_MMRL"] = out[b] - out[m]

    def judge(row):
        dauc = row.get("delta_AUROC_error_Bayes_minus_MMRL", np.nan)
        daupr = row.get("delta_AUPR_error_Bayes_minus_MMRL", np.nan)
        if not np.isfinite(dauc):
            return "无法判断"
        if dauc > 0.01 and (not np.isfinite(daupr) or daupr >= -0.005):
            return "BayesMMRL 同分数更好"
        if dauc < -0.01:
            return "MMRL 同分数更好"
        return "两者接近"

    out["same_score_conclusion_hint"] = out.apply(judge, axis=1)
    return out.sort_values(["pair_name", "branch", "score"]).reset_index(drop=True)


def build_best_common_uncertainty_comparison(error_df):
    """只在共有风险分数集合内比较各模型最佳结果。"""
    if error_df is None or error_df.empty:
        return pd.DataFrame()

    d = error_df[
        error_df["score"].isin(COMMON_RISK_SCORES)
        & error_df["model_label"].isin(["MMRL", "BayesMMRL"])
    ].copy()

    rows = []
    for (pair_name, branch), g in d.groupby(["pair_name", "branch"], dropna=False):
        gm = g[g["model_label"] == "MMRL"]
        gb = g[g["model_label"] == "BayesMMRL"]
        if gm.empty or gb.empty:
            continue

        bm = gm.loc[gm["AUROC_error"].idxmax()]
        bb = gb.loc[gb["AUROC_error"].idxmax()]
        branch_cn = g["branch_cn"].iloc[0] if "branch_cn" in g.columns else branch

        rows.append({
            "pair_name": pair_name,
            "branch": branch,
            "branch_cn": branch_cn,
            "MMRL_best_common_score": bm["score"],
            "MMRL_best_common_score_cn": bm["score_cn"],
            "MMRL_best_common_AUROC_error": float(bm["AUROC_error"]),
            "MMRL_best_common_AUPR_error": float(bm["AUPR_error"]),
            "Bayes_best_common_score": bb["score"],
            "Bayes_best_common_score_cn": bb["score_cn"],
            "Bayes_best_common_AUROC_error": float(bb["AUROC_error"]),
            "Bayes_best_common_AUPR_error": float(bb["AUPR_error"]),
            "delta_best_common_AUROC_Bayes_minus_MMRL": float(bb["AUROC_error"]) - float(bm["AUROC_error"]),
            "delta_best_common_AUPR_Bayes_minus_MMRL": float(bb["AUPR_error"]) - float(bm["AUPR_error"]),
        })

    out = pd.DataFrame(rows)
    if out.empty:
        return out

    def judge(row):
        dauc = row["delta_best_common_AUROC_Bayes_minus_MMRL"]
        daupr = row["delta_best_common_AUPR_Bayes_minus_MMRL"]
        if dauc > 0.01 and daupr >= -0.005:
            return "BayesMMRL 共有风险分数最优表现更好"
        if dauc < -0.01:
            return "MMRL 共有风险分数最优表现更好"
        return "两者接近"

    out["best_common_conclusion_hint"] = out.apply(judge, axis=1)
    return out


def build_bayes_specific_uncertainty_value(error_df):
    """Bayes 特有采样风险分数不能做同分数比较，单独评估其相对价值。"""
    if error_df is None or error_df.empty:
        return pd.DataFrame()

    rows = []
    for (pair_name, branch), g in error_df.groupby(["pair_name", "branch"], dropna=False):
        mmrl_common = g[(g["model_label"] == "MMRL") & (g["score"].isin(COMMON_RISK_SCORES))]
        bayes_common = g[(g["model_label"] == "BayesMMRL") & (g["score"].isin(COMMON_RISK_SCORES))]
        bayes_only = g[(g["model_label"] == "BayesMMRL") & (g["score"].isin(BAYES_ONLY_RISK_SCORES))]

        if mmrl_common.empty or bayes_common.empty or bayes_only.empty:
            continue

        mmrl_best = mmrl_common.loc[mmrl_common["AUROC_error"].idxmax()]
        bayes_best = bayes_common.loc[bayes_common["AUROC_error"].idxmax()]
        branch_cn = g["branch_cn"].iloc[0] if "branch_cn" in g.columns else branch

        for _, r in bayes_only.iterrows():
            rows.append({
                "pair_name": pair_name,
                "branch": branch,
                "branch_cn": branch_cn,
                "bayes_specific_score": r["score"],
                "bayes_specific_score_cn": r["score_cn"],
                "bayes_specific_AUROC_error": float(r["AUROC_error"]),
                "bayes_specific_AUPR_error": float(r["AUPR_error"]),
                "MMRL_best_common_score": mmrl_best["score"],
                "MMRL_best_common_score_cn": mmrl_best["score_cn"],
                "MMRL_best_common_AUROC_error": float(mmrl_best["AUROC_error"]),
                "MMRL_best_common_AUPR_error": float(mmrl_best["AUPR_error"]),
                "Bayes_best_common_score": bayes_best["score"],
                "Bayes_best_common_score_cn": bayes_best["score_cn"],
                "Bayes_best_common_AUROC_error": float(bayes_best["AUROC_error"]),
                "Bayes_best_common_AUPR_error": float(bayes_best["AUPR_error"]),
                "delta_vs_MMRL_best_common_AUROC": float(r["AUROC_error"]) - float(mmrl_best["AUROC_error"]),
                "delta_vs_MMRL_best_common_AUPR": float(r["AUPR_error"]) - float(mmrl_best["AUPR_error"]),
                "delta_vs_Bayes_best_common_AUROC": float(r["AUROC_error"]) - float(bayes_best["AUROC_error"]),
                "delta_vs_Bayes_best_common_AUPR": float(r["AUPR_error"]) - float(bayes_best["AUPR_error"]),
            })

    out = pd.DataFrame(rows)
    if out.empty:
        return out

    def judge(row):
        d_m = row["delta_vs_MMRL_best_common_AUROC"]
        d_b = row["delta_vs_Bayes_best_common_AUROC"]
        if d_m > 0.01 and d_b > 0:
            return "Bayes特有分数超过 MMRL 基线且超过 Bayes普通分数"
        if d_m > 0 and d_b <= 0:
            return "Bayes特有分数略超 MMRL 基线，但不如 Bayes普通分数"
        if d_m <= 0:
            return "Bayes特有分数未超过 MMRL 最佳普通基线"
        return "需要结合AUPR判断"

    out["bayes_specific_conclusion_hint"] = out.apply(judge, axis=1)
    return out


def build_risk_coverage_same_score(rc_df):
    """风险-覆盖曲线中的同分数比较。"""
    if rc_df is None or rc_df.empty:
        return pd.DataFrame()

    d = rc_df[
        rc_df["score"].isin(COMMON_RISK_SCORES)
        & rc_df["model_label"].isin(["MMRL", "BayesMMRL"])
    ].copy()

    keys = ["pair_name", "branch", "branch_cn", "score", "score_cn", "coverage"]
    value_cols = [
        "full_accuracy",
        "retained_accuracy",
        "accuracy_gain_vs_full",
        "retained_error_rate",
        "rejected_error_rate",
    ]

    piv = d.pivot_table(index=keys, columns="model_label", values=value_cols, aggfunc="first")
    if piv.empty:
        return pd.DataFrame()

    piv.columns = [f"{metric}_{model}" for metric, model in piv.columns.to_flat_index()]
    out = piv.reset_index()

    for metric in value_cols:
        b = f"{metric}_BayesMMRL"
        m = f"{metric}_MMRL"
        if b in out.columns and m in out.columns:
            out[f"delta_{metric}_Bayes_minus_MMRL"] = out[b] - out[m]

    return out.sort_values(["pair_name", "branch", "score", "coverage"], ascending=[True, True, True, False]).reset_index(drop=True)


same_score_uncertainty_comparison_df = build_same_score_uncertainty_comparison(uncertainty_baseline_comparison_df)
best_common_uncertainty_comparison_df = build_best_common_uncertainty_comparison(uncertainty_baseline_comparison_df)
bayes_specific_uncertainty_value_df = build_bayes_specific_uncertainty_value(uncertainty_baseline_comparison_df)
risk_coverage_same_score_df = build_risk_coverage_same_score(risk_coverage_df)

display(same_score_uncertainty_comparison_df)
display(best_common_uncertainty_comparison_df)
display(bayes_specific_uncertainty_value_df)
display(
    risk_coverage_same_score_df[
        (risk_coverage_same_score_df["branch"] == "fusion")
        & (risk_coverage_same_score_df["coverage"].isin([1.0, 0.95, 0.90, 0.80]))
    ]
)

same_score_uncertainty_comparison_df.to_csv(SAVE_DIR / "table_same_score_uncertainty_comparison.csv", index=False)
best_common_uncertainty_comparison_df.to_csv(SAVE_DIR / "table_best_common_uncertainty_comparison.csv", index=False)
bayes_specific_uncertainty_value_df.to_csv(SAVE_DIR / "table_bayes_specific_uncertainty_value.csv", index=False)
risk_coverage_same_score_df.to_csv(SAVE_DIR / "table_risk_coverage_same_score.csv", index=False)


,pair_name,branch,branch_cn,score,score_cn,AUPR_error_BayesMMRL,AUPR_error_MMRL,AUROC_error_BayesMMRL,AUROC_error_MMRL,accuracy_BayesMMRL,accuracy_MMRL,correct_score_mean_BayesMMRL,correct_score_mean_MMRL,error_rate_BayesMMRL,error_rate_MMRL,wrong_over_correct_score_mean_BayesMMRL,wrong_over_correct_score_mean_MMRL,wrong_score_mean_BayesMMRL,wrong_score_mean_MMRL,delta_AUROC_error_Bayes_minus_MMRL,delta_AUPR_error_Bayes_minus_MMRL,delta_accuracy_Bayes_minus_MMRL,delta_error_rate_Bayes_minus_MMRL,delta_correct_score_mean_Bayes_minus_MMRL,delta_wrong_score_mean_Bayes_minus_MMRL,delta_wrong_over_correct_score_mean_Bayes_minus_MMRL,same_score_conclusion_hint
0,case_B__VS__case_A,fusion,融合分支,one_minus_confidence,低置信度,0.519331,0.604776,0.898133,0.912157,0.869151,0.873645,0.084302,0.070495,0.130849,0.126355,4.573148,5.521984,0.385525,0.389270,-0.014023,-0.085445,-0.004494,0.004494,0.013807,-0.003744,-0.948836,MMRL 同分数更好
1,case_B__VS__case_A,fusion,融合分支,one_minus_probability_margin,低决策间隔,0.496205,0.538871,0.893264,0.906052,0.869151,0.873645,0.139805,0.117809,0.130849,0.126355,4.272695,5.009164,0.597346,0.590125,-0.012789,-0.042665,-0.004494,0.004494,0.021996,0.007221,-0.736469,MMRL 同分数更好
2,case_B__VS__case_A,fusion,融合分支,predictive_entropy,预测熵,0.520132,0.605232,0.898350,0.912799,0.869151,0.873645,0.267498,0.234820,0.130849,0.126355,4.095566,4.878790,1.095554,1.145640,-0.014449,-0.085100,-0.004494,0.004494,0.032677,-0.050086,-0.783223,MMRL 同分数更好
3,case_B__VS__case_A,main,主分支,one_minus_confidence,低置信度,0.577263,0.606662,0.892116,0.906457,0.842717,0.863865,0.116407,0.110792,0.157283,0.136135,3.789263,4.138131,0.441098,0.458473,-0.014340,-0.029399,-0.021147,0.021147,0.005615,-0.017375,-0.348868,MMRL 同分数更好
4,case_B__VS__case_A,main,主分支,one_minus_probability_margin,低决策间隔,0.524946,0.572258,0.886487,0.902977,0.842717,0.863865,0.188433,0.172951,0.157283,0.136135,3.540921,3.864500,0.667227,0.668368,-0.016491,-0.047311,-0.021147,0.021147,0.015483,-0.001141,-0.323579,MMRL 同分数更好
5,case_B__VS__case_A,main,主分支,predictive_entropy,预测熵,0.562597,0.564152,0.885540,0.896922,0.842717,0.863865,0.380948,0.402049,0.157283,0.136135,3.344538,3.529091,1.274096,1.418868,-0.011382,-0.001555,-0.021147,0.021147,-0.021101,-0.144772,-0.184553,MMRL 同分数更好
6,case_B__VS__case_A,rep,表示增强分支,one_minus_confidence,低置信度,0.587344,0.593145,0.888736,0.884712,0.833994,0.823685,0.076019,0.044656,0.166006,0.176315,4.851130,5.861320,0.368778,0.261746,0.004024,-0.005802,0.010309,-0.010309,0.031363,0.107033,-1.010190,两者接近
7,case_B__VS__case_A,rep,表示增强分支,one_minus_probability_margin,低决策间隔,0.560376,0.562616,0.884877,0.880527,0.833994,0.823685,0.127749,0.078920,0.166006,0.176315,4.534128,5.419044,0.579230,0.427668,0.004350,-0.002240,0.010309,-0.010309,0.048829,0.151561,-0.884916,两者接近
8,case_B__VS__case_A,rep,表示增强分支,predictive_entropy,预测熵,0.587135,0.612405,0.888731,0.889681,0.833994,0.823685,0.230147,0.132737,0.166006,0.176315,4.307924,5.417530,0.991457,0.719106,-0.000950,-0.025270,0.010309,-0.010309,0.097410,0.272351,-1.109606,两者接近


,pair_name,branch,branch_cn,MMRL_best_common_score,MMRL_best_common_score_cn,MMRL_best_common_AUROC_error,MMRL_best_common_AUPR_error,Bayes_best_common_score,Bayes_best_common_score_cn,Bayes_best_common_AUROC_error,Bayes_best_common_AUPR_error,delta_best_common_AUROC_Bayes_minus_MMRL,delta_best_common_AUPR_Bayes_minus_MMRL,best_common_conclusion_hint
0,case_B__VS__case_A,fusion,融合分支,predictive_entropy,预测熵,0.912799,0.605232,predictive_entropy,预测熵,0.898350,0.520132,-0.014449,-0.085100,MMRL 共有风险分数最优表现更好
1,case_B__VS__case_A,main,主分支,one_minus_confidence,低置信度,0.906457,0.606662,one_minus_confidence,低置信度,0.892116,0.577263,-0.014340,-0.029399,MMRL 共有风险分数最优表现更好
2,case_B__VS__case_A,rep,表示增强分支,predictive_entropy,预测熵,0.889681,0.612405,one_minus_confidence,低置信度,0.888736,0.587344,-0.000945,-0.025062,两者接近


,pair_name,branch,branch_cn,bayes_specific_score,bayes_specific_score_cn,bayes_specific_AUROC_error,bayes_specific_AUPR_error,MMRL_best_common_score,MMRL_best_common_score_cn,MMRL_best_common_AUROC_error,MMRL_best_common_AUPR_error,Bayes_best_common_score,Bayes_best_common_score_cn,Bayes_best_common_AUROC_error,Bayes_best_common_AUPR_error,delta_vs_MMRL_best_common_AUROC,delta_vs_MMRL_best_common_AUPR,delta_vs_Bayes_best_common_AUROC,delta_vs_Bayes_best_common_AUPR,bayes_specific_conclusion_hint
0,case_B__VS__case_A,fusion,融合分支,mutual_information,互信息,0.884976,0.481142,predictive_entropy,预测熵,0.912799,0.605232,predictive_entropy,预测熵,0.898350,0.520132,-0.027823,-0.124090,-0.013374,-0.038990,Bayes特有分数未超过 MMRL 最佳普通基线
1,case_B__VS__case_A,fusion,融合分支,variation_ratio,投票分歧率,0.783570,0.446030,predictive_entropy,预测熵,0.912799,0.605232,predictive_entropy,预测熵,0.898350,0.520132,-0.129229,-0.159202,-0.114780,-0.074103,Bayes特有分数未超过 MMRL 最佳普通基线
2,case_B__VS__case_A,fusion,融合分支,vote_entropy,投票熵,0.784042,0.452991,predictive_entropy,预测熵,0.912799,0.605232,predictive_entropy,预测熵,0.898350,0.520132,-0.128756,-0.152241,-0.114308,-0.067141,Bayes特有分数未超过 MMRL 最佳普通基线
3,case_B__VS__case_A,fusion,融合分支,logit_variance,类别打分方差,0.661963,0.206781,predictive_entropy,预测熵,0.912799,0.605232,predictive_entropy,预测熵,0.898350,0.520132,-0.250835,-0.398451,-0.236386,-0.313351,Bayes特有分数未超过 MMRL 最佳普通基线
4,case_B__VS__case_A,main,主分支,mutual_information,互信息,0.858021,0.458462,one_minus_confidence,低置信度,0.906457,0.606662,one_minus_confidence,低置信度,0.892116,0.577263,-0.048436,-0.148199,-0.034096,-0.118801,Bayes特有分数未超过 MMRL 最佳普通基线
5,case_B__VS__case_A,main,主分支,variation_ratio,投票分歧率,0.780693,0.483055,one_minus_confidence,低置信度,0.906457,0.606662,one_minus_confidence,低置信度,0.892116,0.577263,-0.125764,-0.123607,-0.111423,-0.094208,Bayes特有分数未超过 MMRL 最佳普通基线
6,case_B__VS__case_A,main,主分支,vote_entropy,投票熵,0.781591,0.499965,one_minus_confidence,低置信度,0.906457,0.606662,one_minus_confidence,低置信度,0.892116,0.577263,-0.124866,-0.106696,-0.110526,-0.077298,Bayes特有分数未超过 MMRL 最佳普通基线
7,case_B__VS__case_A,main,主分支,logit_variance,类别打分方差,0.604754,0.209961,one_minus_confidence,低置信度,0.906457,0.606662,one_minus_confidence,低置信度,0.892116,0.577263,-0.301703,-0.396700,-0.287363,-0.367302,Bayes特有分数未超过 MMRL 最佳普通基线
8,case_B__VS__case_A,rep,表示增强分支,mutual_information,互信息,0.880505,0.565541,predictive_entropy,预测熵,0.889681,0.612405,one_minus_confidence,低置信度,0.888736,0.587344,-0.009176,-0.046864,-0.008231,-0.021803,Bayes特有分数未超过 MMRL 最佳普通基线
9,case_B__VS__case_A,rep,表示增强分支,variation_ratio,投票分歧率,0.840847,0.562653,predictive_entropy,预测熵,0.889681,0.612405,one_minus_confidence,低置信度,0.888736,0.587344,-0.048835,-0.049752,-0.047890,-0.024691,Bayes特有分数未超过 MMRL 最佳普通基线


,pair_name,branch,branch_cn,score,score_cn,coverage,accuracy_gain_vs_full_BayesMMRL,accuracy_gain_vs_full_MMRL,full_accuracy_BayesMMRL,full_accuracy_MMRL,rejected_error_rate_BayesMMRL,rejected_error_rate_MMRL,retained_accuracy_BayesMMRL,retained_accuracy_MMRL,retained_error_rate_BayesMMRL,retained_error_rate_MMRL,delta_full_accuracy_Bayes_minus_MMRL,delta_retained_accuracy_Bayes_minus_MMRL,delta_accuracy_gain_vs_full_Bayes_minus_MMRL,delta_retained_error_rate_Bayes_minus_MMRL,delta_rejected_error_rate_Bayes_minus_MMRL
0,case_B__VS__case_A,fusion,融合分支,one_minus_confidence,低置信度,1.00,0.000000,0.000000,0.869151,0.873645,NaN,NaN,0.869151,0.873645,0.130849,0.126355,-0.004494,-0.004494,0.000000,0.004494,NaN
2,case_B__VS__case_A,fusion,融合分支,one_minus_confidence,低置信度,0.95,0.025117,0.032031,0.869151,0.873645,0.608466,0.735450,0.894268,0.905676,0.105732,0.094324,-0.004494,-0.011408,-0.006914,0.011408,-0.126984
3,case_B__VS__case_A,fusion,融合分支,one_minus_confidence,低置信度,0.90,0.047735,0.054695,0.869151,0.873645,0.560847,0.619048,0.916887,0.928341,0.083113,0.071659,-0.004494,-0.011454,-0.006960,0.011454,-0.058201
5,case_B__VS__case_A,fusion,融合分支,one_minus_confidence,低置信度,0.80,0.081609,0.086037,0.869151,0.873645,0.457067,0.470277,0.950760,0.959683,0.049240,0.040317,-0.004494,-0.008923,-0.004429,0.008923,-0.013210
8,case_B__VS__case_A,fusion,融合分支,one_minus_probability_margin,低决策间隔,1.00,0.000000,0.000000,0.869151,0.873645,NaN,NaN,0.869151,0.873645,0.130849,0.126355,-0.004494,-0.004494,0.000000,0.004494,NaN
10,case_B__VS__case_A,fusion,融合分支,one_minus_probability_margin,低决策间隔,0.95,0.022891,0.028135,0.869151,0.873645,0.566138,0.661376,0.892042,0.901781,0.107958,0.098219,-0.004494,-0.009738,-0.005245,0.009738,-0.095238
11,case_B__VS__case_A,fusion,融合分支,one_minus_probability_margin,低决策间隔,0.90,0.044211,0.050584,0.869151,0.873645,0.529101,0.582011,0.913363,0.924229,0.086637,0.075771,-0.004494,-0.010866,-0.006373,0.010866,-0.052910
13,case_B__VS__case_A,fusion,融合分支,one_minus_probability_margin,低决策间隔,0.80,0.079956,0.085707,0.869151,0.873645,0.450462,0.468956,0.949108,0.959352,0.050892,0.040648,-0.004494,-0.010245,-0.005751,0.010245,-0.018494
16,case_B__VS__case_A,fusion,融合分支,predictive_entropy,预测熵,1.00,0.000000,0.000000,0.869151,0.873645,NaN,NaN,0.869151,0.873645,0.130849,0.126355,-0.004494,-0.004494,0.000000,0.004494,NaN
18,case_B__VS__case_A,fusion,融合分支,predictive_entropy,预测熵,0.95,0.026508,0.032031,0.869151,0.873645,0.634921,0.735450,0.895659,0.905676,0.104341,0.094324,-0.004494,-0.010017,-0.005523,0.010017,-0.100529


## 14. Difficulty-aware 与 class/confusion 分析


In [51]:
def difficulty_and_class_tables(pairings, case_outputs):
    diff_rows = []
    class_rows = []
    confusion_rows = []

    for pair in pairings:
        m = case_outputs[pair['mmrl']]
        b = case_outputs[pair['bayes']]
        assert_same_samples(m, b)
        labels = m['labels']
        p_m = probs_for_case_mode_branch(m, 'standard', 'fusion')
        p_b = probs_for_case_mode_branch(b, 'mc_prob_mean_probs', 'fusion')
        pred_m, conf_m, margin_m, ent_m = pred_conf_margin_entropy_from_probs(p_m)
        pred_b, conf_b, margin_b, ent_b = pred_conf_margin_entropy_from_probs(p_b)
        c_m = pred_m.eq(labels)
        c_b = pred_b.eq(labels)

        u = predictive_uncertainty_from_mc_logits(b['mc_samples']['fusion'])
        mi = u['mutual_info']
        var = u['variation_ratio']
        agree = u['sample_agreement']

        q_conf_hi = torch.quantile(conf_m.float(), 0.75)
        q_margin_lo = torch.quantile(margin_m.float(), 0.25)
        q_mi_hi = torch.quantile(mi.float(), 0.75)
        q_conf_b_hi = torch.quantile(conf_b.float(), 0.75)
        q_mi_lo = torch.quantile(mi.float(), 0.25)

        groups = {
            'easy_mmrl_correct_high_conf': c_m & (conf_m >= q_conf_hi),
            'hard_mmrl_wrong_or_low_margin': (~c_m) | (margin_m <= q_margin_lo),
            'ambiguous_high_MI': mi >= q_mi_hi,
            'overconfident_wrong_bayes': (~c_b) & (conf_b >= q_conf_b_hi) & (mi <= q_mi_lo),
            'all': torch.ones_like(labels, dtype=torch.bool),
        }
        for g, mask in groups.items():
            if not mask.any():
                continue
            diff_rows.append({
                'pair_name': pair['pair_name'],
                'difficulty_group': g,
                'n': int(mask.sum().item()),
                'MMRL_acc': float(c_m[mask].float().mean().item()),
                'Bayes_acc': float(c_b[mask].float().mean().item()),
                'Bayes_conf': float(conf_b[mask].mean().item()),
                'Bayes_MI': float(mi[mask].mean().item()),
                'sample_agreement': float(agree[mask].mean().item()),
                'harmful_flip_rate': float((c_m & ~c_b)[mask].float().mean().item()),
                'beneficial_flip_rate': float((~c_m & c_b)[mask].float().mean().item()),
            })

        n_classes = int(max(labels.max().item(), pred_m.max().item(), pred_b.max().item()) + 1)
        for c in range(n_classes):
            mask = labels.eq(c)
            if not mask.any():
                continue
            class_rows.append({
                'pair_name': pair['pair_name'],
                'class_id': c,
                'n': int(mask.sum().item()),
                'MMRL_acc': float(c_m[mask].float().mean().item()),
                'Bayes_acc': float(c_b[mask].float().mean().item()),
                'delta_acc': float(c_b[mask].float().mean().item() - c_m[mask].float().mean().item()),
                'Bayes_conf': float(conf_b[mask].mean().item()),
                'Bayes_MI': float(mi[mask].mean().item()),
                'harmful_flip': int((c_m & ~c_b & mask).sum().item()),
                'beneficial_flip': int((~c_m & c_b & mask).sum().item()),
            })

        # confusion pair delta
        cm_m = Counter(zip(labels.numpy().tolist(), pred_m.numpy().tolist()))
        cm_b = Counter(zip(labels.numpy().tolist(), pred_b.numpy().tolist()))
        keys = sorted(set(cm_m) | set(cm_b))
        for true_c, pred_c in keys:
            mask_pair_b = labels.eq(true_c) & pred_b.eq(pred_c)
            confusion_rows.append({
                'pair_name': pair['pair_name'],
                'true_class': int(true_c),
                'pred_class': int(pred_c),
                'MMRL_count': int(cm_m.get((true_c, pred_c), 0)),
                'Bayes_count': int(cm_b.get((true_c, pred_c), 0)),
                'delta': int(cm_b.get((true_c, pred_c), 0) - cm_m.get((true_c, pred_c), 0)),
                'Bayes_conf': float(conf_b[mask_pair_b].mean().item()) if mask_pair_b.any() else np.nan,
                'Bayes_MI': float(mi[mask_pair_b].mean().item()) if mask_pair_b.any() else np.nan,
            })

    return pd.DataFrame(diff_rows), pd.DataFrame(class_rows), pd.DataFrame(confusion_rows)

difficulty_df, class_level_df, confusion_df = difficulty_and_class_tables(PAIRINGS, case_outputs)
display(difficulty_df)
display(class_level_df.sort_values(['pair_name', 'delta_acc']).head(20) if len(class_level_df) else class_level_df)
display(confusion_df.sort_values(['pair_name', 'delta']).head(20) if len(confusion_df) else confusion_df)
difficulty_df.to_csv(SAVE_DIR / 'table_difficulty_aware.csv', index=False)
class_level_df.to_csv(SAVE_DIR / 'table_class_level.csv', index=False)
confusion_df.to_csv(SAVE_DIR / 'table_confusion_pairs.csv', index=False)


,pair_name,difficulty_group,n,MMRL_acc,Bayes_acc,Bayes_conf,Bayes_MI,sample_agreement,harmful_flip_rate,beneficial_flip_rate
0,case_B__VS__case_A,easy_mmrl_correct_high_conf,946,1.000000,1.000000,0.998461,0.000676,0.999947,0.000000,0.000000
1,case_B__VS__case_A,hard_mmrl_wrong_or_low_margin,1033,0.537270,0.528558,0.649939,0.084888,0.848209,0.103582,0.094869
2,case_B__VS__case_A,ambiguous_high_MI,946,0.608880,0.599366,0.617946,0.103489,0.830285,0.102537,0.093023
3,case_B__VS__case_A,all,3783,0.873645,0.869151,0.876283,0.030934,0.954071,0.030399,0.025905


,pair_name,class_id,n,MMRL_acc,Bayes_acc,delta_acc,Bayes_conf,Bayes_MI,harmful_flip,beneficial_flip
83,case_B__VS__case_A,83,39,0.948718,0.769231,-0.179487,0.849328,0.050726,7,0
31,case_B__VS__case_A,31,37,0.810811,0.675676,-0.135135,0.724291,0.030653,7,2
95,case_B__VS__case_A,95,28,0.785714,0.678571,-0.107143,0.791051,0.067543,3,0
67,case_B__VS__case_A,67,40,0.725000,0.625000,-0.100000,0.798247,0.054742,5,1
74,case_B__VS__case_A,74,34,0.911765,0.823529,-0.088235,0.751180,0.095898,3,0
29,case_B__VS__case_A,29,36,0.888889,0.805556,-0.083333,0.803313,0.058748,4,1
69,case_B__VS__case_A,69,28,0.964286,0.892857,-0.071429,0.815272,0.048510,2,0
72,case_B__VS__case_A,72,28,0.857143,0.785714,-0.071429,0.945726,0.012552,2,0
35,case_B__VS__case_A,35,45,0.577778,0.511111,-0.066667,0.693102,0.043392,4,1
4,case_B__VS__case_A,4,31,0.709677,0.645161,-0.064516,0.814398,0.049555,3,1


,pair_name,true_class,pred_class,MMRL_count,Bayes_count,delta,Bayes_conf,Bayes_MI
162,case_B__VS__case_A,39,67,10,3,-7,0.642346,0.119558
331,case_B__VS__case_A,83,83,37,30,-7,0.911842,0.032939
121,case_B__VS__case_A,31,31,30,25,-5,0.741675,0.024976
101,case_B__VS__case_A,28,84,4,0,-4,NaN,NaN
285,case_B__VS__case_A,67,67,29,25,-4,0.848148,0.042337
290,case_B__VS__case_A,68,56,5,1,-4,0.372010,0.133167
42,case_B__VS__case_A,14,47,3,0,-3,NaN,NaN
76,case_B__VS__case_A,22,23,7,4,-3,0.657481,0.046185
106,case_B__VS__case_A,29,29,32,29,-3,0.865769,0.042630
137,case_B__VS__case_A,35,35,26,23,-3,0.771791,0.028631


## 15. Optional: training dynamics 日志解析

如果日志里有 `CE / KL / acc / ECE / NLL / sigma` 等字段，这里会尽量抽取。不同日志格式差异较大，所以这张表是 best-effort。


In [52]:
def parse_training_dynamics_for_case(meta):
    text, files = collect_log_text(meta['log_root'])
    rows = []
    # 常见格式：epoch [x/y] ... loss ... raw_kl_rep ... val_acc ...
    epoch_pat = re.compile(r'(?:epoch|Epoch)\s*[:\[]?\s*(\d+)', re.IGNORECASE)
    metrics = ['loss', 'data_term', 'raw_kl_rep', 'raw_kl_proj_rep', 'kl_term', 'kl_beta', 'acc', 'val_acc', 'ECE', 'NLL', 'sigma_mean', 'mean_shift']
    for line in text.splitlines():
        if not any(k.lower() in line.lower() for k in ['epoch', 'kl', 'sigma', 'ece', 'nll', 'val_acc', 'data_term']):
            continue
        m = epoch_pat.search(line)
        if not m:
            continue
        row = {'case_name': meta['name'], 'epoch': int(m.group(1)), 'line': line[:500]}
        for key in metrics:
            km = re.search(rf'{re.escape(key)}\s*[:=]\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)', line)
            if km:
                row[key] = float(km.group(1))
        rows.append(row)
    return pd.DataFrame(rows)

training_dynamics_df = pd.concat([parse_training_dynamics_for_case(m) for m in case_metas], ignore_index=True) if case_metas else pd.DataFrame()
display(training_dynamics_df.head(50))
training_dynamics_df.to_csv(SAVE_DIR / 'table_training_dynamics_best_effort.csv', index=False)


,case_name,epoch,line
0,case_A,50,MAX_EPOCH: 50
1,case_A,1,WARMUP_EPOCH: 1
2,case_A,1,epoch [1/50] batch [20/50] time 0.255 (0.307) ...
3,case_A,1,epoch [1/50] batch [40/50] time 0.257 (0.280) ...
4,case_A,2,epoch [2/50] batch [20/50] time 0.264 (0.269) ...
5,case_A,2,epoch [2/50] batch [40/50] time 0.249 (0.263) ...
6,case_A,3,epoch [3/50] batch [20/50] time 0.131 (0.217) ...
7,case_A,3,epoch [3/50] batch [40/50] time 0.116 (0.176) ...
8,case_A,4,epoch [4/50] batch [20/50] time 0.144 (0.138) ...
9,case_A,4,epoch [4/50] batch [40/50] time 0.129 (0.135) ...


## 16. 自动结论摘要

规则不是最终论文结论，只是帮助快速定位：有效 / 无效 / 有害 / 局部有效。


In [53]:
def automatic_judgement():
    rows = []
    for _, pair in enumerate(PAIRINGS):
        pair_name = pair['pair_name']
        po = paired_outcome_df[(paired_outcome_df['pair_name'] == pair_name) & (paired_outcome_df['bayes_mode'] == 'mc_prob_mean_probs')]
        if po.empty:
            continue
        d = {r['case_type']: r for _, r in po.iterrows()}
        beneficial = int(d.get('MMRL_wrong_Bayes_correct', {}).get('count', 0)) if 'MMRL_wrong_Bayes_correct' in d else 0
        harmful = int(d.get('MMRL_correct_Bayes_wrong', {}).get('count', 0)) if 'MMRL_correct_Bayes_wrong' in d else 0

        bayes_name = pair['bayes']
        pdiag = posterior_df[posterior_df['case_name'] == bayes_name].iloc[0].to_dict() if not posterior_df.empty else {}
        kl = pdiag.get('KL', np.nan)
        sigma_mean = pdiag.get('sigma_mean', np.nan)
        mean_shift = pdiag.get('mean_shift_l2', np.nan)
        snr = pdiag.get('SNR_mean', np.nan)

        err = mc_tables['error_detection']
        err_fusion = err[(err['case_name'] == bayes_name) & (err['branch'] == 'fusion') & (err['score'] == 'MI')]
        mi_auc = float(err_fusion['AUROC_error'].iloc[0]) if len(err_fusion) else np.nan

        agg = mc_tables['aggregation_conflict']
        agg_fusion = agg[(agg['case_name'] == bayes_name) & (agg['branch'] == 'fusion')]
        prob_vote_agree = float(agg_fusion['prob_vote_agree'].iloc[0]) if len(agg_fusion) else np.nan

        label = '局部有效/需人工判断'
        reasons = []
        if pd.notna(kl) and abs(kl) < 1e-8 and pd.notna(mean_shift) and mean_shift < 1e-8:
            label = '可能无效'
            reasons.append('KL≈0 且 mean_shift≈0')
        if harmful > beneficial:
            label = '可能有害'
            reasons.append(f'harmful_flip({harmful}) > beneficial_flip({beneficial})')
        elif beneficial > harmful:
            label = '可能有效'
            reasons.append(f'beneficial_flip({beneficial}) > harmful_flip({harmful})')
        if pd.notna(mi_auc) and mi_auc < 0.55:
            reasons.append(f'MI error AUROC较低({mi_auc:.3f})')
        if pd.notna(prob_vote_agree) and prob_vote_agree < 0.9:
            reasons.append(f'prob_mean vs majority_vote 分歧较高(agree={prob_vote_agree:.3f})')

        rows.append({
            'pair_name': pair_name,
            'judgement_hint': label,
            'beneficial_flip': beneficial,
            'harmful_flip': harmful,
            'KL': kl,
            'sigma_mean': sigma_mean,
            'mean_shift_l2': mean_shift,
            'SNR_mean': snr,
            'MI_AUROC_error_fusion': mi_auc,
            'prob_vote_agree_fusion': prob_vote_agree,
            'reasons': '; '.join(reasons),
        })
    return pd.DataFrame(rows)

judgement_df = automatic_judgement()
display(judgement_df)
judgement_df.to_csv(SAVE_DIR / 'summary_automatic_judgement.csv', index=False)


,pair_name,judgement_hint,beneficial_flip,harmful_flip,KL,sigma_mean,mean_shift_l2,SNR_mean,MI_AUROC_error_fusion,prob_vote_agree_fusion,reasons
0,case_B__VS__case_A,可能有害,98,115,64.077103,0.482468,5.509155,0.1933,0.884976,0.992863,harmful_flip(115) > beneficial_flip(98)


## 17. 保存所有结果


In [54]:
# 保存 notebook 运行产生的关键表到一个 Excel，方便汇总查看。
# 如果 openpyxl 不可用，会只保留前面已经写出的 CSV。
all_tables = {
    'implementation_audit': implementation_audit,
    'case_metadata': meta_df,
    'posterior_diagnostics': posterior_df,
    'branch_main_metrics': branch_metrics_df,
    'branch_disagreement': branch_disagreement_df,
    'fusion_rescue_damage': fusion_rescue_damage_df,
    'paired_outcome': paired_outcome_df,
    'feature_shift': feature_shift_df,
    'feature_difference_sample': feature_difference_sample_df,
    'text_feature_difference': text_feature_difference_df,
    'feature_shift_distribution': feature_shift_distribution_df,
    'feature_shift_effect': feature_shift_effect_df,
    'feature_shift_error_detection': feature_shift_error_detection_df,
    'feature_shift_correlation': feature_shift_correlation_df,
    'top_feature_shift_samples': top_feature_shift_samples_df,
    'fusion_margin': fusion_margin_df,
    'counterfactual_swap': counterfactual_swap_df,
    'difficulty_aware': difficulty_df,
    'class_level': class_level_df,
    'confusion_pairs': confusion_df,
    'training_dynamics': training_dynamics_df,
    'automatic_judgement': judgement_df,
    'uncertainty_baseline_comparison': uncertainty_baseline_comparison_df,
    'risk_coverage': risk_coverage_df,
    'uncertainty_gain_summary': uncertainty_gain_summary_df,
    'uncertainty_score_correlation': uncertainty_score_correlation_df,
}
for k, df in mc_tables.items():
    all_tables[k] = df

xlsx_path = SAVE_DIR / 'bayes_mmrl_mechanism_diagnosis_tables.xlsx'
try:
    with pd.ExcelWriter(xlsx_path) as writer:
        for name, df in all_tables.items():
            safe_name = name[:31]
            if isinstance(df, pd.DataFrame):
                df.to_excel(writer, sheet_name=safe_name, index=False)
    print('Saved Excel:', xlsx_path)
except Exception as e:
    warnings.warn(f'Excel 保存失败: {e}')

print('All CSV tables saved under:', SAVE_DIR)


All CSV tables saved under: /root/autodl-tmp/MMRL/analysis_outputs/bayes_mmrl_mechanism_rebuild


/tmp/ipykernel_100077/791948232.py:43: UserWarning: Excel 保存失败: No module named 'openpyxl'
  warnings.warn(f'Excel 保存失败: {e}')


## Save all module analysis results into one file

这个 cell 只负责保存结果，不做额外分析判断。

- 输出一个 Excel 文件：每个分析模块一个 sheet。
- 额外包含 `manifest` sheet，记录每个模块是否存在、行数、列数、保存状态。
- 额外包含 `missing_modules` sheet，列出未生成的模块，方便检查前面哪个 cell 没跑。
- 同时可选保存一个 pickle，避免 Excel 对复杂对象/长列名/大表的限制。

In [55]:

# ============================================================
# Save all module analysis results into ONE Excel file
# ============================================================
# 目标：把每个模块的分析结果都保存到同一个 xlsx 中。
# 说明：这个 cell 不做新的分析，只收集当前 notebook 已经生成的 DataFrame。
#
# 输出：
#   1) bayes_mmrl_all_module_results.xlsx
#      - manifest: 每个模块是否存在、shape、sheet_name、备注
#      - missing_modules: 未生成的模块
#      - 其他 sheet: 每个模块一个表
#   2) bayes_mmrl_all_module_results.pkl
#      - 保存原始 DataFrame dict，避免 Excel 行数/列数/类型限制

import re
import pickle
import warnings
from pathlib import Path

import pandas as pd

SAVE_DIR.mkdir(parents=True, exist_ok=True)

def _get_global_df(var_name):
    obj = globals().get(var_name, None)
    if isinstance(obj, pd.DataFrame):
        return obj
    return None

def _as_dataframe(obj):
    if obj is None:
        return None
    if isinstance(obj, pd.DataFrame):
        return obj
    if isinstance(obj, pd.Series):
        return obj.to_frame()
    try:
        return pd.DataFrame(obj)
    except Exception:
        return pd.DataFrame({"repr": [repr(obj)]})

def _sanitize_sheet_name(name, used):
    # Excel sheet name limit: 31 chars
    # Invalid chars: []:*?/\
    s = re.sub(r"[\[\]\:\*\?\/\\]", "_", str(name))
    s = s.strip()
    if not s:
        s = "sheet"
    s = s[:31]

    base = s
    i = 1
    while s in used:
        suffix = f"_{i}"
        s = base[:31 - len(suffix)] + suffix
        i += 1

    used.add(s)
    return s

# ------------------------------------------------------------------
# 1. 显式列出所有核心模块表
# ------------------------------------------------------------------
# key = 希望出现在 Excel 中的模块名
# value = notebook 中对应的变量名
CORE_MODULE_TABLES = {
    # 审计与元信息
    "implementation_audit": "implementation_audit",
    "case_metadata": "meta_df",

    # 1 Posterior diagnostics
    "posterior_diagnostics": "posterior_df",

    # 2 Branch main metrics
    "branch_main_metrics": "branch_metrics_df",

    # 3 Branch disagreement
    "branch_disagreement": "branch_disagreement_df",

    # 4 Fusion rescue / damage
    "fusion_rescue_damage": "fusion_rescue_damage_df",

    # 5 Fusion margin decomposition
    "fusion_margin_decomposition": "fusion_margin_df",

    # 6 Bayes vs MMRL paired outcome
    "paired_outcome": "paired_outcome_df",

    # 7 Distribution shift
    "distribution_shift": "distribution_shift_df",

    # 8 Feature similarity / feature shift
    "feature_shift": "feature_shift_df",
    "feature_difference_sample": "feature_difference_sample_df",
    "text_feature_difference": "text_feature_difference_df",
    "feature_shift_distribution": "feature_shift_distribution_df",
    "feature_shift_effect": "feature_shift_effect_df",
    "feature_shift_error_detection": "feature_shift_error_detection_df",
    "feature_shift_correlation": "feature_shift_correlation_df",
    "mc_sampled_feature_difference": "mc_sampled_feature_difference_df",
    "mc_sampled_feature_sample": "mc_sampled_feature_sample_df",
    "branch_logit_difference": "branch_logit_difference_df",
    "branch_logit_difference_sample": "branch_logit_difference_sample_df",
    "mc_logit_variation_by_branch": "mc_logit_variation_by_branch_df",
    "mc_logit_variation_sample": "mc_logit_variation_sample_df",
    "conflict_sample": "conflict_sample_df",
    "branch_disagreement_outcome_sample": "branch_disagreement_outcome_sample_df",
    "branch_disagreement_outcome_summary": "branch_disagreement_outcome_summary_df",
    "branch_margin_conflict_summary": "branch_margin_conflict_summary_df",
    "top_feature_shift_samples": "top_feature_shift_samples_df",

    # 9 MC uncertainty
    "mc_uncertainty": "mc_uncertainty_df",

    # 10 Error detection
    "error_detection": "error_detection_df",
    "uncertainty_baseline_comparison": "uncertainty_baseline_comparison_df",
    "risk_coverage": "risk_coverage_df",
    "uncertainty_gain_summary": "uncertainty_gain_summary_df",
    "uncertainty_score_correlation": "uncertainty_score_correlation_df",
    "same_score_uncertainty_comparison": "same_score_uncertainty_comparison_df",
    "best_common_uncertainty_comparison": "best_common_uncertainty_comparison_df",
    "bayes_specific_uncertainty_value": "bayes_specific_uncertainty_value_df",
    "risk_coverage_same_score": "risk_coverage_same_score_df",

    # 11 MC sample prediction conflict
    "sample_prediction_conflict": "sample_conflict_df",

    # 12 Margin sign conflict
    "margin_sign_conflict": "margin_sign_conflict_df",

    # 13 Branch conflict inside each MC sample
    "branch_conflict_inside_mc": "branch_conflict_df",

    # 14 Fusion conflict under sampling
    "fusion_conflict_sampling": "fusion_conflict_df",

    # 15 Aggregation conflict
    "aggregation_conflict": "aggregation_conflict_df",

    # 16 Counterfactual branch swap
    "counterfactual_branch_swap": "counterfactual_swap_df",

    # 17 Difficulty-aware analysis
    "difficulty_aware": "difficulty_df",

    # 18 Class-level / confusion-level analysis
    "class_level": "class_level_df",
    "confusion_pairs": "confusion_df",

    # 19 Training dynamics
    "training_dynamics": "training_dynamics_df",

    # automatic judgement, if generated
    "automatic_judgement": "judgement_df",
}

all_module_tables = {}
manifest_rows = []

# ------------------------------------------------------------------
# 2. 收集核心模块表
# ------------------------------------------------------------------
for module_name, var_name in CORE_MODULE_TABLES.items():
    df = _get_global_df(var_name)
    if df is None:
        manifest_rows.append({
            "module": module_name,
            "variable": var_name,
            "status": "missing",
            "rows": 0,
            "cols": 0,
            "sheet_name": None,
            "note": f"Variable `{var_name}` not found or not a DataFrame",
        })
        continue

    all_module_tables[module_name] = df
    manifest_rows.append({
        "module": module_name,
        "variable": var_name,
        "status": "ok",
        "rows": len(df),
        "cols": len(df.columns),
        "sheet_name": None,
        "note": "",
    })

# ------------------------------------------------------------------
# 3. 收集 mc_tables 里的所有动态模块表
# ------------------------------------------------------------------
# 有些 notebook 版本会把 MC uncertainty / conflict / aggregation 等放进 mc_tables dict。
_mc_tables = globals().get("mc_tables", None)
if isinstance(_mc_tables, dict):
    for key, value in _mc_tables.items():
        module_name = f"mc_tables__{key}"
        df = _as_dataframe(value)

        if df is None:
            manifest_rows.append({
                "module": module_name,
                "variable": f"mc_tables[{key!r}]",
                "status": "missing",
                "rows": 0,
                "cols": 0,
                "sheet_name": None,
                "note": "mc_tables entry is None or cannot be converted",
            })
            continue

        all_module_tables[module_name] = df
        manifest_rows.append({
            "module": module_name,
            "variable": f"mc_tables[{key!r}]",
            "status": "ok",
            "rows": len(df),
            "cols": len(df.columns),
            "sheet_name": None,
            "note": "",
        })
else:
    manifest_rows.append({
        "module": "mc_tables",
        "variable": "mc_tables",
        "status": "missing",
        "rows": 0,
        "cols": 0,
        "sheet_name": None,
        "note": "`mc_tables` dict not found",
    })

# ------------------------------------------------------------------
# 4. 可选：自动收集所有 *_df DataFrame，避免漏掉 notebook 中新增模块
# ------------------------------------------------------------------
AUTO_COLLECT_OTHER_DF = True

if AUTO_COLLECT_OTHER_DF:
    already_vars = set(CORE_MODULE_TABLES.values())
    if isinstance(_mc_tables, dict):
        # mc_tables 内部不属于 globals variable
        pass

    for var_name, obj in sorted(globals().items()):
        if not var_name.endswith("_df"):
            continue
        if var_name in already_vars:
            continue
        if not isinstance(obj, pd.DataFrame):
            continue

        module_name = f"extra__{var_name}"
        if module_name in all_module_tables:
            continue

        all_module_tables[module_name] = obj
        manifest_rows.append({
            "module": module_name,
            "variable": var_name,
            "status": "ok_extra_auto_collected",
            "rows": len(obj),
            "cols": len(obj.columns),
            "sheet_name": None,
            "note": "Auto-collected because variable name ends with `_df`",
        })

# ------------------------------------------------------------------
# 5. 写入 Excel，一个模块一个 sheet
# ------------------------------------------------------------------
xlsx_path = SAVE_DIR / "bayes_mmrl_all_module_results.xlsx"
pkl_path = SAVE_DIR / "bayes_mmrl_all_module_results.pkl"

used_sheet_names = set()

# 先生成 sheet name 映射
module_to_sheet = {}
for module_name in all_module_tables:
    module_to_sheet[module_name] = _sanitize_sheet_name(module_name, used_sheet_names)

# 更新 manifest 中的 sheet_name
for row in manifest_rows:
    module_name = row["module"]
    if module_name in module_to_sheet:
        row["sheet_name"] = module_to_sheet[module_name]

manifest_df = pd.DataFrame(manifest_rows)
missing_modules_df = manifest_df[manifest_df["status"].astype(str).str.contains("missing", case=False, na=False)].copy()

# 把 manifest 也放进 all_module_tables，但放在最前面写
try:
    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
        manifest_df.to_excel(writer, sheet_name="manifest", index=False)
        missing_modules_df.to_excel(writer, sheet_name="missing_modules", index=False)

        for module_name, df in all_module_tables.items():
            sheet = module_to_sheet[module_name]
            try:
                df.to_excel(writer, sheet_name=sheet, index=False)
            except Exception as e:
                # 如果某个表写失败，不让整个 Excel 中断
                err_df = pd.DataFrame({
                    "module": [module_name],
                    "sheet": [sheet],
                    "error": [repr(e)],
                })
                fallback_sheet = _sanitize_sheet_name(f"ERROR_{module_name}", used_sheet_names)
                err_df.to_excel(writer, sheet_name=fallback_sheet, index=False)

    print("Saved one Excel file with all module tables:")
    print(xlsx_path)

except Exception as e:
    warnings.warn(f"Excel save failed: {e}")

# ------------------------------------------------------------------
# 6. 额外保存 pickle：一个文件内保留所有原始 DataFrame
# ------------------------------------------------------------------
try:
    with open(pkl_path, "wb") as f:
        pickle.dump({
            "manifest": manifest_df,
            "missing_modules": missing_modules_df,
            "tables": all_module_tables,
        }, f)

    print("Saved one pickle file with all raw module DataFrames:")
    print(pkl_path)

except Exception as e:
    warnings.warn(f"Pickle save failed: {e}")

# ------------------------------------------------------------------
# 7. 显示保存概览
# ------------------------------------------------------------------
print()
print("Module tables collected:", len(all_module_tables))
print("Missing modules:", len(missing_modules_df))
display(manifest_df)


Saved one pickle file with all raw module DataFrames:
/root/autodl-tmp/MMRL/analysis_outputs/bayes_mmrl_mechanism_rebuild/bayes_mmrl_all_module_results.pkl

Module tables collected: 50
Missing modules: 8


/tmp/ipykernel_100077/3593633949.py:312: UserWarning: Excel save failed: No module named 'openpyxl'
  warnings.warn(f"Excel save failed: {e}")


,module,variable,status,rows,cols,sheet_name,note
0,implementation_audit,implementation_audit,ok,5,7,implementation_audit,
1,case_metadata,meta_df,ok,2,26,case_metadata,
2,posterior_diagnostics,posterior_df,ok,2,24,posterior_diagnostics,
3,branch_main_metrics,branch_metrics_df,ok,18,13,branch_main_metrics,
4,branch_disagreement,branch_disagreement_df,ok,6,7,branch_disagreement,
5,fusion_rescue_damage,fusion_rescue_damage_df,ok,6,11,fusion_rescue_damage,
6,fusion_margin_decomposition,fusion_margin_df,ok,15,8,fusion_margin_decomposition,
7,paired_outcome,paired_outcome_df,ok,12,14,paired_outcome,
8,distribution_shift,distribution_shift_df,missing,0,0,None,Variable `distribution_shift_df` not found or ...
9,feature_shift,feature_shift_df,ok,5,6,feature_shift,
